# axSpA — completed cleaned-data experiment: training and analysis notebook

This notebook documents the completed revision experiment: **20 runs (four model configurations × five seeds), 824 prepared PNG images, and frozen train/validation/external-test folders of 579/95/150 images**. It is retained for reproducibility. All computational code cells are unchanged from the completed notebook; only explanatory Markdown has been updated for release.

| Split | normal | osteofity | parasyndesmofity | syndesmofity | Total |
|---|---:|---:|---:|---:|---:|
| Training | 211 | 210 | 81 | 77 | 579 |
| Validation | 37 | 34 | 12 | 12 | 95 |
| External test | 55 | 54 | 20 | 21 | 150 |

The reference configuration is **YOLO11l-cls, seed 0**, fixed before this revision evaluation. Its external accuracy was **138/150 (92.0%; exact 95% CI 86.4–95.8%)**. YOLO11 accuracy across five seeds was **88.4% ± 3.52 percentage points (sample SD)**. Results from the same external cohort had been examined previously; this revision benchmark remains exploratory.

Source PNGs were prepared by M.J. using Windows Snipping Tool to outline the complete examined spinal segment with surrounding tissues, before the radiologists assigned labels. The computational pipeline preserves the complete prepared PNG field: aspect-preserving bilinear resizing and centered black padding to 640 × 640. Source preparation is distinct from automatic model transforms. No additional manual brightness/contrast adjustment or CLAHE was applied.

The authoritative cleaned archive contains 824 unique PNGs. The executable source also recognizes the older 827-entry archive and removes only the three documented duplicate training copies, yielding the same 824-image manifest. The 827-entry input manifest is provenance required by the unchanged preparation code, not the sample size of the reported study.

For a **new reproduction**, change only the `PROJECT` output directory in section 2 to a new, empty directory before running; retain the dataset split and scientific settings. For inspection of an existing completed run, retain its directory and let the guards verify its frozen artifacts. Do not delete evaluation locks to resume fitting. Executing all cells in a new directory starts actual GPU training.

The original AMP preflight repair and corrected batch-consistent Eigen-CAM replay are retained. They do not authorize changing the scientific settings or selecting a seed from test performance. Source PNGs are available from the corresponding author on reasonable request.


## 1. Środowisko

W Colab wybierz **Środowisko wykonawcze → Zmień typ środowiska → GPU**. Dla batch=100 i 640 px wybierz A100, jeśli jest dostępny. Test techniczny sprawdzi pełny batch każdej architektury przed długimi przebiegami. Nie zmniejsza automatycznie batcha ani rozdzielczości.

Wykonuj komórki kolejno. Procesy obliczeniowe są uruchamiane osobno. PyTorch pochodzi z oficjalnego indeksu CUDA 12.8. Dokładne wersje i GPU zostają zapisane. Przy wznowieniu użyj tego samego środowiska i katalogu projektu.

Instalator przypina także wariant CUDA: `torch==2.11.0+cu128` i `torchvision==0.26.0+cu128`, sprawdzane od razu po instalacji w świeżym procesie. Sam numer publiczny 2.11.0 nie rozróżnia buildów CUDA.


In [ ]:
import sys, subprocess, os
assert sys.version_info >= (3, 11), "Wymagany Python co najmniej 3.11"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "torch==2.11.0+cu128", "torchvision==0.26.0+cu128",
    "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "torch==2.11.0+cu128", "torchvision==0.26.0+cu128",
    "ultralytics==8.4.163", "numpy==2.2.6", "scipy==1.15.3",
    "pandas==2.2.3", "scikit-learn==1.8.0", "matplotlib==3.10.7",
    "pillow==11.3.0", "opencv-python==5.0.0.93"], check=True)
# Check a fresh process: notebook-kernel imports cannot conceal a different installed build.
subprocess.run([sys.executable, "-c", """
import torch, torchvision, ultralytics
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__,
      'CUDA build:', torch.version.cuda, 'ultralytics:', ultralytics.__version__, flush=True)
assert torch.__version__ == '2.11.0+cu128', f'Oczekiwano torch 2.11.0+cu128; otrzymano {torch.__version__}'
assert torchvision.__version__ == '0.26.0+cu128', f'Oczekiwano torchvision 0.26.0+cu128; otrzymano {torchvision.__version__}'
assert torch.version.cuda == '12.8', f'Oczekiwano CUDA build 12.8; otrzymano {torch.version.cuda}'
assert torch.cuda.is_available(), 'Zainstalowano właściwy build, ale CUDA nie jest dostępna. Sprawdź GPU w Colab.'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
"""], check=True)


## 2. Zapis na Google Drive

Ten nowy katalog jest oddzielony od poprzedniego eksperymentu z innymi transformacjami. Po rozłączeniu wracaj do **tej samej ścieżki poniżej**. Zapis obejmuje wagi, przebiegi, logi, źródłowy ZIP, hashe i wyniki. Nie jest nic publikowane na GitHub ani w Zenodo.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, hashlib, shutil, zipfile, datetime

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/axspa_spine_2026_10_04_full_region_v3')
PROJECT.mkdir(parents=True, exist_ok=True)
CODE = PROJECT / 'frozen_code_ampfix1'
CODE.mkdir(exist_ok=True)
DATA = Path('/content/axspa_spine_full_region_v3_data')
RUNS = PROJECT / 'runs'
ANALYSIS = PROJECT / 'analysis'
CAM = PROJECT / 'eigen_cam'
PREFLIGHT = PROJECT / 'preflight'
PREFLIGHT.mkdir(exist_ok=True)

def sha(path):
    with open(path, 'rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

def run_logged(command, label, cwd=None):
    logs = PROJECT / 'execution_logs'
    logs.mkdir(exist_ok=True)
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    path = logs / f'{stamp}_{label}.log'
    with path.open('w', encoding='utf-8') as log:
        log.write(json.dumps({'command':list(map(str,command)), 'cwd':str(cwd) if cwd else None}) + '\n')
        log.flush()
        process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
        if code:
            raise subprocess.CalledProcessError(code, command)
    return path

print('Projekt:', PROJECT)


## 3. Kod eksperymentu

Zeszyt zawiera wszystkie potrzebne skrypty, wspólne transformacje i historyczny manifest. Sprawdza ich SHA-256 i odmawia nadpisania odmiennego kodu w tym projekcie. Nie zmieniaj ich po rozpoczęciu treningu. Testy statystyczne uruchamiają się przed pracą na danych badania.

Poprawiony kod trafia do osobnego katalogu. Jeśli poprzednia wersja zatrzymała się na kalibracji AMP, sekcja 6 aktualizuje hashe protokołu po przygotowaniu danych i **wyłącznie przed rozpoczęciem przebiegów badania**. Wcześniejsze pliki i logi są zachowane. Nieznana zmiana kodu albo już rozpoczęty trening blokują migrację zamiast nadpisania wyników.


In [ ]:
import base64, zlib
ASSETS = {'eigen_cam.py': {'sha256': 'e159f0bcfede0da4ce3348ca54f444de934b2b015cb42a5f577bc1f76dda78cb', 'zlib_base64': 'eNq1Pe1y28iR//UUE6SuFnBAWLK9eynucuu8Wm/iKsd22d6kcloWCiKGImIQQABQEq3oOfJAebHrj/kEQEqbu9tKLBKY6Znp6envaf72N093Xfv0sqieyupaNPt+U1fPT4IgeLnLiz67LGUs6qYv6iorxaviSlaz85d/Euu6Ff1GikrelHvRt1lRyVw0bbHN2r3467s3787OxLbOZZmcnHzaFJ2A/2Xi7au/iL/vshIg98W1FBlA3XdFl4jXvchr2Ymq7kUrV3XV9e1u1Qvo2tdtsYLRfyqudq0UXycAUIou20pRr9fFqoB3OGLStDIvoA9M7iZrYTpZ14lu1zRlAZCbtr7MLgsYG77FNHt5m636E+gG71ay62AJRdXsetHLqqvbGOaXi66BucIQa5n1MD7M9d21bMtsDwuC6bTyCqYoYWjR1wA1608YRraGpzTMuq2/yErcbOpSzt6//QPA7Rq56mcwcifb66K6AjBd8UXygIAhANxkeQ5vAH9va0Yl4xmexep7J0sAA3sDy5FdP+skTH3HLWCDtlkFuBZdvWtXcgZbcyWdxeKosCmNbAFfW5j+JWwk7hTA3pUyER92lVrEqpRZlcrrrNxlOFzS7BOkkRNY2Vak6XqHmElTUWybuoWlV7CN1LI7OdHP2qsmazupv6+6a/1xk3WbsrjUX/kPPEi2ss/yrM/0m791daU/N2XW48R5DtBK9gVQhHqrvwNi4N8vdSW5HewlDqWbvYev/KLfN4QQfv6y2puJV7tts4cdE1VzcnL+5uXHj+nbl3969VEsxEVQwQyyMohFUAMR1Oui3+MXWGnW7atcdlvzzPu+PPnx9cf3b17+1YH11sB6h7CazR42Fb++d6DZpx/9J8uTk5NcrkW3yZ59/U26LkoZ4mrntMhIzL4XcKTmJwL+y+Ecdz0MqjCfcKcworc3Rb8hRCV1I6swaC+DCNe/AeIsJUPA/5AHrDa76jMcGlEAnYRltr3Ms7lqmbQyy8Oz02cvxBOBf6JYXAZBZCHYuSS7BrcsJHg8jVYCUVX6/Ube8ieYJC80a9tsn6qH9GUOW5RUOX32Fwy0+kdYKiKnAZrIYbcln+7zWd3mQOKXiEQ4KuroE2Ykn/BdBcsC3lPvugSJHgHSGIBAGDDrgFv1/J4e82SiR2DaXztSd5IDtXXhXUAzDeaiLPTqEnoEOAxo+vCOH9O3BJZ6H3uIHf0Hy+rTz3LfLT61OxklsloBFwmjqbkw6L4mtISEokVwHkSP25qbFsghxfU4JBgL5B9yjoeLductnEreHiI27tTL297FBPWJBWwKchjAdVEsfsrKDp4VQP9Vv3j2wLJhs8qyvkmrrOKekfidCH6pAoQKGIBjvwh2/Xr2+0DPX6KcS1fZNkXekIKQKK6ZnYXO5xG59bumlBf2YSzczyialoYcz0sQTrPsqgK+UazEumiBTEB+VquiycrZqgbmU8HyUGz9jZk8SDtx/q9//vFf//yLFUYnBPAcWgIRy2y10a/gaAIbBiFRg7QyQqypu4Imn4iPf/5RdMVVhUJgXdySAFC7+/dd0SI3ROFVAuNG2ZJddnW56+FBneXEKisWbjTztrja9AJlyg46iGuYcd0SNBCKl1KNCxI/5PZ6cjdFBYesYkkMvBrIUryVV6wd2KXTQSRwq7JoGpa1X2Rb0xkmRaQGeKN+uLYOdAfucHEai7NlIlAfIWC0fDi914A+1TqXgMgtSFnaFoRu1JJdh6BJxaApAZcos0tZwiYgsB8lEI1s4fQ8JfUlg83T2yT2hSxzXCjQ4owmvs0aIMBb0E1WRQ86FCo9cMzyRFMI8xnNYxRnsdSnuNgC3q5hS/pvXvDZLIAzAtEVW/GbhXhOakBRhZniHuI7cYbP1PeLsyXwZv3l2RJeP7PsGbQNWPKf8QS+atu6DdfBq1tUXKTVijRBejO7AmzdKaj3gZkXIhEXU5Yh/Cm6NaAZeU0UHRkTxGI145buIN8CNIGa6A1IJo34XI21igWwmxvAnZoEPd1mfVvc0kPYEnwcYkNAwE2UfEpWdbNXfFm1nC3UJ1BEMsDhbdEtgIQ+S9kAfhUbHS6u2ofcyVkUbThtJH7qwhBnFw138PmzyKKBOSw2j+lfNb1Y3HncLsgN1YFAwAmBfAC5CwwUvgeAan3UOr1lQGuaPgOfdQag5uHRTvU5Ton7dgDpYglw6eimzQoet0VWrSSwx4wOA7Q4TU4H0PB0pdBsU7epmgS0Q66Pc8xuUntOU6DRSRjDZtntsNk9fUpjw3xAzmwY2WUB5sVV0l3naktisd6VZUpfQAVW4oBJxjDcBfS/OF269MCLgDdF1SPpAkuEmdCnyy40XaPIUIN5dsF98WSdWoKwoz1ZiNkZC0HLtRZCTVj8l20aGaol6uE+mq3SemFOxXa3DS2kGHGlaZpeQksitlB3xV5aBwCJB+8N0KemE6xIf/xenAqgJmmoOS2Lz9KAU8qErGrY0qwnrPGA0L6DyRkZ8cTsmBodtX1obQncJ26zgIVPAEeoVj+5mH+9BGWGdKnI6XmUoHnWBsQpcMon4hkgxVlchJhxF0uY8ac3fQz4gdNs8kCorTKPE+TlUXSkGx2QcTfa49HK+5t6iLW0kllb7lMQxTlAuqzrMvSOZAkmgdk4oIYzEpLEz1dl3UkXY/ZQgqCJRQtbsDiTs/8E1Vt9PHN4nprfvatiAkWC8EM2GTpsMnYOy+RrJCWlywGaYVYpKBhb0M66sK1vOlarL0gji5GwUomyBp4XxABekDpn21itrW5BTvdzpfV05CAh9RXtaJj0WragVILyhepdIl4psPQV9ZQKepal0trArgdOvC5y7DK7zND/ANMF1vSt0ixEXQG7hmUUFcDkwcv9U9AS9WelFAH4Yl2gXO7huGSJeNmDHQ9TrG1HoXBAU/UmyMtRfPBaK5LZdVaUtMDBumK7EnQaKCy0slTqFyr71pIEOZ3vUM2BYyzICQHjokujYB+E3hmytdbZZUstB3qQErCn4ruFs1/47cUx3cFput0BNkATvZT9jYTZnxLdvlAaA1k4gL4FWUkyJzJBUb9fsFEr2rloLwK9yBQXGSwNv8dTcTd+TzZyi6qyGuA+QrUMW6sHR1WfHw3ePOR2CNB6YwTOVa2DPUIydykcnQvLE22xE2EhgDarrmSIU3GcGq4qtgIEFWgNdgihHa0F1w1L7kHpSAkqrBeYMw+AyIWXqC+PXi7NECgpzSi+X0CvJMlA36/y8O7JE9uUeEtg3F/pVVvvGlR4FKkH94wOtfePnj7szWDSPFc4lR1YhBbc/yk+Fdgj6DyAMcIyjujND5V8h6d5WPUaatR6eGXE7To6DEdJ2oN1bxAC1GgXNXcnPjHT7xcHp3oJauxntzsAHs8GuQKMhvN9xEoBROS1wn5JltObA6fbEKK87RUhttPEZ9hycH8AS577RAPWjrtsLVPijzR8iI7PTvZpW9e99qB4E5wjsydJhS/nCnQJ24bfQ6+tYVTwFCR1qq35MEJrMEiSgEgYXjZZ23dHbcCfK5yqZUk0Z8EzuvNG1bYfKK51eU0U5S0LlCloT4otvg89Y0p3wukaqH3tAbA9o6NTfk1TlN0K1OdOKAiCMTs95YlJkEt1NM5P8PRt3f9U76qch9N9vM3WD9Vms9+4+EIMPM1BXSnKLgTZXXXo1O6sl+y4U4m4PJDB0uoo71BlKCoQ4+iIh08Ua+BIju/dAIV1tbkuOnJ9VEbEb7KiMo5Oco53DegfqZ2ddpN/xOcfKHLxPstj8enDy7cff3r34U/p+w/vPr07f/fGwvAG4+4OQDDiP50oVgvDA6FcAU76vnVwAofOfglisiStwUXdis7xKk7L1HMKTmi0aIcOhrMM7G9R4dmRcgII3FVOiKfm0I8ikirbMjMnDRQstDTFR2nK8TE+/jAvZt9S+U5SbE3dRvhLPtWfVOjpU/JW0Yhc6kXqcYbgUW750B/nxHGCU6OpmImYaQxpRPl5CAnOsaE5UgO0ohFghPvCSO8FBQ2clzj3Md1cBPgO+PtU829enB7bYRulZE+m7NyFQudZU9yi71HH41TwbQZicNZkuaWEQBumWeX74XhKz5bkGDrkx+n6/EAveHOoE1raAJPdVrjY8HlMPBr6jB8qhxOOBKrw6VFl0tn1VsqZ9sB6nEjFfrQDAJ1EOUZSc3ldsN9NocRQm999eiv3dVn7DQND0qH2m1E0R2JwOGSkOoibHu0iwIbBcgKVvpBHRI1HoU14xCDQbnqMo9h+U2c54Brj0gMU58UajlGnuKIlTEN1aNz29aouA0+CMFZ41njklCTheBUIxrS9ugwp9JyuNjdzT0xgX/8JwBnFL+zX+ZTz2cA+QrsD17P2LJ8Si3r+OK704Q8/iPM//kXHvBUbmvIpw4pRpcjQl9TnF/NYeRfx36X4HS178DRKCM8NeirOYvEsFqcP+adhlIhOG3xgBwwo2LMzOXthHqJ7BX0hyenp6dkxsniNcrmTM00SJuMAOGS967silximOFs6MqivyYd7pTUtDDe2Wdf75AEzxtgIzhbWFIszHcwCnYei8zZ0EeKBJA0jVskBqRum6zHi0/P7Q6G1y2z1GZThlPZIgVIDpJ9vAABoMPjlESqM/Wx1GEw9wKNh8jusmK5B/sRqURyjynoVPmrIN4EsDFAprtostwFbrXKgFqIUDcZLntIGxPa7CdosyO0Ntp7yaa55r9JNXX8OOUtCLZ0XTOschDYpBMYh5QU1g7PgGjUkFmGbkdWuQLYraqd5JiyCBwFzJqpPcAAVTf1cadaFUbCZGo1PD004iBwb1F20Nou4S4Jq6GoTRnyq4e+q2cG/lACB38ghHllsKEQdx0eMdA1jTcV9p3HA7Qc4wKPGLwyHefEAXj4q+0oRNFjSe9kCVeZKs6djk4kfkNfocBZjYgphJuKqcKYm82twdimBvaOv3js9iVYvU5W3hCeVcWrJTcUhKBtnoRY07kid3F3hbn27t6jyx0YvUmhXS+Kf8W6UPWoZDolQdruyxyOCvERnXoXMTRZgkIQOY4li8eSJzx14yDUGaEp3coQhWNi2NuagWbj/WPsTPIIm7xoFOL13eu/812oN/HDItIEDYebQUHciRgP2FbpViRs/dZPNXMNBp4xQkBIk+rWskLoVaWEoyrIZfxHk3BrNXfMNWPetJ1VpPYY6R29QYA3ePpsvo6lI74E1Z+TFL6WSVOhzAgV7B/YCIGKYHRc4ngZCLyrumHrXHTLO/FH/RGxrcEi1f3vFGpSXyacG9J4BSoejJ2j0P3BaXXE6yBa8pU1x94KFK5yQHA8gpSaG2pXtxxeYU3h+nLwpKNZg4wxow1vxR1ABOCCWqWybNYKsFjCVexKNaNvIDveHXAs7xAwFxdfZZyl+fP+aHDSXssQcwoEU3GZ9U9aYYqfD3eprsgO1KHh5daW3cdg+afb4Ce31puyZrZH/i6TlhSTLVJL/T7v1gRTkxchdRr7Nz3K/HGgY2B8eI4TQeHNj17cWLQej0ifqyJ+gq3oJQ9Mn5cQFLG0r7IL6GnIAehlNddX8iTJOF7jWRO0x/EFjcRFqcE/EiwT0LQMPGMoT8Sz5z1NQQE+T338N3A8kDxDBIuC0VsqV1XGHHXN0Bo6eSMBPkaNvInQgxnry3KuUfS/JEU1pd0F2uQJivNoooMoLm2KCFDAaszQ4tFuKpmqwg1w+GkL30sGicb+BKCjAjMQl0FIunHFdeGDz7i7NyljxvgHeAcLiNDl9Hg1CjOvexwl0RrILaSxM49F+asNKKP/oeJ+z5XiYpNh2m/pGn9yLgEUuKNFo8xWYVdXUJbGdRYChUdm56pQZegRHO4oeCwU3ABNMiPBxZjEDHiCbhCE0S9ADih9SUJHD4bp6JqoKc+kUrfhNzDSbrOJAg+rlh2B6YHdIZTruGdBh1n0nj7Q5sxwKB4NHPxjjHtfQF30JhyoI73gC95G443Hv5zbu+Et15+XLXuhJfGXDJV8tl/fBsVTA4Jfq5arHjGhXNwauDvYUHenfw0mtV4sAJze5ye6EjZdpro0TmR+epg0pPThNsYaJNhh8cXo7wuir5QGw8+T5+t5bj8vF4AA2PHeU7GXdYr7C3qb0/1K9lTeTGfwBJ7Qo5olqBW7t5BKmQc/xmsA0aGe2Z88196qZfEEQ8XYNHXXAWDlHfqZy5Hkrk1+qIWKHaZbvz19i4h0oakVZ7ihWjn4uLaviX8g3vMp2HQyLyXlZxW6vZAj5ESgJfqhBm+aIt0DuQHKb7i085irA/9sixcE1Dunllkgr5B0ZE1aD9wyafM3MX+s44ilQsJNBS9DSO1B57uH/CXQK4kH7UfMEoAbeZLJrCR9DGhLALOD/MCGQH7ALdbsIbjYgCgcL0H3y9cN9UMJz/gr39pTBCxg2QacbLZc+aQVQJyilZgnhk1hMxe5uJPKQTn817kZ+nnJiOMX0DnpdxrkrE7plsb3qvpB2eRAOunNXksZC7n4aWI0Uvn9zenqwp806SK1Nw64e8Q9S7wEAudsOQdB3WNJ2V/GU/Y6kEiM8m3QDejAmf2LAdD1D7xffNvIuEuFtJXapZiXeO9jrqWJODeyFzpT+AMeQHWiYxuOF/mI+q2mRx+oWRSyseGH/lvo8YXKEGZyPXTsDobjFxD7cJRTm7o0RyjyIMAEZjiNYPSqn2UnlWNUbUrhVsgoalZjUU9urPhj+xwN8Kd3koXVRUoDKz5ohfLx//Ubr8RQG/bcMAYJkX+v+dJCQ14x9bKYXmGFtVu6BM5m4IYqAE20nAuXpKH74HEgP6e+oa/3nTlImNQsHOEcI0hpBmCYPcND2BUgI3tqkznFJJGhPfRdGZNBiFMV9iXp1XrQTcWWM976irtpx/Y76Ac22lPW+19EuuW36/beUNp6RBDQt5uLOGWwQclZ84tdEnFUXA8a9A6TfeTFCn+ccwbXxoSkwIq9pkkAKq42yQzlv3Rzspw45462XQUAdWdiRAfFoYMoraIMosmAb8OLh+IQAjgm/vslDSQz+CBwCpGSWQX4HWpXqwNP9LEJGQFFmk4KDd7lsNhB9c30P915+ih5qhnNI8KZNOOm79cIe741bnhZQZqvPHajAKgttCmR07yinOi/GSxtSx4kzkl5w7AIbedlYg0YPTBQjF8Buc2ZynM/o7AoDtdMaumOcMJKaiYvG5aHLDGqFXmsnCvpCRUGn4zZer+jRC3RWZSHs1ZUWH/Mq/Oov9jtx6sZm/ZffizN3yjpndrDA3RYz/c9smuzD2/NqatY4ohYl7gUZumW6w5wMdGT6a/JT3Ac4pAS5ISU9NLWP2TXdDVZGkkqGJVMXHSLowgZ5Sa7nou8mvXt8fQ0dypiheSRRKhaHE7kmmCPDtMtSPGD54Kn9yJPhgBxyOUxnJaaICUUA6itvCl8t7wcJmuj1nchM9nilSeY8ejvGSaDVCbtBZKSrq25pweR7YTkOjc04JmCkNF70wtBCakIBsfj5zacPL9/89dPr84/pn199+Pj63VuPeEZXdxOMd2LMMXD0gIAwPgFrCu+ef3hN0p/updWrz4DGnx3t4m4C4j0oWtdFW1eolDnUhGGLGK+NqHgIekBG6w1d7MVKbV7wH4+mNBBYlbfOB0+tSVWBrSF1Dm2PnNO+9R37geG82sjV56YuzHLQ9Jx7C4PFYOMQ4zBa/mNct/u8CJQvfZjCRGEczisgHxMpV6lJcKIPFwVnfxZ+uiu9i6IlEYAN49HjWEWByRlE9y+4tckXcMcB7Dna8tFkgXODBMVO+BavcwhdyPrw/Va8shfurApD19sdvv9VR4HUbHeFRAPb8dP7589sCJp1bI5jAW4o8EVG1wJ5J32KBsQSA48o1/riqgKsvp5MRddhaoszzNIAW2ar+6FBqz/DsboE7uFeYfqteA1yr6C0gqfrnTonSk220X9DrCocqaOI6GLAmKG6zngwrneU+2qWdXG6HHNhigO68T8vFunk/LljA77pNehe2mXkJf75MB4VYxqkACKWCoO5XLzff0ILZgJliowukfd0zny9ORycKPdSM9RBSH5K4Th8RMlz/OxidrZ0MgqRtZzrs/vIyN2Qc/hhNMYYzFvd5j33OYOKmi+Ens6zpfPcZw3EFthBwnkACM+PNONb+rxDWUcXp3TTTsFcuoFdd5jHhGjP613JQcNdVfx9h7FJtI17WBDtsFqOjlfSoryV0lAmuM4D66Creoa7g0GG4I4TMekp7hGvJE3vk+EbdFLwDiqvr5PGZTK+DDKn04IHx8EmqurQJ/IdxyPtJTwAK3RWdC/CO2cx91HgwghUZRPiCXU118zjYQesSjrp8GbNJSOcD4EJcxnTevsZDesGelc936FFZxiwoLT+7FyppchEl0LbkXuRoxZdMGinIOvky6xlNUuXAkmq+ibU1UCSXb+KQP2uEY8YCVbuvoquhsTKqnVSgLSRSXImNpccTGRMM76YR16cORr7/4nmOgxIx+5taJtDMEzz0hleOq0rFgNm5fJjN7Pzf2e04aUOTnSkfGHnDqhzcda3egb3+L7BS3hmNlE0MlIGvdleOWCi+FbihM1l/bGO0WVvLE7bXwPNlNQ4kyLtpXkqZdUxLZGWpg0FETxUz2IdhLBW8Q/Yvz4TzT8Wdz6658k3V/fRt3QJoOsLMPow9b9WGRVDU52TKP000tvY8ik3Fwo5Fc4VMOPdO41Fs8pSdYH3cdUz3LvuHcldj+DIVZkgAJU7bcam6yLQwb+WinmPpqZBLOy30yWoHQztg+zQUsKEiB9ev3n99tXLD0cv4f9W/Ldsa1vNATa4g9lu8abBriLfp8wp5wJOfKEyFe0SqfjEpr6pEgNxIzMKLuN51a7TiwDO7WUdLEOFiOgiSZJYzJ8vJ+62GlAqwIypLrBWzjMjX4naiAv39vSSFfDwNPnmVDyhLcfkhBf4Rc/JsbR7SffEmc+md8Tz5qfP8vuUSVa7zb5aXszPnrnxzOG2wcxJsuCIT8Szr7+OnDXt4M3vo4gCNaHD7zGAdIeTuFdpihgzih4cQ2Pk3xlH9R2MhFfWoc+XFD2SLRUIO9CfZtIlVfMlOF6OhrguL2sBp0ypI+4hWbilNI7CojO4co7hYnAqj/a2J2rhHmz8rAhxof4eh+Px0YXvMPITC3njqKiUdrxkaLkVV5gHOLj+yemDpG0s2HLUDflOicM+UFR7hQP43r1JTyD6DeZadk9c3WPnzzCfYViLwhPN80l57fqUVQvzfQhPL5F9TnPfAxXb97Da9KbI+026IVseCxvYVwOgjh94PvYMa+c2WQLQwDG3L0aNh/N1BOr8mATWPvOpQY50Gw/nahvzgRPYVHMQgR+T9PpooT5R/IHGUDUlnH57R46y/LLguQiF02AAzYuqOKP74ZbhhrE2Tro69HF09dh/maqqX476PlyN5S0pCT+8rKFriN3a+mGDdjxZr5za7RBRlieNITv8yo7hd5gYwlUFhju/ynC/lSAbvORpI5+eawnVPR1JCyzIxyz9QFOX4U/UnGF+niI/H/edZPb3A4Zk7xzTV5dfsZHhXGyHoxcLJ/VsLugWh0kim2tpHz9wLO51AkSpkmDd7FRr3TjmFEX+nRgaTdbL/vuCl0p4SbGZvBuUpHduKtkwu4y9lNYNS7E4LjWKC1CFHBN+kmqHMfER9c5rpj6Ekb1z3oBNg554yhp1HbB4htCZYz7wDVT8Skm/VF6xwPpw+MkGuN0qhc7ML9RAuMojvm7VShuV5E33CttwhhsXLOBYJw6vq6Hic0zGklPJVKZmKsa5j5VfdWgzWFEKRZ6C1YvSg23j2I6u3zzOVnaL26zaonG4iBtcoVvoKX1OU6/6jPJJ0yvkdNBRR70V49MtHs1L2a/NjhRf5qCEcEIAc3c/Y7fWEDpqCTttSJmTvv8KS3pu8fo0RmATFQNwijEGSkN30W5t8RTM/B4MD5wbW9xOM4/LNyCpVnjkg59Q2RF93cxKeY3VXl3XFZDfFoDDfgEBKE8uel20Ew/VerBN1N1C1QCIf2fvYB6Ywb8vhAKvuCy0uptSeHBp7/S9rtK5JnnuuSffa39XzClzXlFfUBs7iiqqyxUORxzpQ9pZluLkkLXCnyMuNfeWuSGnwZPBGL7Xji6lzvkWr6Ol+I3wUumc7vQe0Ey09FrVwFyxuBTNI3ATWGdurWJ1N8h66n5FiWH2FLRY08u7qWqiUHnRNSiHhkoB9ObK0CmXGk6VXrpq6wbL6YLmncpqg+EgPG0gn1OsMKzcpbCeiQhIoKIjjKest9G4cXun4qmTQgindFPnY/JjDqHTJG25PLwoWKAzMsVqIAdmxTpJoFxTVDwT+K86kjNdVlP865/6koi5YK1Lz62oRieyadkOi3QGY91X2XA46Lnu6Xr8VBMKFlLh6aOlOIOp2ny28iWO8kbV9SRT7YoEBkAzZT/BiqmKfpeTv34Lp9ZcE/9W+DU9qXBLZ0t6BseOC62vLBoxVb1TFfnEIOI13oy9tPeNlOb+rallSCmv2juDvUZqnfH+4Jg/AL/ATHw/Ox9H5Hl7ZysHSVh1dP19iMct6FAbBfNtLf6Q7YCHoefMJWOqxk0EOjMdvuXjdannoc6YcKY5zRJwJPIYsQsJ1w2g+NZwLF6c/gd5dPDp78Q38K0YXzn28t8HJSCLbaFKd+NAr90MYk20sAWYl0U1OFX8dYbcBnn3VGIxpdVbtkJpNhYvsN9wkrKrDCPH6DBr+xWgvgSkVD4W3ONujPWJc04ZYmQco4mCN/0wsyI2dxOA8HekCWCizUTJpWHdKlPLh+EM7ZHHQP3Nr4TaMn8cFpwr5W2xqoFhNRtAted7OF6ATtUXhiYPVXXjE4SMwB+MCo/FpgojpoHFfm03quI8WdTObFYyOj8qbGKQxxczydogM/KyQ8HBVe3UGeOkG1spDoSexIu8usydTgjgElRcEQ9rWHaCoR2gqMnUZlQXp54j+Lv72ISi5sKYSIHKeCcjDK1H/OuWNnSqYx9Mg7cjJdgwiJUZwSbFsJ8aEUNAiOT+tg8it572yfH7Gb5e6FSmOGZelLNV2TmxisDmpYgQtQ5mCHQ/UdNZJ+HraZSIc12+0C1d6BUCxOGDQb1KtfMgBabPARdiRPmFobmjJQwTF/hrW05xcj5KrkrSpWB5D1VHdEAPjhRX79s1KGfwBJkxKIpw+NxQlWOqVeYCx1zZJqP8UCZDI0ScxA/GGQd09Y9hqJ+qUDFapTpi7p0DPNv1NagugN5f97sVNiSsdWYPIbo6A3bg2+YbI1hmJMqUY8bt9CMvi2jQzW1EIFooubrRDeyHqC/VPQFVqNyzIhzgDZED4pI0CE18j6nITjY4F0nvesdSc6AP7kcr480x2AZJGWi20Wz2NJOVf7XHAezefrD3fEjaHrjfE2vJK7FoIBXicuWyT7jIPq+ZGAHIqtzRno6kcyJ0JiTnWhSgasTqxxNi9/dJHNBkSMYDW4MIqF73N7hqlfigfl9F5ZerbcxWuBNZtadb9njtA69ExY63ZljT37/7q71SE3GAWDyQIe457EdREde17vvAY/HkYh00KdXKug9MtszgfshyDNJ3KQ+M8WDalzvpfR2Dto7UgbM0HrlAl7aa6xGRRSwoWXXXIHz4l0PQqYYyBI7hIpj6vYWpnxUhuYVJHwAp+RHs87/Qg5DbxXAwZZmTGb7wLkXbviz68CTJ1rkba/2bXOuQ5LUfdVrD7qOn7rPca4l+gZfU3WvpmpSKtX7CDe9HkPSPadw5ez+3/lI/m8IkV8aGNGx2iVtR9N6/meouGTTJEAd2k26Q5u/Qp0XqmVtI0dnJKJqPfjgmGlIMO1oxS7vSVXLdvJ72qqwvw+BJEHEcGoczd1ge0nwogA5UmzLr0EqPWoKf58T55HeupnivVBXzQwCOcmPEOMi8qWs3pkAj6lfqVh+lAnPSRqqn1k1W5LQZwvbhkXCpo0zaDk6RJFUeAqg+9u75+WWoX65WsunZgDQuXDe7RIkl/lkPrbcpYY+MlvQwUxVCg2A7YjGcJO2Pcjqnui3vkNcdetIPtyDyuCq+AUo/BcS66IgHmARJ0yUBDhdqHzkl9Fg3+bCwm99HLTU1KlmqstAICDr9lOL6q+EgyhjM8ctp5we2Y0p55m1Aw0Tf89EkucNSeh5p4Sa488fv2D89DUzxfKRPvYcI4inakdXUlunW/pZ5MB63ZSb0cWTLqAqN1+6xmzKAbrdg/NJ1/0eDTPLjYE1u7yh7/+hGv/dOld4ri9nRRbkDNhHskLljKeWXY4fQecRtnR3lB/5+OgAftZsKKGHFEB/qj1qJIBT5J2W63dE6lE4SHA34VDOXmQJNl5gAnP4RLg64B5PXG0eHxJsQI8ib9sX0lJdHLyXq3aVZcTG+HF2sqBzb9ABzVRZDLn6Jb14ppouQwAqWqto3nwSunOEdb1U/nWieXtvyT3TpgkexBa34O6XdW0PcLA6bacqcbDA8Ig44l4V4AN1t8FhH9KiD4/gLLP+lvHR9hB5AtLMM4wFQta+n1upU65/oebgo9lTjh4pk6xrZtu/Mc1JYWkaVxs7WBtBAIQzoKglM+xE1bzVSu36Xa55EN/92/nUpW/3Wj9VpDcvEKw9zoQNSiM6exJ9TsT+RRcaAFjgtsFQNdjBQwtngBoRq7QssfDKSVwbooxichssSY3QKogePyfH6uuPLynraShhcyrLmgp7kkNBXAUc3urgK7wBHZFL9amsKLTl0MOskOm1UfWDriJvrn/Xjdk62O/Tmu14MxE36sHXmswcuXLKirirLR6Y2rHmkqtXR1gf+mzNzTXnyYthDhY4B4NPvSDP4/ul3ZMl/jwk/7s0e8sg5KZ/TBR/UTz/RUl2/fhTb5+aCq3npIsH2GignCdXHCt01P3jZ94PvcfRzzWmds3Vd5vpnXvwLvoNfosSBMzhDdVfchuZHIc3RcX/dkaIYmZfS+OClavI+WueQ/TU9VozQ6agE5hSD1FRpcqXGSZje9Afpl9mR9EsnBmESMDM/AdPLpXQJw0+FHNYFGY8wzNm64NsQOJ7jGlhGBx1D95H5nUz+8ZAjP8uB+PILzqMHoEkO/YZC5OBvNP8DE2JfAKVbDX7PQZ3ip9QrSpRPgPOq7937XbQW1l++PtW34QCQemF+J0e1o8tFvdNLL+6xxSTi0b0w9vErpq3KSoClhtNRRZvGllnVfDkkw5Bpp91uDUZ3GFByoMPM8dfoQGaFun+sfpS0KVafS32LE/k3e9288rj8Y6zdhWfj2N8zO2bsKG1OA7BnweuN5GRfTVHSAXj6rIygGTXx8bD2KZ6wCVheHPVheBOV3vUQo/tLzn2Ti3b0fjBYFP2aIgznH/9MOvzb9/+NUZnsCgwAW6pURxEpEVL5GtgUmPsKFrmkPR/EXExWZxp7aKaT8jy3TDQJaWRmTkNyrMtpOENtMAXdYxrU4Cg9EhyepUlw+pAdgaNVx8n+rl45DWPKJpofMYimJzJlh84fNFSn90zxsVRj6bqod125p+oSVG90PjDcH+4xPWs4X5SZ8BGdV/T7U1RWm368jdyNcF6y/BrJm4Kal/jjwXqwGbm8bO6G91t/VGNB442uQ3axc1iUaxZDqXCir67ntnDuRAUx98el206q6uv0OXnZXu0wN+c9vQlzyVmseIEmTfN6laaR05OKiWaqSxjMZkrizVDi4QHlJ/orQOsXgSsV4SFdMdM/4cQChW++HnQVb2TZLLA4EJpGuSpRoH+4SNVAU1HWonpKob2n/GPIqPQGRxeg6Wr2wOyOArFc4n8Fhh3yDi7Vg7xoDTIdp/2/O4zK7kWI62xXYi3eXZ7NT48jCmuHqQGxnJ76ccZucaHrlC0tQKxapn4anspCKJD0B4F2RLTGv3uM0nUtklEAwtZDB3iJf3uYHnneMPVsYMLzDCbKFrrgFhMDqJTsBXuWpz0sh+sGTqZvL44a24yYBWHnEFyHNHjO3tUGrsrxgHvlgSqJChWqjAeWkeQHTfHriiQu7MfYq8qDCMXf6gBNzxR7QHdgmiKrS1NV7IH4XnTyP6wyjCs='}, 'evaluate_models.py': {'sha256': '50f37e547b741f97295df291c64225a9b1d0acf96e186c6be3f7736be993e2b4', 'zlib_base64': 'eNrNPWlz20h23/UrerGVBPCAsChbc1DLqTi2N+Wq3bFrZjZHMQwKBEEREQhgAVAS7dL+9ryjTwCkOLuzqUzVWAS6+3X369fv7sZvf/Ny3zYvV3n5MivvRX3otlX56sLzvN831eesnDT7UmT3SbFPurwqxaZq4P99I8pql5dJIdoa/0yaZJ1Xt01Sb0VaJG2btdHFxQ+V2FXrrBCbvOvy8lZA427bZO22KtbmZQkvM5E9dlmDELus7USbdaJK033Tim3WZNHFm6IQTfbnPRRma9E1SV5iYxheK3Z7aLHKRFrt6iLrMvgN48wAbN6KusFx7UTbJU0Ho/pQrrM6g3/KTuS75DabFNk9jHFVVV0LcGsBw9/vshYGlnENUWeNyK12NSAD/t5cyB9WYZqZ4Wyz9A4Gm9zCYHFOgDcobrK0atatWB1o3nl5D3PKb5OuamB4gPmLTVPtRBxv9t2+yeIYBlFXTSeSsqw6Wob24kK9a27rpGkz9Zy29+rnNmm3Rb5Sj//TVqX6vUu6rfpdtepXXSQd4G3H/adVUWQp9aYGsM42yb7o1nnacZ110mVdvst0BfkcCvz3MyCQ6wGacCiq2ifsXvVa7nf1AXAuyport2leHyJYrk53vMq6JBRApNUOqUPWuyuypCmjXdY1earrNlUaJ/s0bgHLMBB8BDK6zy4u3v7hzU8/vf9JzMXCK2GeSeGFwquAoCqgxQM+AC6T9gDr2O70O+d5efHHj+/e/4GBHKqimk6LSVq0WBEfr77Wj0DnZdZdX+LvbLPJU6QUeBOvLgHMhz+++df38fv/+Pn9Dz99+PgDAvziRXV56z1dXFwApkW7Ta6uv443eZH5iMBgdiHgvy3UlEsbcRU/oIKHvNuKCqiQauMAVl6AiN1wQ/wPt++qqNI7IDuRw4bzi2S3WiczsYmaLFn708ur1+KFwD8BYNzzAtOYeo/2Na6yT1C44yYDOi2haJs9rvNbWCAYEc8hhSUqYyQ9H3lIJqHlG5G3uCcS2C9cEgokK6s3CfUL7En/Lpg5oAKayF0o7nEe1D6C2exaP3g63oFf5G0HpLkHJhECuUXlOmma5BAMe10MezNdLU90AVDzsstusyYcAQtFEg+nQWyKKkHeGAr6NQKJ3ktYCAd3dJS3G+CKXabeZ0WbiR9wF1otqUwuz0MDtXmWTDOwtInsDDcp010EuwJIN9rdrfPG54d2/nODo80eAadxdUePQb8hw++AtfvYSbSGvd76Fm6pv5DYZ9nNrwBe2SLPS4AJ5PPfJzCBUCRFUT3EZVLyi0B8Jbz/Kj2snFZrQNPc23ebybde4EwLWKGcVVM9tHJW+BM2EFKCT6/VQgBrpcI+pv82XAz25AMMu8weirzM5t7YFHoblqbSwIhhNtE72CH/Ti/8DZBGnhXrMgFJNdfTWVwug6DXmFdhC7sb2gUOK4AmSNXuvPstodReMngMFKIblEY+cHCgnKxUKLYpFMrESypEJMNfJkou9GBN9aJlj0kKvBF5fJ4UfrtP06xtYbuCzCtCEEflJif5Or+MvruWfYG0/PmhmrRQshZvi6oGST35BGIBRirefkDKSos9Ihjk+x73+0GkCWonKGfl0lMPYj4Xl0Pu46Fshs2VeTPaSbCEaR4DPTrP2/x2a17AnLMGxTm8wh0vpwJ07sEESG/iwkt4A9JrW63hwesN32NOlhT1NoH1n4qJhQMqglFAwWV0iZOQndA0GMcoNKO63vgM4qWA7eViFUCqZl+JKZMGTgW7G0DlFi5kHNQQOgIb9OCICgetTAuq7UtuGNiI5hrwO3DQza/xIfglWMdC3ccp9EsRBkjft6AG+YcQlMkMxVS2hl0cS0V3/tolfODfQMUpkFuHvDxpSchga+g5AOGqWwKirAoWbKwYChhykZW33XZuWrx4Ia6CCFXopM58/d4ajtpPq6TNpRbU8tCrVbLKixw0VsUN+69h4YcjDt1BWtWVeCJYevgAxKkVgY66Sx79BNjjXFJZukOWNopZrtDVPJZ1ntz66S6AAXQHmLHVYbuvUeELUV/Pb0vqON1FoLurrkLn+VKRYAoCpVXg72FD+V0dGnDVvpujDN4Xhf+a9YSkBFgPaIjMZTXxG9hnDHAzdWBdwQpb8GCN1fieA+0PmwRWPwmaQ0l64N7AUgGdATADOwaoxD9IxG5Rl4AqGtg/W8BUVSIjqn+X1DWyF+bkuouJBBQy36Hf3AGwuC2IShpDdsj818HiwAoRED2w1/gQql+4hP5BfD9HkpXrxwTl0MdiFoqr2dJeOJ7vPsVVWjB0FFcpCaukvMVuDa8Gw+4267gzYFSp05muheAimC1Yab5jI/jcfmRU6ZKkFuIMZrsvczA/Ze0gwK6umB/KdaSufivegDKWNpVQLA50PAGiJwPdDGgUlroES+4gqg2Zf2RMA/G3dZbmINLXyn7GZsmqRWWDyZ12MpoJekqeWi/gX+on8LR9+ZABU4S9FKcVdBfTIkMd+hua5qukwIVdxxYcKbZrMKsSEPa8WQJilTireDMdVNpModxA5XrVfYMoHlTGZXCqS2Lhuu7CDClqZD1UpZEVsQcFdmtOmI1XTZ41g3HBXyRAl8EB7TO9844BVsP06c4XVSOc7jjgkxDHMHEEjN5UE73TXBhPFpkoK+3Lxvsizd5FunySqzeF38MN9RScAcAsKy7kKThSGjI4JZOUjyVGI6kBU2QomlCU6Xrzq8tL0JLaLFvPX19Np98oUQsKHbEg6HRd7SLpmYjhvY+VpUiqoBuod1oU8qwTdB3R9kKVegZsh2aGDzg5AvWkOVFsJmyN1uJJ+SMy1fJWWYOtD9Ng5gvTyT9nc34w/Envb2e0tjgOFvnj8rg0pmJXv8fRh2jx4XDlwko72dX30QCqHfuxV8HC0QKhLhUr1YZooJa9hYVAPBps8TAMP5ToHA4FCQKaOh3pwqIKQTvlZf/zPik7dIxgi1AsQAm+ug5BF/7um2tm2gSKeIEvmYHDpvG/NSxMiTojyktYxMLHNcGGYIWVa6xfd7V88TvQibPJ9HVgmQg4Ux4keW8snVb9tLVYHL+lvW5zR/20XZGghqYZzc/sGPQiwUjyddxkdZGnALxFoGrE5HCS7lG3ikWgUMlsQdwnUIx/SEdWyIizXZ030LyI12BXNvlqj15AqGrqkJRJs6ZLYB0PMRjPOGvvDZh3qrHlUE3JcwkIu8/g/b6sQKw19yDqsqYB8gDTGF2/bQSi01oTxSKQclA4giA6iKRD7GyylP+iF4/UFBCW2EV2z/YRSlc9vBsAhRA6tjGFsjFZ0k4e8tZ0BoYhG6p6GExD3g+VIPRb0zJYvhHshNRgjMBneDg2Q5FoJ0pHleSSTEvK4thWVZvFUD/W3vJjGrzRfRxOMVS6DG93lfweCzmiix3XgkA/vbLM5gSR+W/oYnqPa+t7/4ZI49iBcf1LglhleqOsZ4DYbitFm7htqn0N+k+TkdM/bxCRUt2uGzAu6R8FD3mG9vNqdU46gNdNJcXNDiwM6Ep6kaS29oEcBBnpYknxkBzaSZndwoDvM6CCNEf7BH0yCSDgNpccrKqRNtCrQIwM7STBrBP53Y4VNmd4gDlS8slB2OblxjfFbGGV2WOXbNC3QwOP0GTiEqgd2LPloR+qPbpT5ogL0AkALfQamsW6iJkzPzJAaSUAe8NVURbfpki6sio/Z03FA0wLoEDZMLRgggbUVcUcOSFoQw3+vlQwt9AEOzXAF0hnbPyZ0S9M+TJYul4BXUurP3Y76mDJBn2T3xNRxS3swQ55zz2781NyWTG/AlX0Md+BsAUmV7awePd5dwDrSuraKT5NxBSbAfXHzb4gLoYMGmNQhl6TXQVrTdAS8unBeoNuXhV7WHFAAlADMh1Ci0fjY8LRIHE/ffL70QWYyiffeUEb12CBtG6Fe4BjLwQOGRR8hcyYHA7SvYF71aCZtHcNFPi+jOvENgvPyxhDK3GaAwzaIdoBAjt4RfZhOa6sAXedTy/Pcymc8CZIMtI+roEfYeBEqBqwT7qSHFTCP+V0QNPgMOSHK5CIq0O/oxEmGI4asqMKD8iGLNmR5ep7HfCeIlllSJpmZqE9dIStVP9d1dSwRtUtBp54cHKLSKgRRklB6/I3HmritmrujRqw4ahRHAy19qCnrUHnUrag6si9W3y+eoC1z2iRFktUvC6NoU3uhl1WSi/BDhjjjkwr8n0hufTcAfwSt6Krvq4cLRvbuUrpLmnvoA+7Q1DknCrIkHBHYFVaUEvjJneGUkXmfQc0uQkJAwtsvJStYbQuBDL/as1slb1Wc6uAlNGyF4qx0agU6S9IBLD3VrBZQWEkE3AlXkrkoMKF7kl46a/IW2sXYbsylKaooQK05/kVxjnlTOMNWPpSoVPvFkZ3XdoqqymXr5aOCusU07vlUzCwK4aGBNLNV0Ablj8K2akvUTkZHdcJxRuxQM4DJEoPoGvxAb9ZXpB05kmDRcuIexKgWiQFKIBr1jww5A+2RDhd3sDANt2ExCBoxw26UyYYxwEqSbO6E6A5djK4ADxueoN6L0gTAoKqTV1UoNSub8T7t++FcscYmWHxVFaCvePqoMy+yEaYr2HqoCNUY5azq+47NvSv6fyV+uEh4PgDKvUONyZ3NeqKvrKBbQfeUGt8/4j6OmAMtBLGLHHRluyyHx5f98B7ThjPtmRdCROBDeEHw+EBi4LXvxMTkODfjhSj5iS+F1PceFDj1Mg/lGwmWLuQQ64jo4TRsKLVQ5Ytd6ZG5brWGtepAbxFo6a3suuKugTAoK2gA+oGnkULBmaJZhknQ+SfmR4xZ4ZNK6N0/7rOfcSAoVTLaaKMMLbHf5mrqE/r3BXy3fEOpJ/HMdtlCP75EJ+x29nHpGDcWObh2w/IT8Bm3VQ9D2+TbcAUQBUHByouvSfXW7MnH8lpr1Uv30FNbGF8wstxmeZba0lKkZJrtkMKCIDjST0HfKgdYpmMt2W+1EFsh3wtpW66W6TkRr8Rm9J6J4UpcPquxjKrPmktbnGnmkoWg6/R4inJ7DFyQQ/aiFR6ROFIqPW6Gn5jcMbbkCRAmbnBdxt81+G7zvIaD//zLEMC3TwuehEyDO0rAIxSxzIzRupC3zCxr3AGJ3us6/tTPdUU+xyto3soT/dATmFDQSgfAV3kLEZ5b1y+I3VUoRL8yjUNK8YKs/1WUh4ymedVbscSIvm2sOzD5YVDaNJ7P+znH42z3CZCWf8vpxtIf8PzffzF7qQ7p5O/9HqRfg/UZuqKKCyLyQkCaFdKPiAI11p6JwbFsysspjWCrjhREDnUjzo1M0+FsTDYx0JJRigbEpEWIDLRcZc8/lS/ERh5LWEoILds8xG3iKt19EjrzE3mBK8G1A1Ui3UkSxp08ffZhb/SXntyfRql9NIeKHmBIlCUtGNWDt8raRmDFGnyx7hr9lmMCm2s2TVY+MV+R5puumMZhHmP9r7EKSgmCO/171CMWJgzSXTYvVFHEfoJg19mYIBUqsmx8d31P7APjLyoKPJgKmtStYGSMGE4Rz2Yc2FZVqIKZ2UUp9UWw9Tk8wdqvOH8X+N7QdOTdOcWYD3iD3SSsTOWIRhvUCR+QoF6nzS50r1Am2kyDITD+zZDD0yXFYdI/LzN5IClcgTkiB4UknOctawdJU7e8kTKenxvOVBol1DoLnasjUHkkU0yHzhDnAq/jtPJIU6D/74CzJKpC7Z0dMV7kjMK1uiiAGm7glmjW1umVws37ghgex29fB2SVxPrYUOKNIP6MrnPmnbfThrE879gTRkasgZwGV1z0iPpsthykg71ykjYIUs5MZiS4mAwM/WzN8FpJN4x2ol2McHgln2wQD73GPsg+jFTkq5rpB2az8f7H+UrCqhTtBL9sdu87SqOP+ySct+mTV53WgFXLq1Ngp6uagfkkLfoRLSULMWkpT5zVJnFKiyEOH1MbnKgX9y6pUlGmcI7bGO/u8R6NT8jo8XZ+jrNg5y6UgeS+VA3JrNDwv9nBVTWMDkd0i3M1b6S1U6lqpA/8XRuCsI0qSiaubkJCKdyD3SCyXhuyXP5BWrl6gRjAdba0YrEWqPGhwroqDkaRv7mm8vp1JjAmFd+2kdpgAfK2pn2q1CXVjGvZCzdfNCF+onuKtK7qVv4K2uSoeCoDbo1aQvqARWM1Ujtv9jVrdpMEDkS+Dohj5zs6ysJhtV97b7SOfY+1wutxujmuw6i2mQdW4DJwTWVbkBi9DGgln8ggo9vOUwAPl66GsTn73Rw/k6HkuPk6ebMpIC/Mnx/OnYPU21PzZIj9wlF6MNnaq2GcXyaqESB605jaxFD5ou7JWZbruDvM6F9TMPXMfY7Hcm3EtPX+UaSfOtG8u+yQ8iBTDOgYRj//1uk3prOAmbgRuvN9o53OUgMuTAyWVRSFzebKHqmx+di+79GtN7yMxDjE+ekCfwdAvvvrHi4DvDrYPe6ymT4ve2SVZG3W0BUvS+kf/PPe1DPCg7LN8LSmMYi7ehg6WnQZoEkW7OWKH5oqvIWxY9kV1ZtKrLryuao+arahoPFiGFcAZvhebu0zHZJE7O+3z1UMeWax2jP1NRekxY11U9KYIHKuIuT9f/sgaUy51RRDPnUC8fzS8e5CoOhIwAcTW1Bl9WQWHoQdPLNYVImuktpf6hKUmbvSzoxN9fBGjoBkJR3IbI7x6VDPdpeRt0WxZt8CoXTC+wOBIa+fPnK5WNqlPgWAeWlD+IiVLAdxULVjbqKjjaonGZNyZv8dj/mFgul6xzwB6q6noM59oZ+eTwUx+Ej9Rjt28z33tzeKhdtv35UH/AXGiF10UnsFbwsdDhEdncjX597PkR6tfEw2Q/6RNpHPJFWbw/IEODxkxU1Nm9/ct8spea3Aa58n/mAH3bVWYv4W/Em7fawXWswo/AwJcUipMe3wcOLyGCSjhMx1mxiY27Lp3e/h/IdHmIU97B/qiYy0im/jbBH+CtxAPqo9g69urxc1zmdagNFos7nrygmsaoe47wEk7Cdex1qi15wJsCv+wC//tsARvV64z0DABY8Yjc9gDju4R5xik/tqKr0sSMPL8VCHm1CSliaJQID84GtBBijMh9CkLpZvc53TEccUDTQiGumO3u+mKeKHjYYeLtfIeG2OHZSXfyvo29CcQ2KHEWkWzpJCwytSA5oARjS5H2ACuNjBH3DyHwaHibVg0bt/QtuccDdPezk+SX+TR7n07HBUbqTe+5oPJH7mTxv7V5+RD0WlpDGs2jCdDmLpv/w5I11jscH0x1VCgaQYGp0OA0sF8wnegS+sU3mHkrVrMG52Q9pVVTN3HvYgr5DXZnuxff8ICNFL0R0fc3de6siSe885EfACHABplcWT3yM2qzzH8HIvWvnasahOPRfUA1mFnP+Iyv13j3Sj7n3Sfv42RcEtWXJjzr2oEq6vCsyKKgeJhbyNHUL9kiNItd726/lbhuYXO3DJG+zLramgHlNjTzKPH91zUhvRvYtoXyVNH6+Q5qeJ4+gZ2+bvLybR9/a6rdieZYjzYx1fOimJvl9WpVe9tz2wc3z6rnNc85pBTdhD3M0Llyd3U21i7B/SnHvbRidDxePpsGNnWq4wextWfu8cxAXvW2DWPFN10xcsCV5gTEfRfhv/vRWfAHws+jV5imQ+FVtKcweCvnHm0zMDrsFlMBT8aBPgshtUuS7uQ8NcIeo39ElPiq6J7Y6UU4ndDpkFu3j+gwKJfl/7DnGfvz4VhAiWzPwAnXVtV9U6ZxTNDgxwN7d3ypzVVMk4ri6byQUyjZ6LlcTTPz6jAjJxYBUVodRMvl1aPoUsa3QUq8dxnaEQv4Vow6wCa1ABBNK7+jHIQR4DuU4YRKZAzLi1l463DWl3GJ/MdXHJhZORGBpp5ggPepKVoihXwkMgrsMCLXy1LysNNcJuaNhhiZJ1ARPhhj6Jfvg/2ovjKyR3g7uFM7fESO7Yrh2Dgtu+/Q6Da9CTbPTy/B19HUQPsuGUWbc5eVazg6Z8uccpRJ6I5wEvTFiwjo/V/WE6tiREaw/gie7xtLi+pQYBGo+nri2T15LMnaaLXC4ywWnNS1xh0PthQclhrYflT2IPGSxWgxTv5YmhQ4BIdOvVm2/2TArbNDOcus8kMGDYCYuGJUeNuyTstb6vep0sX59zv2yKYyyo1AFgGWEIiBgeDNf0FhCAt7bkGlSE4W8GsklxE5c6Unkik6lUQx+L6Jv+5JP3sIChOyV8y+rxT+V/0TJl/4oCHTnjSEZ9+kBdc+5P7mmJaaBkGryKkTl3OhEvWIPU9Nws0LrtMKrZGDimw3emEPuqdbefd/8HTnOJLp8hUznlcV0/ojBGZNhYuPC8J6PKulS40OxHvp3IELt3aFcATL+l8XqKiBAbpZ9Bi1GPTdVpe4Tcd4pk92tyC6ZAtcPCil+Yrd5KTz9AmtFeBeCpxtBfbrgoqiSdetrMHSnCt9/EWhxTc1B0fI9GcrDzF33nYwQ8/EKGSU+lfP1s7oOicbCATP2GrFHqhV8DY9zzDQpK473U00rOw1XGwEBPWMcFHgQjQNf8+U3lEFHjhmuhp5GVYuvoiCt9zo4maj2M97OZAWC5UGPFu/7GDkWy2Mhdy9VQLFFHQv/MopeB9YEDDaNM5CconycmhL8dBV17xSnv0OX93m1b4sDuvx2KFaglTxdhILl/GWga6Aw4bvLNwcZu+5nftF0TJ9C9dm7DYtj5GqG6gIpJzWLQ9YYD3FWzr36hzrVdeSy9RjivhwhfoaOfhNsFH/Bf5+8Xh50l+B1Lmr/ICDYNfAnVvdyWZvGtnSgB6dxRI6ywRlFg/MfwV7LdxLrG++9wpVFTHQ9EZ4q0jeFYTdqIDNQN/flkxeMzsDdzO7QBhvamoiqyXSFF1jtWyY5T3XsIe259Qi1XI2xXDWcse5CQ/INxWQayL0IczsPOyohVSNCjoWMdtnHOD7k2pitAty8q0BUxHzvlBcMZzOs8pu5JLd+0fLMCbyVuFsTSYEAqcrbFvNYE+1v75C/fc5KM0nV2fjMrGwURahqGqCc6ELPjaQl7D6dOzdzmdoD5Mn6uLAjwH8xGsyYCQgm8+7yFoySdAuzNIBHltFdI0fEjEqYswdiSRprLCMYlzzLZGYy2c+Y5gdBK2sNZuzC0y8iclyDbkDHmob4nEm8hy7vMcX26jmbG+GtEmAb5ZoOTqg1U+9QWwJeDaSFF8tYVRTp66LeGXik/1B88VJOZYr3HWZTqsvyorJ68NV9eRGUBVHeVnS0tfPp6JateIzPQ2sclKFmJo3XI8q4o1wBDj3ST8zK4F94auJZMTgTHMboC1W+x2gk7xBAtCTKQO2kANwBdm+ZblFbx9SwpDRiTmaBPeAZP9i8OZpIZBu1oAjf49FM5LwIgM5g8JBMQo/WBmkxd0mZbzCHgZ5QhKGJwO9ivogLESbly7FKSk/U5XhcD184tdThR7kjXeYwUhM7RlvVamErf2hA6FY2mzWVcDygiz1bDWPEVY3HooZVbU3JGQ1wA/W7r+k4bMB7x6PQKDNBYOICQ4ZMmqkErTQZcz1ab3l+yRVp9pVu6pa0H/m2s406m0oeXgz703HRGZ6E8Mmo5uOjS3SY0c/+tWh8Uo6LMG7Dm1EexqfEWBi0t9S3DsqefoMZBlx1dv3NdyHVnn13HXL92fT68ukkfjfen8pMHYIhsoC/cuVlJ8j06YfitbAbyzjH3Uy/EJPqN14YSaFmVOIxi8X6c3HyNjh7JxDGmqzg/GMssY9lITnxLYSxOufEZ228CEP9JZfWeAsr2Q/yUjx+Q3LoVU8VPaXO0LjyUtMfYANf2XJHDYg6WFwusQtn0fnuAVVhaioYCcm1VIo16mVUgV1Gy2C8yVnTQMcVkCwpLxSWfEkgXrKn6ZmZYRYILPXwKlCaT9JS3uWjH0S0eTJfbZ3AuaPTQhTNQYH1aFq6E1RI1W9M+5FkptFnutPFRHtnoOHdXqa+vMxthPCtux/ejUx9VAnTLBoUOcNqB2oYL75Uu3D4skiNmnfJGcN+S/mqhJm1moHMzRErjMePDFyhLUrWa1/hM7ix8EUlPYyqCjw0qsGDVsmi+JuQQOxN2ihyh3aVJbzsZdrIpcOZ6xpRc1tUK9974QX2XiYEc5oVvWr3G7BlI3LA8Yr3r7LVzNAZHVstA9I4JVvk5cvYQKr67MeQskVtEGMXy4QhrSIpeUOBQQryzfgvanoH5mCor47wtVnfUwtc8ckciZe+k/xzZp3roIR9n4+hWloFsRujXMwpHt7TPYbKBpVbUUvH2h9l1WoGZ7NrVeay7B79G6C6+jiBjUK2eS/L3mPQachUBZOeVYENlLVYGzBLDUHK4rEOR0qB03Pq+bjlM3DsOMKYM/D0wBGtSELyuSd/lOhzB+AMqbef5MD6e+qMEbbJhhLs9r2xnhoiCxvOJCZyoaxxJd0c5FNNYyxGeGfvoz/AbPDsSDFqUq4n0BMwIAL7Uubl2CbkyGCRsxIjPMH9Vbu+cJNeFKUpIzdzpQHMTMN/dgofCK80lGfGjA7wXoof9evEdtGyXGP0d25l/SkNBgFY56dfh8Fs9MA0MFs6a7rvnV4eXKBzWoHQC/pXKBHaFpD+UmR0vWQF4tzIJymXkKsvNC9eLka6H6QzGyCnVBa71hmrfGSlP2nmPuHNpOf4zMIbzm18HnqwMz1SMqdtDj0TIyhA/4iSaXr85BnQu3XGW0m9VEfAlOZKhdKBIY+TqXa495nKZNoaOVasY2c9UCP1AfCLF182Xh1zWp0+wkH1FvmSVY781PHhp6e+yIso1fUuO8z5knrcsTNxXKpptoeK0JlaUGpJAVaL8IXDUliKvMQrVKSCRPmH8pyfdYHIohkMzGS7mWkt7RsT0AJTINi1i2/1G0FOe6sK1lDzPBGKsAkX9GhM19cfx+gpraU0ip96apQZsPoeAXr+QZnaJc1Be5N4DPK6rrn9hQi6bX9oXHI7M3RuumCuyE7BZeiYafKJjzuizWVucX3oXcFjaUi+dCwq5YtaB3wXPCn8dCTQ5+7VEYbhVYQmbN3YSSBWFiPAWzrp2pzCQJeBzQnMonf+orVYGg/9hFtUXm8lp6GwMJMTIm8fx3Hc0wV8+MAu4f4DeUuM3qH8mg9aBfp6VVnZ5G/iUOi4R9yu+23bbu2v19UGL0btAfh+6oCQVwENOs+f65uu3hppSNznVMOhW1IF6SZt/6io/BBNiyyKw3VWQIkYdnsjQ454MR1raoOPwnCIcq1ON6LPF4ZjQWI80kGiyHMdxc69L5hwrbRx+sYLplGo771Eb5rbPfr2PlGJv874qCNmVMbxukrjOLBaouEYJ7KJ700myOAmyOAw9I6axyf+UoLUFawcmiMAlGPvbwKihKnbfpsV9dx7x6yEIj16tIIz6pEbO67SKG3vvZNdVfuu3neTdd78tYNdZ/d5illSksnNvXS/TmaXp/tdoY4wwcVW/dKFgQrGq6vTrdW5nol1SGgMDh5wlC6B5pYypxgg/UGQra+dvvgUmctSDGTxOzEFOKdC3H+C56SD7Yb3H2Hl0fswiT3SXur264NF/Po+GYqIcMYGc8nxZA4a6kiihtbC5sdc/9TSMsLpWTdD349TIdKE5Y1QlowXsHChkS2O+OqXjl6rXuqFiZgMYyDDc4+IcH8qItEDguM12JVYs8LspC3r9iOBdRXocWPedpvRiDcZJtxyMR6fIl1b4epIDb4XR0JRQSmnoX75rFMOLCCYG0YaLFa7x3vUjweJMToWCSJpjDgIxis0aOiUy0Fy8pS2I9Ez1krKpDjQvRhjVwxZn9SxsKhoXd06+ilrlK8TExVm8lID/mAZhVa1JNKjNcMy16fCBNcSJLDJDXs7+N4Csdk3eCpZJ85QDyC+CEcjKRMy3hbJb4ukdw7NDdKhdMDQoj/6xhoo7oYA6VtTOhgyDGKeppBQHNlms5FN5gQ2BwT0C0Kc2n+pkXBi48RyhtbuMc2e2zvcGJV+JIJnSVznF/EZ+zbUaQ6UiKGYWyq907ziY0v9kLSCE2SeJWI9mVCYL36RF5aWLJbUpT4hx1YH38rK9hHtqRZbw+rKO0tCUd2DPZNnD3S8OKQ7JP9uyUyj9HtGRhOmD0iWMMZ4z4BApoN1hFqfAdG3m8yHl0L+Fu8mUbJxi1cPy9g3oIxT4TL4dXDYmPriYZbTlpfxSp1eB+s8OGRlxTSdUObQLVI3aPJvvPeSgKDLLzT5J75RgWfOAOfSmMQc1GLfbnvnB3oxW4dc8Fxp2BPLyvwOBelRMeW0sv6in1EJQs2M3/Pvoz2e4bnvaw499/0QdL7uefjQVrQcU31bkV19oDMNQek4bQ+aFpm/ANZhCMb27h4BNUhkUvdD2KAAFjl70idvOXCjHIOsvmv39euR1dFb7Vy1aPC1Mvrem7Vl6ZAnE6O10K3U58y335zbYms6KvqZEmhgi4J0PQdkWX8GkGrF5xyCpxWb84KG4kA3Os37J0XndSisdKy5hWR9OZRZ4pEDjMRgFjQmVD196lt3Cr2NkOtG7v35nLMYZqOXw9k86uiN9MFoU0t8ONjzzKVN8hZq4JsaIGsN1qWmwTjPCsdmidTGyBj/wIXxteDucq5UtXoMT9hGOXvArA+sHbleVXwl/MshluUNKSOLqBQv9cmVgRNIyy43Sa7vGxrLhbPjJJzoaCkq/tEc3cBRXdycyODp2A4crrjZL1Lyq0WWcx5ColtG6LiOvB5Bn5Jpe8dk9EI/v/+tdvEXhP4k2YC8qEAetRkfjfK06Ys59XDUDWzL4zTHOrDRfM5NgOyvrXOL3NAJ2L/D9NiFzrJw9DZnU8Znc7hQ30Aii+XzcoQIlE73a8zxxYsvd7P7gcv1Lrw/vQK47+5Yq9F3U4XHbqUKR76AFZp7qMLep6xC+1tV4fCTUmH/U1Ch+0mn5dPTODdWbGWcfEYvvdA7KBRHaF7W9sZZtFTLF46bPDglQsx9RdbJA3OeYW5A9r+LTHplYN0iq29koYXa8S4bqvsUUqT0eRugpaDSrVt2zwZyr1srnR2vgFZe/ZguAPf1LBY5UD1BNTErc4r7VfBccNv7lOQN3YRosGWOInx4ZyEWM+DkldbKl0xHRVL+zE1BX+yzQ7Y6yDG8As2MH68ANU+v9Gzw11HR1u/DCCINCh3nvVV1rklyEG9lBeuvc3Ts5dV73B5yQLa5ThJWVrgZlkV6OvZDIx3cf2PftoMK7/Hreywt1QK/tAJW8iKZWp0YtaqFuk8MxssznKovawyyJ1izi56UHPHjDda1JXl/qUTmF2+T7PLiQEEM0Fgy93zUf378w8fpVJ1ht/YYor5VwY1ewvWNnZ3tOfcZmXu3scNP+o5PvPFTF93wveI58sKkzNCRiHsZOSNSV94dJvouH/lNdGRammJomnKljRQfQY5k9DHsDkpql/LbFq1nQFGtYy2qGIx6PAMEnjuL7VCoBOFER51hBWesvfIp9tQwXHR5jmhmHSIKze9f6zDBr+uHs12CFJYaV0hj+hPHfGhB+yrsUwv9c4gvrLfq2KHzVTPnMrWjLM/9EJq6zPaNOnjI/YX2JbN0a619KC8cHubDO8Ui8ZFdMBQnbLI0oywx55b4VkJzPkaGV9HyTfEVutJyYjN1sW/pqCNHKyV9CaQJ9HanfDlzfQCdDDkr4KWjD1DxmxiZAV5QJL+PXCPvANkXq4I4JlJCThCbUCaz+w/WTXPWlXD4tZU9mKp4k+xY+BNvpWNTGD+7do9TvkXu0vJ5SPJhwQQAy2k2UW1Y9HXscAeVq20nbDzJG3spckRJKTnO376yT2qYfek0rGlJpbMdwerjB+z/MgfRDH3PxJfedu67vi4uQPGIyb6PY7IG4xhDu3EsDW+O8178L3wavSM='}, 'original_manifest.csv': {'sha256': 'b2b4d559fcee93f4f93f8b613d204fe78d6d7f506685910917616ab3f8e83746', 'zlib_base64': '...'}, 'prepare_data.py': {'sha256': '4ed7d4cc66148fc1c917c41797433d7b8cc8886d585b2a148071b9cf7c6e024e', 'zlib_base64': 'eNqtGmtz2zbyu34Fyk6uZCszelqyc7oZN3GbzLWpJ0l71/o0HIgEJV4okgVI2YrH//12F3yLUpzOafIgAexise9d0DCM34QM/D1LN4KpLEnCQHiMS3cT7ATjkcdcKXgKj0yJhEt89ISXwToXnj3m8ZQrkdq93tuYxTJYBxEPWbDla6EAj2Db2At8QGqzD3cxE/cIGaThnu1w49puejl3XZEA4sseUrQJVApIXUA5H83ORJTKfYO6IFUwM8lnAKUK4ojdBemGTpRupBA9niQy3sFObpwEuE8IZ/L2TIotDtvs+xjWwxovcwULPEBGW0qR8iACuJu3P7LVPkXQyCPCFN8K5oaCR2zLo8AXKoUDwkSO+0zPFaQGikVx2uQcXwNyu2cYRs+X8ZY5jp+lmRSOA+xLYpnCZgDEUziR6hVDcg1SUKJ4d9WueNxwtQmDVfEaxMXTf1UcFc9xiUltsjQIi7dUbBM/CEu8nwL9SpS5cRgKl+goSHsZZ1EqZB+O5PMsTL3ATfViUAiRBsCdfGXx3mf476c4ypEmPEVyi2U38NpnN3D+m1gF9/jaK4iJsm0CYgceJhr25s1PBdwb1LRe7+rdy9dvfrt23r++Gk3P2YIZ7kSszmfj2WCyuhh7nLuCCz4azPnYG/reaDDyLgYDdyT8uTcYjt2L1WzMBzM+hj+jwcrzjd7VT++ur1797rz86frqrXO4w4pP3KlwxWw+m13457PVaHDhz8cXrnBnU3824HwyH7iD2cqdCzH0hBgM5v5kejHhF+fnw7EHO7x8eX3z4fpVgfw9oH3oMfg1d7tkRmUJTmFlRl8vPUEmAOYK6dR1DyAfez9fvX3zw/X7D/UTjVYTbzq98F0hLsb+xId/5iukdTSY+HBQ79yb+dPB+fl8ejEcXAxn58NzvoJFYj6eTc6NniaiA/V4MDwHCYjVnPuwdOqeT1bD6YU7Hc4vppMLn3ueGM+mY84Hc280Hgk+EzNvMh34EzHxfA9RX71/Tyy6NaJYbpEBzIhVKmI/SPf4gg5K7SNPqG051nhf9m6u3rwjHMQ700glWOFzje/56HxsJ9EawVrjExq3+l1Q80kn0HzaBVPS+3w4bMHVp0rYZe/63zfXL1FJXv7y69sPlYpoKOOS3Y6Gwz4bDQd9NoeH2WwJKHfAHpgaz/psPOmz4Qj/4gR4sRRnptM+m8LMaICwS1CIHtgyuAXuoOWbaJ/WJe1E7jRORESDgEOuDAvt0dfz+ANfmcmocEI2onC8AIJAavoohQ0HRTAseyPu82FLb+jyKI5Ir+V65SC8GWzzjWEEjhtsbTeOwLenpvHux+8Nq9exod7A9HHF5QMA2neBl24e7+l5I4L1Jn28NGwRubEnTIt9h9jtNCa/blodlG2BvSY43N3iLXitnCSeAEWFF7av5DrbQri4wTdpgqK5MkjQUy4cx4tdx7FyKBsU3OH5ctM4O8tDA4g/3Sdiof2fFH9mgRTe4oPMxFHQOEuTLD3zAvlXoIt41YTN/Ti9mQ6pABBvo+idCIKdaRRuxykDHgQfw8r3kWuFjElsYgxuqIh3ejqIgGBHywiWlUqGy+ycE/WVHwOI7Qt24B/ttUjNOrIcyK/DQbBFgdV0kwdKsN94mIlrKWNpGle1uCzgiEJSulCmJGUiA7bcnGnGd5VmXpmQ2EZJTfOABcMs9tWCtXzjKSpfV9lPgYK5G+F+VNmWbQO15am7KXYlI21sZ5PJRuIuhCRmYYC8SfmDaL0wstQ/m3cYMfeAFQtMK+xXENLf0YDpWy8YnD70UMYhUGXqlTYNonooWCHju9Z8yY8QCMFp4gBkcqdOfX2fQLIBrIZ1lSAwwYPcrTjtau+gL0JnKG8NKUJIk3aCxozlJZPMB8FJ0Aqi6rFORw76FFJeFRGz4j+CllTE2yBFSoGKFe3o9NkKN6Uo81j4KZ1EQsxpkYUUHVJPmSKsyJEvCcvXlFz6wT0g2nSoBfgaMnBF2lomvGXIB7oDqZjYiSjHd7cRES3Os+AyOcYkfKUAF6NcKz1IasnUSqUndHisj0Ik4EdknJQcqHgLJ+XRvuD9La5d3pL2LFEOxThCF+MlbP7DPWgG0ZtFRMGoD1wJneqdHD8+aK8PLuqygetAzn7d0posA2GwMI7WYBVkbUJdsgek/rHPHpDaR6Put3ITKIRbd2aL7hyOiVCJUkkIlcgNwNHgpHHdil6pU7V9oXUkVCeFPQHPLnapjkDfC1mz8Mxcu6wvIvNWCyWRYhfEmQr3TqE8T8HbmYk2ELtYVdBZgc234OVMOLOCtSmcFVBQ/YUm44ZcKYpLesKtGVbOSouGXBzKM8e2RuG8wvlWhlV6i5wc0M/WihM+wzd+jQoB6gKR/gdyc3SgP/qh0Bzy2TqkQ8YkMYCC8dnbj/Bi6hdFER389z2Ixok/1gI8Fm7AMArbRRGHsPhsgqDAZSwMfq8S7hQxzAHbANyL7o3zgA5hw8HaHnDTFs+ZkZf6Rq7vnoDwgrUyJW2CxPb4AnxFAsFPQd2KI7fLF9gBEE4p2rxyNHPy5b5iJoWwvPK0/wiSHw5SBIxXn5rWjIv13r22cIPIj1G+n2x8orjUcgVl8uDHNhgKcrxjhVbNKA2iTBxM5mGoUbmahBApQxW1urZEMNwSXG0cZilmpECxYdsGUkyzIA/gGIyGeeatRyhwTXC8GrsdkBuFkvQTh4o72rsbo/scpxW26FcgZlDUxjEKfW1gEyGafirN5vm/rZE2vFxanSxAYDgrCfDJxL4DraP2yR9vbthWbFdCAqGAqos8Qn0Lk0ssI+AwvRYJoM4mLSKmdjneQ8Ig8VKQQ1Uur9Nfn9VwHxKWVSwv0dRoOeskpYP9h/xBtmgKvJgyCQpbzRz2oXLMj1p/sYPHjE4ZNBlq/KxPDzzP+QAmX50GhquXtkQ6gtXlEY06DHUHC1dhjMXhJxuDillJulPVWlUiwjYKPpQ+hhqdQyyfrI3UfTrDErLMx49rI7k3gtCZeRDb32Px+eYXTRB5t2DbvTkUwWEMB7W6Z31coIJPAk8C7ifFyKkToaXVZ/lAnhAtrSNO7sgpXwWQWipq/3mB75+yODKQyt/fAlX4umTfLdiwm/SDQFIYbHdn4Nj5jSvynLAh1IgryBOUeeKUGNciqI0Vl5LvTQ3lbngUiRDKQ8OyjoNWAc6GRBtINx9a2nqJatz/vDnVftjNiZQOwY4vObVbAZEPUk9NIHULqYTJMdkZWNYX4kZHHAAfNelHsQP6fwB29jcGT39no+nUsqzHo767Wad089rD4mRRZRPPEfLoyi/MfTpR3MkgFY7u6pBZHfigk4VKIyv4fFpMdxCHGkz1DZpixxzVOE/2MFf5fciXVSP4W8s4SzAzqrXoTUyCqiUHSXOTLI3htstCD/zzclkYw+Gc1S4DMaHZWaBsQyJhhyTozewdHh77cZ+p2epZS1WyQTkC54AywU/zlg7dADEFK/PODoUtcGV2jVXkmc0iz6WsvdXl0h7buIPU+UsaKoQcFbLeVPkXDWBbtGqfLPL2ynfs1qjzGz1fEZWsDrRa3Te6TXMk3HYLt40DwrH58O23EurmIwRcsqdpQs1jNFnZ0QA8xfO61nQjAgPrvHD4nO78KCIheVmh1dooFN5U1fmoSt0qfWpd+9UIhYp1y+W+bNGXDrgsvbLUBT4Wd2J2BEwvrsVsmLOgJIh9vD+ApKTp5A0VZ9LFziql6HWhVO3Q0yDoykoAfGkt7/Bwh9s1L5iOY2jJCiBbQmqBdkoYoDrHj29LN89O3i0E8Plo1s2U9kL0SVViauEtR243ThYFf2Y5hMY5aRNP2Q5SSw99pjsUtDrvPrQg8qjplL7L0f03AGh1Up4I6EA4FgBtXBXNv9LVscJCWRqz1rV6wdg+hfNIuEJBThSE+e04D6lDgm0kVrWRbKNFVNFuOiCq1X+izKizM9VCeNhi6uLU4aoWGnEPqY7jylgph/pIFRYovTFI1LpRmr7bZbe+YBqLe1IYraW4qClVSgiMDVxcVssSu/El0VpnKU7xhQRA/QD8Fm1WUHsT3W0p4lq7st78RH4oIVH2EIa2oL6Yc71g3f6cZaCfTNuiufqG7s2KO7PivuwbCEgwoT9+sA7Erg8JWbMHEXGNlH3AO0l4fi4wkOueI9Yt6FRhP7q5Ly7zcMtvLOxFKZdLDKOaaxSO4yyFhdskVkEKMzb7FeiX4R5XIUU6T6BudWGnB+RBPArwzNX5XWLfL5HQX6iwRFAXBVexO46GkpDxwYF14zsDSmSDg7DGjaXH4MQfEQWCoenwHQ9CvgqFrWu8M5RYfl1T1OJgZ7AigPFi0zppB/TntT9ac8RhgRMGYPyFohrvwFiDnf48BbYQCWgmA76FdN2gEro3kWJNLkB/ElPVGGzF3Y+Yc0Xei/yWhcU+HJqnqEb0qQ8KL79TVNjDxBiWOwD2Nmbc8wJ80V4kg//A0JIEJQTeguffEm2EbJysatOZ9caiwzPAZuO3KpBt6SQ+FfepiSO2l20TZebhFdMuhR/KcOUGwULbDHEyShcjvNk1/hM104d2q5OKiYMCMQ69MtDoW9460JE0pd1SQjnXEZV7YceuzHvqK6j7kJ/t9kgc7CgYdFqDPdJr2qG8PkMfBDLAaghUgKQYFJU7sOgFKGUcK/ykC1JZpg+IDWFIpmO5t40vTiMPLjybTCv6xlj8HdYGf631Uh1TKysomrZorOHXAnQ6oxP6oLqbz54RgkSGSY8mu7xwSeyS2DQ+dTLL5spBV3VvWk+pzJGnVHyeRirXUL+axrcG3aYk2C4mHluPbaXrOsBXn7kyKuT5eNn7K+zOycTWWtkSBdS1zib15L5MEF+zf0I1S663vPlteiPyvXcbQJ2/4L7w7m50XttCVzS00UuXCbyXSYSipAaYI2Q9tWkg0CB6huTT6RUaqypUyjiJK+9ynOhrNCCQm7D9AUXPmXminFCp9HHENJ79/mz7zPvw7PWzn5+9d575fxjkKXOPe3rf/4c/7iiWG5smEpuUlaLpKqvQskJ2L6r79CJLgOgokT7MEmo48V6x5d4V2HMScleXnf22KCtYTUvtpA9GtQwz9FS2LRdQqzjcgXFiQtiuB044hc8XaU8uWU7tciTtzrEcmT2N8jD7zrEdTpxG1FVfMeOwFPxcjHysaZ0Wpo8eJNw3vkNA2R9JAPQ3sbbcplJoJbF6PQBx6KrZcagP6DjYY3Kc/IKNvhKzev8Dt8UmjQ=='}, 'preview_training_inputs.py': {'sha256': '1a5c92fd646779f586515678cbaccfa1582d92db426b740ab4dedd07c6feb718', 'zlib_base64': 'eNqNWN1y27gVvtdToL4hmVC0nU3SXTfMjHeTzWQmTTx22l6oGg5EQhLWEMElINtKJjN9iD5hn6TfAUBSlJ1d60ISwYPz//MBR0dHb4QV7UbW0lhZpqwSN0LpZiNqO9W12rEbabZcMV5XrN5uRCtLJutma1nTiqWSq7XNJpOrtb41TNeC2ZaDV71iFx/fsUa0rFTcmJSVetMoiGKtMPKLqI4bXlWiYkspVJU69rTdCIHVCSRLPPHtihThVuo6Y2+0MKzWlgmnCAcz3TLdiJrdcCUrR3Ys7mBPDY3lhq+EySZHR0eTZas3rCiWW7ttRVHgXaNbC6lg57aZSbfUrhreGtE9r7lZK7noHn8zuvbcGm7pRcfqAo8dEdRrdoxD2cbTXrz/0NG9J606Qqvbst+14bZR2pKw4W+2NSKOzlerKLlPlzU7+keSGmW9KNPAcQWiUJulbjemk3tF65fO9xccDv98ef7x6tdPl38vLi4/ff70y6cPKVtspaqKQw6TyS8fzq+u3l6xnM2iGktcRSmLtLFCL6Xd0QN8xs2uroTZ9Guj5/lkMqnEklUSQbExeS85mzB8WoGg1J2jM7Pmz168jMmfniprBa+Kxc4KEydJthZ3gUfSszSN4ruiXS1iK2qjW5jCy2tRV0GEXDL/grg3gv0lZ/EPKXv5/MR9JZRHgaCyO0/gYpMtleb2h2dEQJnnF6VZIsetCNKSjCsVB1HOIi6NYOfGiJZy623b6jaO/lGLu0aUFkm/0ZVQoYycRqg7J9eJqaeePWX1ViDwxPNa7OD/aKeVLnwM5BeXuRFZF8xlOUi2CrFTO1SziZhQ0CRyaqOQQX6w2fHeCF6zzmJvU3w/P2ZQYT6LiDiaJ7OzlH1ExfrvueNjbPVoNqD9DheEEVyCb7Oy2cYJe+J4P3WaJhnaymYL95+m7FnKTpLMFVycdLEGhwwdDftesempmL4kv7pFfofF1+wUrGj9z0L2Rkw7dx1ETaLbba2RlWCX737G9nrVRSrkc91kpZJNDLnQMWWnXbpuOKnmRbtW08Lcru1k5+1qSz3vwr2JUUBlKxvSKS+KSpdFkeztzNBECx62xNF06trvtNXaogIppXIqpBRK/b6Vrajyz+1W/CEHmAUDp5Vs/5QDthnoHhi5H2Jlhkgggsj3uMlqvhEM7YQ1cKDbmDlVC1I1Q7K3EIjYYE+D+ircU0J1GJrPYaj+SbVxv7L68ePGDiSqSrQmGvTNvH0kINtckxjoDdONsytl4g6DsNDXe2Yu5QoD6k44W5XNzHZBbdfEz1OGLoLX1FXz+BTJePoy6bKAFrFj3HhjajeBoNRtRTxnPu/JOxKDUhlyUT/j4uCAvf5i9LYtiXdwb3zgT3ZMXJJspfQijp5kTY3xkcxO5j2HW2nXfhRlND9jzzGhSaJbuZKYn4M4+nSrGfphFeLbfdyghTY9TanrG9RRHKE0ooG2vk1ZjUxSYmmRWrrBHu+mbCX0Rth2FzteGa0N+5AT1Htjql7PI6FGB0862ECV3i2/Zn6NRLDXeXgiUfQw0tolREf6FMqxVwNP2oG1dVhLxs54uFm8rx0KYbdrrcTU4ZpgHmEd1pm45xBTckWOI6nHjMz7jvl8YWAim3pXZ7eyQvie+P3O6OzFCT6n1Occ7bqnXQuCaA8RP8amd0FpVq6pwxk4h1u9kSUhQkM1h42gZwux0/CbrK1YoaG1eltXktKuF9KXaE6tkRvetnwXewd5u5MwBXO8D3M3gV+evXgxeMzBw8LXMNXO12/9O6qfbg6igOLRIExHQzA6CKigScu9cvto4mFAFP/KMVUHjNFr3y2Mq8NnL9msVKk08FwvLu29ggZjtcppKKHV0t/DpPt+kN56dlhgjWyEQ85Vh5WB0kF9Iwb/7+fnA0lJHz/CN7wG8i/I5/HpX0+eoyLkmC7g88f6zXfYx7ltw821TxUAAxPHPVbrkmShtbq/ZYbKPeuq17eas77C52DoYncvQAhOvYvpB9XTWzUjjnMHGBCXR8fjfO/UEgqnYgsFKxkdesZlcT+rZ8EfpO1+qvRqjVsD6X2QXxw+fSihDvWniiFaKpcvgCoHeowKaJ4eqjkqqHnyqB75szfNIFuWS/QJA4QvofeOOhe0oP5RG5wX0FCgnWBjtLpnuTvckYf2mkmXUENR/aFFs9P5wNBKqzzD6FKUQt4gZnSEPQbyhCIVwTzqIv+i6vGn1rNuyD89iC7IPu0fhfePsXs2kPt/S1nsT9ipVyEZD38Ki7c1vDeHjiZgMgNu+G2eyY3BOTzwS75HZYQtHKt4GX0FTvjG/vef/7KvbukbdF9qoCECND99lwUHSIojvVzuWRPwTMYbAIoqrmRpYwfCCsJ+OQTBAjSAQvGFULkkRKngkhtR0DEvX0bOX8dOpeOvHpE43AilxuXiToh5OAUG6JIGVFQ41RVA3P4wTbtrB/965vHCfMw3hK+gflGgh+SzHqkg+wNu9Pu7bjTvxZatboqFvsvdcWasr6dwOVPgdAArA4xOR6nhOm3eddoxD9/zojAWESWumjUv0Fwq9PtV/sA5KxqTUIl2cBYQtvEpEH0W5bp28/zgVsdlRZ/Cod7oRuhvbq74wxC8ipIy+1lz+nwQY4kRAr6D8zDqS5vPTvxhCN/ZTz+G+rPAr8L6U9A+QEfxRZ0GhVOvgHo3Utw6SDtYw28EfmPPBwOikfnpjxCz5CWyUuk2j27XOGSEdCUQ7zsldvmlQ0DsWTlADCgrEbuqkz2U3+GbfXCMscorbrmbjCgFgxhvTR41qAhRRUO6Nkra3JtJBy6kdOGdXZAiSAiPNvYgkIIj6TwYfcBhpdSrljdriiC66FK2xvZXb+PDEMWNsrjEyJ8uOLQYeO1VWD+vYZW2Gu7LH7orGsiQCVsFQ0ZV6a5vCuQ83hRJRu51fSCODlFB1gB9JIP84MvCn3sP2PYc054OmXBAFGKXhqTNQ2caRJRrUV6bnCpEdYepMIkYXR/WlnVl6hNjBHXpfPAqB4pu7gacm97DBhHOy+AX+B4Pk23vbo6ja+CUSmq4ejLd2CMIsKD70PH8e0BKuCrq7qn8rYkZZqC/Xzp2FxRO9W39MCQJtRg/vgbpLjRCbFsoUFg4LqaVrNpuGhN3+Y8o1BU8mgPLP2XRv7sB2LSS7hveHzQdXx5nziVLRGbIYXHH6QbZhNAwi5NNO1wru+7qUpxKqO9XroSykcyQHpPJBPCpcFlZFO7mrCjobqYoIl/i/qJm8n9hrYNh'}, 'spine_transforms.py': {'sha256': '0db3795dcb4818126e2e80af91ba83e696d2a5ae84f50e22f40c0dcc6f4a3b30', 'zlib_base64': 'eNqNVltv2zYUfvevOMgebG+KkgZJsaXzQ9s1XYYuCVxvewgMlZYoiyhFaiTl1C3y33cOKcmS4wZTjDgRD79z+87l6Ojon0JLfnx38x6cYcrm2pQWbMEMz2C1BSYl5Lo2YCuhOKSSWStykTIntIJKVFziexuPRotCWCh1VksOZW0dGF4yoWDFrcg4uIKTBqGEWp/wDZN1gLCpEZWz8FBwBVKzDM9HacHTz5UWytlL+Eia54jyld+xDFCL5UYwif97EwXeto6tSK9Xf+KtBMVKHo+Ojo5GudElJEleu9rwJAFRVto4YEpp562waP789c3Hq9v5n8nd/HZx+/b2A8zg2wjwGa+5Lrkz2/EljO8MR/Wb4BBXThgOGL1X6C5ZiC6oNUfvebbmEVStNLMVTzEmpC6CFC9yc1yhO06D/bfGcMfjKGgjGNT08vy0eYFiFJXErFf4/v40Avosm1NBUJWW3hFv4fUHWAlKCzMtJpNVwZKCqQzfr0nqN2FTZjLwJ/AgXKFrB6nG0FjhUIg8cpRA6zSRYf7+DVDaMNktapvPpNBGfNXKMZnkUlRJhRpO44t9sZUR68IhXaw/f7F/niKEYdYNTpGVmS6T1GhCvWLS8uEJy3P09cAZq9clBvrwLY6KQij6h1stdaKwBohebUS/jUvO1CD0mCWX0ZsXEdBn+dj6ok1abITFm8/hxOc/XyBQfH7x0n+dvuxhnsZnZ7/Q67Oz8/B1QfiPo9EoMHtYEZdeM/I8vOlyifWnN+giVukXLi1oAykyYE2vmGJOl1jGcsBMLGPCukZKrbmBTzEBfkKOWE2dQD9Y+EtiiuTWidSOn7QD5IlIMQSe1Vwirsdz1Bos2yCLuiYDQlnHsQB0jpZuuCHO0T0q54znrJauqROg3Metl8FElMCKRtq4JJmgqjwCsnWGVTMNAaFH5IAlji2DlDGV8gkJRajbTSkePl6/zuB0d4Ue5KPl8DeR/Z0x2kx8TYautsJahlAjWNYiRGo87e6TLT5s2D7oa2du20Uac0XJ1jwhkZ7BDyJzRQQFp0JBhJ1Q3ycv5e0mJxrh/+PGNcFBJrAsiKG2c6l1KO57guzwXnQenUDJvkwGNu7EQ//LkmDczItiaZRCTTqECIyuVRYg4MegYjp9CtIF4DmURuggDHZ6NTQp2gOPYIcIx0PZKZycwNl3JBrHvUifjGiE7MjoE9dLrB9B1Jmb2eMzscupz0uT7hjbINXDZIwtt5ePQdwjkDzH305XbYo6fgUUz6z9wHY6wv+TvWRGwa4YOwkrKxoV8ZvrD9c3717Pd1ApUxtmESnIKv4QLO2HK9qxBkEnTd+c7oPEFXZ7PmmMQ8HOqafJDDf6ATe8Mk3Ae5FuxPPxsEv6yp9968x69F1gNzlnraMR5ELKGdqMP2g8NZ+ZnxHTMXZgUr2qhcwSvxElu61p0s6xCFYs/cxV9qQnYfv6GFYrW6FWJk/IBqxFt8USlCLdvoLPnFd+uSCQlVb8mBo0NVkYDJSmV7/pZupJOz6hGXwexeLylvKGYSuOlvJABtrDms0ChaD5cxRCSJubBf6FpU5uYSXRlBgWBS4zgOaDVrRedPtcf3XEo+2uV3fU783FtgR6l5BOi1HT3JrYhcatcGjWvYkTDSbs+LGX9v1m1+AEHumKm7DpIW/vl62u1oEdTE/wJ5RcxHO/LvzeLThXuN9Mni6L988tQ8tpNGjMvWcRv8WVw/whHHJxsluQZs+q6C1Sy+9CD5+WHM8DdxvYchqC9ANc4YjBvU9kYcb7WY6fXBjrLvvrANSWWz++xd7+4HSDZXiq14raqadmWCyFyrnhOJx7y0FWG7+5hIXCs/1Adg5UeGP2oFIw5YecPrDr9fk3m8GAeoA7FH9uwVuODhJooRdY4UhI7CWL+Ka5wieDy/dhNcQlcO81rYRtLprORpShLZ1Pdrqmo/8AvFNznQ=='}, 'test_evaluation.py': {'sha256': 'f6def3b89a9e6275d15cc56497a89a89791ba30e9286c904fbaeb2286827595d', 'zlib_base64': 'eNqlV0uP2zYQvu+vIDYH2QEr2F5v2t3AhyJIi16KHpJDIRgETdE2NxKpkpR3jaL/vR8pWZafm0UMCJbImW+G8+bt7e2nv74SsZbimyNmSZznXjmvBC9ILpdKK6+MdoTrnFRcWZkTpZfSSi0kGbit9msJapJzz4fp7e3tjSorYz15ckbv3mugeOn8zW5B12W1JdwRXd0srSmJE6rapkG4Iy3NQmlTRq6G4lshudVpKb1VYk/FCw5VcsaFqC0XW+aEsZISYdZSs2+8qvhuaTnevVkjGK9F89lpJTe8qLmXrDS5LFzQD5s3ouDOkc/NJozxBTq5we5Mafj8xJ0cPt4Q/GA04qT/Wg2cLJbtYviJksxw3pRby7eDLLsf0ykd0dGcZnd0+kDHdIzXERbHH+IqFrA/fpjPhx2It7VfU1LBD0oEZYCZzSmejmRpLFFwErFcr+Rg2lNht/10ebsTksoXL3U+yNScvAe9H4gyU/RpPhx+7MnvyJ5OyDrcYIh02z99lACcuHNwmI5kv3oEVDVUy7ooBoNC6haMToeUpKPJMXUWEeNZ98RDeiR5DtD0YXrTuTC4lhVqYbndMr6yUpZSe4Y0YAurpGXC6A1WQnIcezpGlQOiTBfcqTbMGrJ024o+0hMhJq3//E/Ni0FDniW7kE7g4PThLMOvRWncEVutn6VarT2SopcEAeQkJw5V2lv8u2WdpF8QcyEnf1hYyYU1bDkOMna5fAGU8o20fCVnDVPyViFmY0OBCJIOasWhOFrWhVcsFohZAp7kB+QGqIjUBFgQPX3f7Uquo1Lt3vAoUhvGXKJkKs29sS6GqnzhwjNkpbQoXyeB6tcAX5sij7Eq1sY4yUDIuo1LMYtt6BvZdlXzyDQdBJ2M6OTnPas2tuQFJbCANNV662OqNIBZUoXUCodJ5tnjZH45SRqYLHFSO/SojfKIvSxBZ4EDYADY7348fDN/z4QB4f4KQu8A0LvanGH/Zc8uVGOt6JHY2xQwxnf3dHw/2pPJl0oKpC6IuwbYUSHATehUCHAm1ACtcG3yWRIxk1cjTagsEYoV5hmq7eSk+PxezjWqSp81fF+xz8lh0cwmw54Oo7cwjycdd6vHee/+4f40Wp4TPgI3rKlKBGsyvyI79IldPGIOCk0gmBw89MNx4tU6Tkmh2CHfcrmSOsSfZAtjvPOWVydZdy539nNBmAiGtDcnhCFg1A4E8S3FQED/TboESx7T+/+QZVdOtDtNOzyFSD2qc1kCVVTOrKwKDH842ysOOgN5qfG8AfuLreVZbU+qY8i1ztiyrJQNMyvLMbxatahbh3Uywjia5hg7HUJDFKGchqVW1pDyAjHJNNez31Ao5bGbw+ir9Co6OeoSToKScexblYeZIAzPwb3NwAxjlBW3Kog7rJHt31XXdYgwgtCy5Lat6v7ZMIfdnFUXk+EEAuZBR8m5bo70mpMPOJft2B8dsu/3+6SCg1sShhpYu/bgbTG8nu0I5JLx/KlGucvS0ZimoymeO4Q6Pu/w+iE8PY8a3DwszNwYMRWm2g72u+/Ib8biihKyRIZ5daNM7YotZiBrUb7aNtSbZB3xhiwkebZGr9JDMdnjXZgQocmEYkyk8X/Ua0/t1eh7nB4Br/u8gciSvT1brXsWZVFP2PXuTUCRqw/TIr/ioB3QtRBMJ/fHadPVKGbrQjIVRhTMIoiOeMc8zp535BP84BGd8ItZ8IUq0JoxH6zURuJ6Ju2WmCqkPHKRVAaDDfnb1AjS2ehjD6WZZOB7SUKvgCYk3mTlfuKhuNESXjzzrftJyxUQISGXQrlwqemwDm4tqM1jOqGISdyA+reQabh8xON3dUAvzeWhakvPj/+/W4k0soPAnfUKPGx7bRBpyEv+wrbRGK8ldYOuQpCinqk8pK4wtQ5BgPsT7sNLwlALS8kYmc1IwljJlWYsaVzVXX3DKnLuf1vcXM4='}, 'train_models.py': {'sha256': 'fbc3425fd99b52cf5e201900f23b6d797015c71499b88ce7d0cac34cab12db1e', 'zlib_base64': '...'}, 'trainer_notes.txt': {'sha256': '2005c62a9f92fd4f974f19e815c5ef59f5a7a9833237fa4f7bf6bb537d814ed0', 'zlib_base64': 'eNqtWU1z20YSveNXzC27VSBISZazNssHWqIcVihSRVLJ7l5YQ2BITgxgsJiBJPq0P2J/4f6Sfd0zAEEnuW1VHFsEONNf7/Xr1mY1mS1miy9i9bxYTFfiv//+j1hMfxUPz/P54GE2nd+L9dNsMRVPq+VmebecR9HmqK3Af1KU6lWkuZKlygaZdFKot0rVulClE1lT6/IgClk2Nq115UStXrTVpoxFaein1Lyo+hSZvcCBztQ6lblwtdQlvpiIzVEJ2bijqYVMU1U5Kxw+yqVTFsfjOquckGWGo2SmamGPSrlIWpGaGqc7PqLGIfhz++OH7uhYfLgVLzLXOAPm8BFXtyMY71RdwgZdyIOySUQWwKyDLlvDhvjakO/fmxx32nC/wzOViSfEcXfCc76zqpVV9YvK/EmpKapcOUWeK/0SXt9rlWcUzfaQMXlplbCmqVNF71hxlJmQObl5inZKlXR0hSsy3CZM43L2i8NTqwP5hKDST+pNFmyZrdgJqw6UnCSKHlbLf04XYn03my42s4fZnXiaTxbRA24VhclUbj+Kfyzny6urfJDmVvylQl5lffprzB9fv+ePY7FSdqHc7SgW0/1epxqn4+fB51ESPcBJ3KgyK5RMjx/FKL6Kr+Ob+F2Cr+2RmjL1L4woALZSqUY44JTam5pcsU3ukIdJnl+PvFVDel1UUiP0RUN5gOv22H6lS6FCohpObyJ+6VIdWZWjMLw9Ij2q9GtlNIoVYSzkmy6aQjhTXcV4oc415VlVBq8iog4/JmJhvB1xxIagNnunkBN8AZ7sa1OczQEqYB+wkKpE3KEQZK2tKSnrBbIeATe5qSVAcIIrqWxQAN6Eug2DP5GSamXR8zQ1QIgTr1TpL1q9tuV2joDITfpV7Oj/VK8nhlCNyDlHVaPhGyG6j11N+IExKJRff1rOp4EJNqvJYv2wXD2uIyootQUoSkue2aQ6+Sr+V6O5MJXVmfL2MvyteD0qskVmdOs5bEjwmg5DJelv6gmlDu8tStxUcAz2w8mdRokjIBQM/Y2+Hwq8AxVhCViM7KkolAOXwGEJtyuZ8X3OvH83esOfMRx26kCBNU3Jz4jLHPJq3Si5rfSbyjmECCielEbAycwUwxShwddSGBbjLiTQ4Z+4yTqANz0ytazw7qRhlOElQKJUURclqhd/mFC1tJ7mAiuhxvLTR4Fk6m+mdHB6n+tKVJ+SWyYZKjvroh346OhKZe0wxWs4xYnfkElVf0quWhgEpxPx2bgjuAi2OSS0waM9CIEqm4Iccd7tkZnEVigWsBtFp7TanURlcp2eEh/avDpKCkembSrrDF941eDmxkWUAoMv4MKxIB7Hs9WXz8SwTWDCltsSsfasttd5eFQqNAEcDweocolbfP5RPaUjM0oCTq6/BbLeUw5gQ04xo3ucGSXJFWjTwE/fYfiESKPmZkTlICRRoEsNrYMJn1EWO1OqQSCc9PKGWJjK6aK7D6FnDoh2TXZQjiKwB3fFIOiuAj2a8U/guwaqyXL41jYcUelKUQGj1BmbuqwaRyROgPXuUvb5vD2ozYlcvenUHGpZHWFgdxB3JmITtF1pcdzMBQNwhO8YsciAh7ogXnT4bo8H/LXkEdxn4kWqpC9Wz5XRoikUgYfhaQVyo/cnsW/yfOA71UEZwtcpDgming1HBpdZQl3gKhtJgMVasUPIVZmhXRBdA6170IC7uRZUaqbG59yOCRsH5XvqBXo9o+31gSqYnrInw6O0x+EZXLBLsgqhurLyJdRoaIQtuVHvm/19ymTmlc96utng7zUX30fxnOPI/ATX7N+Sd8nV+5tYgDeqtxh+AOZXo1F8/W7ERQHLCTbUyPBRHIXSAbgbZ2KR16NPyegqzus9/10YsqApPiUfbn6MXxVBeZuB7094PBrdxq+yLprqJgao7DavPz3I3CKjk8cnCI66J44WuBXd1ddDP4Ut4CrEj/VBxBTsMe4RLeQOyiuhhg1SoGM6s4cHkGIFr2R9YN9q9BomPXCli89M5pOkyhddm5J7xjnoVJ51g2LanBH5sRMKHSSvft7+cs3HfKccLt64Gkc++Bz7m+u4qYB05CDEf/3lXixAjKo2L4j3H4U5C8GloGoqfBR31uSwEgWzBVD4+VVMDfc0AINV56SO4t/FnuRITz4OewLAd/+ONt6ogSmq1lbsUF9/eIIXwZtAeOcXfB9GlrQDHzTlHt/I4HAnJrmae+KV0N+06J2SoD5rkEtpUivqvYR42NaXv2na1BJlwXqqIMOS6IsqmQcMpwYY74iUAKzPSGcmj1u1+rya+8pY/zS5vn0vCKFgYl+SkOUZ9YCF8dRnGjLKKupP39naet1dpnrxj55W0w68d8vFZrWcryMhqhMaUulf3HoNS6pkwKPJoDYYOu4nm8mwHR0GA09TA8gdGn/W+CTERQ2Ij6OgHEFJGdpUbiqu9NC84OVrDTKzw0CWzN5gSqCyQEp2OZg6l0jL/8c0QDgDP6gBggdpAOrwRi5L8eXpOW4pW+b5nkS8RL9H3vrNn2UOW4hO5dk0Y3B5YuOR59X0WjzVfscN6O2q4vNNiphcPMDPY0RETB8nA1JxqD1TnaLeWMMY4mIvVEEyF2VHnMJsRpDoleQAdFVd4AY0aCLuk5z4MKFx0+R66sGRp7Oe3ibNZV2TnS5LLIK+zJqUNUmD8dJaBEa0Md52MU5+o84OR3akFj0SjTOpyYc99vMNqmvT+KQm/URXken4AObgEq+0WbiZnKZbA7xRD4Uffq4OWdwryENSve7kx2UpDg3YGOIM0TjgHktH50HLI7dBWEmdU6aBksVSrGdzTHfiYTKff57c/RzdPd9PLlT6vgvOubUsl48k45DqoPB8dQzpjrzxpHPkRh3Yq3PDv4jTEdiOBRdy0c363CRQXoQNBP6Oj4H0ZPInKYapioeSzkT4bWnUhZfgw8ykDYVbZf2GjgIog67wmjMOBmaR55/Yq6UB8Q9Iv64hOagsCm2Z7lTWI8eWDbkPUHy8SiPama7Ws/VmuribisniXqym681ktVlHz9bTMvKkdsZ8/cHyCoWQtvXz+PblRtzX1GpRPb8BjWP4EtYgvVkvarcKbXF6AUMTxzAMGaI3col1y/QU6LAY6Q1W0bkn88DLLdPPn34R0QS5QGEa8BfDWfarripK0QOpUfREWbuo01cX2wo+WZcvRAqUc9bIJL3NKwqgybQL1H+hB6hiaVSwYZw+s0nctWdqvfHKz3U8N6p6cKC+RHNyxKRzHopzGoS6aTB4+ij9WCaemH+HFTQlEccQhBl0kgfCOJIkdqjXoMzaY4Lyvxh7aKnk118elcgUykjRPLrT7hX/jCC+OH0BvyKIYD838HiNiiIyQHH1OITUfNl5wMjaUxwvVkk0rdOEVtdNBQO3EpNfgeF6GBZRnKmwfojalntu3l5y2rMu5l0CTQLxee/whtYhSi8wKfy0gCgUYYl95gUUXbTLNdCFiMz2YXJx7ZaqF8IdCKRHz14URa84rOdHO9a1+mnb1hrTb8yMqcswxgQ26WYm6aI9bbf6I1tmlA34Yma7WBZkZCoATS33Yb78ldG8fN48PQPMmzOJ0yzE3d9f1C4A4TDtIhF3U0uQWm9g4A0MCz8aQEtzXtP4pWW3VxEPNFKhQek9PTgvDZmxuuEasFBkTMRf911vHLZ/3tkeOzLYuEG3cswrFSIx3nlAraDNcDGI2f2FUKNujDGyPPHXO+WKQt5JLmPefj2xLCYNmCkYllFvpibv97Pt1st1i9sf2qHUC466CBM2JQZHv/gFph9iPFX7RtoSKc5T6pvyPqUgCgEaB7/El5NHSaSFDFhewIUG4lchke9uYdcVe3IDI+JFlyHAtBJAEYJeDpRmT9JJtPTzKxxN8yY7d/YtCzqqyXFPEW7bPPKTYZLal3HU13pbxOWbKiH5xn+wNxsfgtj+Tlp7+Npx9CeqZIwOOACHd4I/qdzw911gjFe2ben7L0Z/ArSxakfCbecxAtpU/mm7kGX/ehMEzMo0H2fHvU7Qy9Gw0lVI5vgC9oG+MLow01IPAPSIas50efZlMBsuvRCTNe2wdsq9Ehu1+wVCJmk7KFEAxyME2ENyWxEVpFdOd/fXYrZSLFrbXU4i7rk/R0jU18tXVZhWoRHacret8qD9zvTup8XsbjIXv0xXtFifbGbLBVPMfPY4A8PcoflQkmjUU3WqedahqqffpmAGw5nM/pwCsKIpb3/8MPxwK+ypxHsktFDMATc2InjTPod7VRxKlles/qxzrcXfC+34j2bFyHeS7/QQxEZhvvq5SJyl4NBrQz+Mt5rGqy/yKajsMKn6mSNs5VdKdjzVCWf+bUdFID9vmoZ+bcURw/AdFqiw4jwcZu2vKvoaXzo/24TjmJX8TNTNFezqVAN/g7vJozgS5WD0P/b6I4AZrrhcUo8xKVd2yCxW8pxBG/ioY07bDRU0omz8r3TO+eP08/7TOC+BLmaW7lcfaxrtrV+z9TZ5/vvNxZnfcfUC1eTJj8ROG2Hu4ovlJnTxN5U2ZC2NdGwodTbe9nkp3zJPN7KwZEQ7vTO53CXR/wCVsTVv'}}

def install_amp_revision(project, data, assets, run_logged=None):
    """Install the bounded AMP hardware probe without relabeling study runs.

    Only the runner, its frozen copy and the two derived freeze records change.
    An interrupted migration is resumable from verified old/new file hashes.
    No package installation, image preparation or study training is performed.
    """
    import base64
    import hashlib
    import json
    import os
    from pathlib import Path
    import shutil
    import subprocess
    import sys
    import zlib
    from datetime import datetime, timezone

    old_hashes = {
        'eigen_cam.py': 'e159f0bcfede0da4ce3348ca54f444de934b2b015cb42a5f577bc1f76dda78cb',
        'evaluate_models.py': '50f37e547b741f97295df291c64225a9b1d0acf96e186c6be3f7736be993e2b4',
        'original_manifest.csv': 'b2b4d559fcee93f4f93f8b613d204fe78d6d7f506685910917616ab3f8e83746',
        'prepare_data.py': '4ed7d4cc66148fc1c917c41797433d7b8cc8886d585b2a148071b9cf7c6e024e',
        'preview_training_inputs.py': '1a5c92fd646779f586515678cbaccfa1582d92db426b740ab4dedd07c6feb718',
        'spine_transforms.py': '0db3795dcb4818126e2e80af91ba83e696d2a5ae84f50e22f40c0dcc6f4a3b30',
        'test_evaluation.py': 'f6def3b89a9e6275d15cc56497a89a89791ba30e9286c904fbaeb2286827595d',
        'train_models.py': 'f8ec5657edd3fb1d1b4fca3074d5ceb927165a3ae352803a73c69e98b0518924',
        'trainer_notes.txt': '2005c62a9f92fd4f974f19e815c5ef59f5a7a9833237fa4f7bf6bb537d814ed0',
    }
    project, data = Path(project).resolve(), Path(data).resolve()
    code, old_code = project / 'frozen_code_ampfix1', project / 'frozen_code_v3'
    runs = project / 'runs'
    repair = project / 'technical_repairs' / 'ampfix1'
    journal_path = repair / 'journal.json'
    active_paths = ['protocol_artifacts_frozen.json', 'runs/training_plan.json',
                    'runs/train_models_frozen.py']
    unchanged_paths = ['runs/development_manifest.json', 'runs/development_manifest.csv',
                       'runs/spine_transforms.py']

    def fail(message):
        raise RuntimeError('AMP preflight repair stopped: ' + message)

    def digest(path):
        h = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                h.update(block)
        return h.hexdigest()

    def canonical(value):
        return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'),
                                         ensure_ascii=False).encode()).hexdigest()

    def read(path):
        return json.loads(Path(path).read_text(encoding='utf-8'))

    def atomic_bytes(path, blob):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + '.ampfix.tmp')
        with temporary.open('wb') as stream:
            stream.write(blob)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)

    def atomic_json(path, value):
        atomic_bytes(path, (json.dumps(value, indent=2, ensure_ascii=False,
                                      allow_nan=False) + '\n').encode())

    def validate_plan(plan):
        body = {k: v for k, v in plan.items() if k not in {'protocol_sha256', 'created_utc'}}
        if canonical(body) != plan.get('protocol_sha256'):
            fail('training_plan.json has an invalid canonical protocol hash.')

    def no_running_trainer():
        proc = Path('/proc')
        if not proc.exists():
            fail('cannot check for another running trainer in this environment.')
        for entry in proc.iterdir():
            if not entry.name.isdigit() or int(entry.name) == os.getpid():
                continue
            try:
                args = (entry / 'cmdline').read_bytes().decode(errors='replace').split('\0')
            except (FileNotFoundError, PermissionError, ProcessLookupError):
                continue
            if (any(Path(arg).name == 'train_models.py' for arg in args if arg) and
                    any(str(project) in arg or str(runs) in arg for arg in args)):
                fail(f'a trainer for this project is still running (PID {entry.name}); stop that cell first.')

    def no_study():
        no_running_trainer()
        for name in ['analysis', 'eigen_cam', 'interrupted_cam']:
            path = project / name
            if path.exists() and (not path.is_dir() or any(path.iterdir())):
                fail(f'{name} contains evaluation artifacts; no frozen records were replaced.')
        allowed = {'training_plan.json', 'development_manifest.json', 'development_manifest.csv',
                   'train_models_frozen.py', 'spine_transforms.py', 'hardware_preflight.json',
                   'generic_imagenet_weights', 'invocations'}
        if journal_path.exists():
            # A failed atomic replace may leave our staged temporary beside an active file.
            allowed.update({'training_plan.json.ampfix.tmp', 'train_models_frozen.py.ampfix.tmp'})
        if runs.exists():
            for path in runs.iterdir():
                if path.name not in allowed:
                    fail(f'possible study output exists: {path.relative_to(project)}.')
            # Only the dedicated official generic-weight directory may contain checkpoints.
            for path in runs.rglob('*'):
                if not path.is_file():
                    continue
                if 'generic_imagenet_weights' in path.relative_to(runs).parts:
                    continue
                if path.suffix.lower() in {'.pt', '.pth', '.ckpt', '.safetensors'} or path.name in {
                    'run_started.json', 'run_complete.json', 'run_failed.json',
                    'selection_metadata.json', 'external_evaluation_started.json'}:
                    fail(f'possible study output exists: {path.relative_to(project)}.')

    if set(assets) != set(old_hashes):
        fail('asset names differ from the verified nine-file bundle.')
    new_hashes = {name: asset['sha256'] for name, asset in assets.items()}
    if any(new_hashes[name] != old_hashes[name] for name in old_hashes if name != 'train_models.py'):
        fail('this repair is limited to train_models.py; another asset changed.')
    if new_hashes['train_models.py'] == old_hashes['train_models.py']:
        fail('the supplied runner still contains the original AMP preflight.')
    blobs = {}
    for name, asset in assets.items():
        blob = zlib.decompress(base64.b64decode(asset['zlib_base64']))
        if hashlib.sha256(blob).hexdigest() != new_hashes[name]:
            fail('embedded asset hash mismatch: ' + name)
        blobs[name] = blob
        target = code / name
        if target.exists() and (not target.is_file() or digest(target) != new_hashes[name]):
            fail('the corrected code directory contains another version: ' + name)
    # Old code and all existing files remain intact while the new code is staged.
    code.mkdir(parents=True, exist_ok=True)
    for name, blob in blobs.items():
        if not (code / name).exists():
            atomic_bytes(code / name, blob)

    present = [(project / relative).is_file() for relative in active_paths]
    journal = read(journal_path) if journal_path.exists() else None
    if journal is None and not any(present):
        no_study()
        if runs.exists() and any(runs.iterdir()):
            fail('freeze records are missing but the run directory is not empty.')
        print('Corrected AMP preflight code installed; the fresh experiment can be frozen normally.')
        return code
    if not all(present):
        fail('incomplete freeze records; repair will not infer or fabricate missing records.')

    # A valid already-corrected bundle is a no-op, including after legitimate training starts.
    current_protocol, current_plan = read(project / active_paths[0]), read(project / active_paths[1])
    validate_plan(current_plan)
    current_code_hashes = current_protocol.get('artifacts', {}).get('code_sha256')
    already_new = (current_code_hashes == new_hashes and
                   current_plan.get('runner_sha256') == new_hashes['train_models.py'] and
                   digest(project / active_paths[2]) == new_hashes['train_models.py'])
    if already_new:
        if journal is not None:
            if journal.get('new_code_sha256') != new_hashes:
                fail('the repair journal belongs to another corrected bundle.')
            if any(digest(project / p) != meta['new_sha256'] for p, meta in journal['files'].items()):
                fail('active freeze records differ from the completed repair journal.')
            if journal.get('state') != 'complete':
                # Do not mark an interrupted switch complete after study outputs appeared.
                no_study()
                journal['state'] = 'complete'
                journal['completed_utc'] = datetime.now(timezone.utc).isoformat()
                atomic_json(journal_path, journal)
        print('Corrected AMP preflight is already installed; frozen study records were left unchanged.')
        return code

    no_study()
    for name, expected in old_hashes.items():
        if not (old_code / name).is_file() or digest(old_code / name) != expected:
            fail('the original verified code bundle changed or is missing: ' + name)
    clean_manifest = data / 'clean_manifest.csv'
    if not clean_manifest.is_file():
        fail('clean_manifest.csv is missing; keep or restore the prepared dataset first.')

    if journal is None:
        if current_code_hashes != old_hashes:
            fail('existing frozen notebook assets are neither the verified old nor corrected bundle.')
        if current_plan.get('runner_sha256') != old_hashes['train_models.py']:
            fail('the existing frozen plan has an unexpected runner hash.')
        if digest(project / active_paths[2]) != old_hashes['train_models.py']:
            fail('the original frozen runner changed.')
        if (current_plan.get('clean_manifest_sha256') != digest(clean_manifest) or
                current_protocol['artifacts'].get('clean_manifest_sha256') != digest(clean_manifest)):
            fail('the prepared data manifest differs from the frozen experiment.')
        if any(not (project / p).is_file() for p in unchanged_paths):
            fail('a development manifest or frozen transform is missing.')
        unchanged = {p: digest(project / p) for p in unchanged_paths}
        if unchanged['runs/spine_transforms.py'] != old_hashes['spine_transforms.py']:
            fail('the frozen transform changed.')
        if canonical(read(project / 'runs/development_manifest.json')) != current_plan.get('development_manifest_sha256'):
            fail('the saved development manifest does not match the original plan.')
        staged = repair / 'staged_runs'
        staged.mkdir(parents=True, exist_ok=True)
        command = [sys.executable, str(code / 'train_models.py'), '--data-root', str(data / 'dataset'),
                   '--output-dir', str(staged), '--validate-only']
        if run_logged is None:
            subprocess.run(command, check=True)
        else:
            run_logged(command, 'ampfix_validate_plan', cwd=code)
        proposed_plan = read(staged / 'training_plan.json')
        validate_plan(proposed_plan)
        ignored = {'runner_sha256', 'protocol_sha256', 'created_utc'}
        if ({k: v for k, v in current_plan.items() if k not in ignored} !=
                {k: v for k, v in proposed_plan.items() if k not in ignored}):
            fail('the staged plan changes more than the runner and derived protocol hash.')
        if proposed_plan.get('runner_sha256') != new_hashes['train_models.py']:
            fail('the staged plan does not reference the corrected runner.')
        for relative, expected in unchanged.items():
            if digest(staged / Path(relative).name) != expected:
                fail('staged development manifests or transforms differ: ' + relative)
        proposed_protocol = json.loads(json.dumps(current_protocol))
        proposed_protocol['artifacts']['code_sha256'] = new_hashes
        staged_protocol = repair / 'staged_protocol_artifacts_frozen.json'
        atomic_json(staged_protocol, proposed_protocol)
        staged_sources = {active_paths[0]: staged_protocol,
                          active_paths[1]: staged / 'training_plan.json',
                          active_paths[2]: staged / 'train_models_frozen.py'}
        if digest(staged_sources[active_paths[2]]) != new_hashes['train_models.py']:
            fail('the staged frozen runner does not match the corrected bundle.')
        no_study()
        files = {}
        for relative in active_paths:
            active, backup = project / relative, repair / 'original' / relative
            old_digest = digest(active)
            if backup.exists() and digest(backup) != old_digest:
                fail('an existing archive does not match the original active file: ' + relative)
            if not backup.exists():
                atomic_bytes(backup, active.read_bytes())
            if digest(backup) != old_digest:
                fail('archive verification failed: ' + relative)
            files[relative] = {'old_sha256': old_digest, 'new_sha256': digest(staged_sources[relative]),
                               'staged_relative_path': str(staged_sources[relative].relative_to(project)),
                               'original_relative_path': str(backup.relative_to(project))}
        hardware = runs / 'hardware_preflight.json'
        if hardware.exists():
            backup = repair / 'original' / 'runs/hardware_preflight.json'
            if backup.exists() and digest(backup) != digest(hardware):
                fail('an existing hardware-preflight archive differs.')
            if not backup.exists():
                atomic_bytes(backup, hardware.read_bytes())
        journal = {'repair': 'ampfix1', 'state': 'prepared',
                   'created_utc': datetime.now(timezone.utc).isoformat(),
                   'old_code_sha256': old_hashes, 'new_code_sha256': new_hashes,
                   'clean_manifest_sha256': digest(clean_manifest), 'unchanged_files': unchanged,
                   'files': files, 'reason': 'Count actual AMP optimizer updates in the discarded hardware probe.'}
        atomic_json(journal_path, journal)

    if (journal.get('repair') != 'ampfix1' or journal.get('old_code_sha256') != old_hashes or
            journal.get('new_code_sha256') != new_hashes or set(journal.get('files', {})) != set(active_paths)):
        fail('repair journal content is inconsistent with this narrowly scoped repair.')
    if digest(clean_manifest) != journal['clean_manifest_sha256']:
        fail('data manifest changed during repair.')
    for relative, expected in journal['unchanged_files'].items():
        if digest(project / relative) != expected:
            fail('development evidence changed during repair: ' + relative)
    # Validate the entire transaction before replacing any remaining old records.
    for relative, meta in journal['files'].items():
        if digest(project / relative) not in {meta['old_sha256'], meta['new_sha256']}:
            fail('an active file is neither the archived old nor staged new version: ' + relative)
        if digest(project / meta['original_relative_path']) != meta['old_sha256']:
            fail('the archived original changed: ' + relative)
        if digest(project / meta['staged_relative_path']) != meta['new_sha256']:
            fail('the staged replacement changed: ' + relative)
    no_study()
    journal['state'] = 'switching'
    atomic_json(journal_path, journal)
    for relative, meta in journal['files'].items():
        if digest(project / relative) != meta['new_sha256']:
            atomic_bytes(project / relative, (project / meta['staged_relative_path']).read_bytes())
    if any(digest(project / p) != meta['new_sha256'] for p, meta in journal['files'].items()):
        fail('post-switch verification failed; retain the repair journal for resumption.')
    journal['state'] = 'complete'
    journal['completed_utc'] = datetime.now(timezone.utc).isoformat()
    atomic_json(journal_path, journal)
    print('AMP hardware probe repaired. Original freeze records retained; no study training was changed.')
    return code

for name, asset in ASSETS.items():
    blob = zlib.decompress(base64.b64decode(asset['zlib_base64']))
    assert hashlib.sha256(blob).hexdigest() == asset['sha256']
    target = CODE / name
    if target.exists() and sha(target) != asset['sha256']:
        raise RuntimeError(f'Inna wersja kodu w projekcie: {name}. Nie nadpisano.')
    if not target.exists():
        target.write_bytes(blob)
print('Kod i manifest sprawdzone:', len(ASSETS), 'plików')
run_logged([sys.executable, '-m', 'unittest', 'test_evaluation', '-v'], 'statistics_tests', cwd=CODE)


## 4. Zweryfikowany ZIP i przygotowanie zbioru

Wskaż `baza danych(2).zip`. W kolejnych sesjach zeszyt użyje zachowanej kopii. Oczyszczony ZIP zachowuje wszystkie 824 pliki. Przy historycznym ZIP 827 wpisów pomijane są tylko `train/normal/264.png`, `train/normal/85.png`, `train/osteofity/115.png`.

Kontrola integralności sprawdza wszystkie pliki i duplikaty. Nie wykonuje predykcji testowych. Trening i techniczny test GPU otrzymują wyłącznie dane rozwojowe.


In [ ]:
ACCEPTED_ARCHIVE_SHA = {
    'c4eb673704b93daaceaea208a3d1fd202d900c2ef8d013c9b73a07a37a320bdf': 'oryginalne 827 obrazów',
    'ba4c5ece78779f67b209f839cec75f70aa480c07bc8ee1dee008f4594a96613d': 'oczyszczone 824 obrazy',
}
SOURCE = PROJECT / 'source' / 'baza_danych_verified.zip'
SOURCE.parent.mkdir(exist_ok=True)
legacy = SOURCE.parent/'baza_danych_original.zip'
if not SOURCE.exists() and legacy.exists():
    assert sha(legacy) in ACCEPTED_ARCHIVE_SHA, 'Nieznana starsza kopia ZIP'
    shutil.copy2(legacy, SOURCE)
if not SOURCE.exists():
    from google.colab import files
    uploaded = files.upload()
    matches = [name for name, blob in uploaded.items() if hashlib.sha256(blob).hexdigest() in ACCEPTED_ARCHIVE_SHA]
    if len(matches) != 1:
        raise ValueError('Prześlij jeden zweryfikowany ZIP bazy. Nie znaleziono jednoznacznego zgodnego pliku.')
    temporary = SOURCE.with_suffix('.tmp')
    temporary.write_bytes(uploaded[matches[0]])
    assert sha(temporary) in ACCEPTED_ARCHIVE_SHA
    temporary.replace(SOURCE)
    del uploaded
source_hash = sha(SOURCE)
assert source_hash in ACCEPTED_ARCHIVE_SHA, 'Zmienione archiwum źródłowe'
print('Wejście:', ACCEPTED_ARCHIVE_SHA[source_hash])
run_logged([sys.executable, str(CODE/'prepare_data.py'), '--archive', str(SOURCE),
    '--output-dir', str(DATA)], 'prepare_data')
AUDIT = PROJECT / 'data_audit'
AUDIT.mkdir(exist_ok=True)
for name in ['clean_manifest.csv', 'data_audit.json']:
    target = AUDIT / name
    if not target.exists():
        shutil.copy2(DATA/name, target)
    elif name.endswith('.csv'):
        assert sha(target) == sha(DATA/name), 'Niezgodny zapisany manifest'
for proof in (DATA/'verification_invocations').glob('*.json'):
    target = AUDIT/'verification_invocations'/proof.name
    target.parent.mkdir(exist_ok=True)
    shutil.copy2(proof, target)
print('Gotowe: train 579, val 95, test 150. Źródłowy ZIP zachowany.')


## 5. Rzeczywiste wejście modeli — pełne pole PNG

Podgląd pokazuje po jednym obrazie **wyłącznie z train**: źródłowe RGB, wejście po proporcjonalnym skalowaniu i dodaniu marginesów oraz jedną deterministyczną augmentację. Kontrola numeryczna sprawdza geometrię bez przycinania, kształt i typ tensora, zakres wartości, czarne marginesy i zgodność transformacji obu rodzin modeli przed normalizacją.

W treningu dopuszczamy odbicie poziome oraz niewielką losową zmianę jasności i kontrastu (0,1). Nie stosujemy RandomResizedCrop, CenterCrop, RandAugment, losowego wymazywania ani przekształceń afinicznych. Są to augmentacje podczas uczenia; nie zmieniają źródłowych PNG. Kanał alpha jest odrzucany bez kompozycji, zachowując zapisane RGB. Normalizacja wejścia pozostaje właściwa dla danej rodziny wag ImageNet.


In [ ]:
run_logged([sys.executable, str(CODE/'preview_training_inputs.py'),
    '--train-root', str(DATA/'dataset'/'train'), '--output-dir', str(PREFLIGHT)], 'input_preview')
from IPython.display import display, Image as NotebookImage
display(NotebookImage(filename=str(PREFLIGHT/'training_input_preview.png'), width=1000))
assert json.loads((PREFLIGHT/'training_input_preview.json').read_text())['status'] == 'passed'


## 6. Zamrożenie protokołu i środowiska

Przed długim treningiem zapisujemy kompletny plan 20 przebiegów, hashe kodu przygotowania/treningu/analiz/Eigen-CAM, konfigurację analiz i środowisko. To plan nowego eksperymentu podczas rewizji, nie historyczna prerejestracja.

YOLO: 240 epok, batch=100. Torchvision: do 100 epok, batch=32, zakończenie po 20 epokach bez poprawy. Wspólne są geometria, podstawowe augmentacje i zasada wyboru checkpointu. Optymalizacja, normalizacja oraz budżet epok nadal różnią się, więc porównujemy całe procedury uczenia. Walidacja wyboru checkpointu i inferencja używają FP32.


In [ ]:
CODE = install_amp_revision(PROJECT, DATA, ASSETS, run_logged)

EVALUATION_CONFIG = {'bootstrap_replicates':2000, 'batch_size':32,
                     'reference_model':'yolo11l-cls', 'reference_seed':0}
PINS = {'torch':'2.11.0+cu128','torchvision':'0.26.0+cu128','ultralytics':'8.4.163',
        'numpy':'2.2.6','scipy':'1.15.3','pandas':'2.2.3','scikit-learn':'1.8.0',
        'matplotlib':'3.10.7','pillow':'11.3.0','opencv-python':'5.0.0.93'}
env_command = "import json,platform,importlib.metadata as m,torch; print(json.dumps({'python':platform.python_version(),'packages':{p:m.version(p) for p in " + repr(list(PINS)) + "},'torch_cuda':torch.version.cuda,'cudnn':torch.backends.cudnn.version(),'gpu':torch.cuda.get_device_name(0),'gpu_total_bytes':torch.cuda.get_device_properties(0).total_memory,'gpu_capability':torch.cuda.get_device_capability(0)}))"
runtime = json.loads(subprocess.check_output([sys.executable,'-c',env_command], text=True))
for package, version in PINS.items():
    actual = runtime['packages'][package]
    assert actual == version, f'Niezgodna wersja {package}: {actual}; wymagana {version}. Uruchom poprawioną komórkę instalacyjną.'
assert runtime['torch_cuda'] == '12.8', f"Niezgodny build CUDA PyTorch: {runtime['torch_cuda']}; wymagany 12.8. Uruchom poprawioną komórkę instalacyjną."
protocol_artifacts = {'code_sha256':{name:asset['sha256'] for name,asset in ASSETS.items()},
    'clean_manifest_sha256':sha(DATA/'clean_manifest.csv'),
    'evaluation_config':EVALUATION_CONFIG, 'runtime':runtime,
    'experiment':'Full outlined PNG field preserved; new cleaned-data revision experiment'}
protocol_path = PROJECT/'protocol_artifacts_frozen.json'
if protocol_path.exists():
    previous = json.loads(protocol_path.read_text())
    if previous['artifacts'] != protocol_artifacts:
        raise RuntimeError('Kod, konfiguracja albo środowisko różni się od zamrożonego eksperymentu. Nie zmieniono zapisanych danych.')
else:
    payload = {'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(), 'artifacts':protocol_artifacts}
    temporary = protocol_path.with_suffix('.tmp')
    temporary.write_text(json.dumps(payload,indent=2))
    temporary.replace(protocol_path)
if (RUNS/'external_evaluation_started.json').exists():
    print('Ewaluacja już rozpoczęta: plan i trening są zamknięte. Przejdź do ewaluacji.')
else:
    run_logged([sys.executable, str(CODE/'train_models.py'), '--data-root', str(DATA/'dataset'),
        '--output-dir', str(RUNS), '--validate-only'], 'freeze_training_plan')
print('Zamrożone środowisko:', runtime['gpu'], runtime['packages'])


## 7. Techniczny test GPU przed właściwym treningiem

Wszystkie cztery architektury wykonują na danych rozwojowych pełny batch treningowy, propagację gradientu i dwie rzeczywiste, odrzucane aktualizacje optymalizatora oraz inferencję o kształcie walidacyjnym w FP32. Sprawdzane i zapisywane są pamięć GPU oraz kompletność testu. Techniczne modele są odrzucane; właściwe 20 przebiegów zacznie od świeżych wag ImageNet. Zbiór zewnętrzny nie jest odczytywany.

Jeśli test nie przejdzie, nie uruchamiamy długiego treningu i nie zmniejszamy po cichu batcha. GPU może zabraknąć pamięci mimo poprawnego kodu — to musi sprawdzić właściwa sesja Colab.

AMP może pominąć pierwsze aktualizacje podczas zmniejszania skali. Test liczy wykonane kroki za pomocą hooka optymalizatora i zapisuje każdą próbę oraz zmianę skali. Limit wynosi 32 próby; utrzymująca się niestabilność, nieprawidłowa wartość funkcji straty (NaN/Inf) i brak pamięci nadal zatrzymują test.


In [ ]:
if not (RUNS/'external_evaluation_started.json').exists():
    run_logged([sys.executable, '-u', str(CODE/'train_models.py'),
        '--data-root', str(DATA/'dataset'), '--output-dir', str(RUNS),
        '--hardware-preflight-only'], 'gpu_hardware_preflight')
else:
    print('Ewaluacja rozpoczęta; technicznego treningu nie powtarzamy.')


## 8. Właściwy trening — cztery modele × pięć seedów

`START_TRAINING = True` uruchamia uzgodniony eksperyment po zaliczeniu poprzednich kontroli. Można wskazać podzbiór modeli lub seedów na daną sesję, ale plan pozostaje kompletny, a ewaluacja wymaga wszystkich 20 przebiegów.

**Po rozłączeniu:** wykonaj komórki od początku, z tym samym projektem i środowiskiem. Zakończone przebiegi są sprawdzane i pomijane. Torchvision wznawia stan z końca epoki. Jeśli dopasowanie modelu już się skończyło, a przerwano tylko końcowy eksport/walidację, kod odtwarza ten etap bez ponownego treningu. Przerwany w trakcie dopasowania przebieg YOLO zostaje zachowany w `interrupted_attempts/`, a następna próba zaczyna od ImageNet z tym samym seedem. Nie deklarujemy dokładnego wznowienia YOLO, którego nie potwierdziliśmy. Nie kasuj zapisanych prób.

Trening może wymagać wielu sesji. Żaden wynik zewnętrzny nie służy do wybierania próby.


In [ ]:
START_TRAINING = True
MODELS_THIS_SESSION = ['yolo11l-cls','yolo26l-cls','resnet50','efficientnet_b0']
SEEDS_THIS_SESSION = [0,1,2,3,4]
if (RUNS/'external_evaluation_started.json').exists():
    print('Trening zamknięty: ewaluacja zewnętrzna została już rozpoczęta.')
elif not START_TRAINING:
    print('Trening wyłączony ustawieniem START_TRAINING=False.')
else:
    run_logged([sys.executable,'-u',str(CODE/'train_models.py'),
        '--data-root',str(DATA/'dataset'),'--output-dir',str(RUNS),
        '--models',*MODELS_THIS_SESSION,'--seeds',*map(str,SEEDS_THIS_SESSION),
        '--restart-incomplete-yolo'], 'study_training')


## 9. Ewaluacja wszystkich zamrożonych modeli

Ta komórka odmawia pracy, dopóki wszystkie 20 przebiegów nie zostało ukończonych i zweryfikowanych. Przed inferencją testową utrwala wybrane wagi i blokuje dalszy trening w tym projekcie. Próg binarny wyznacza osobno z walidacji i zapisuje przed testem.

Powstają: identyfikatory i prawdopodobieństwa dla każdego obrazu, nieważona kappa, accuracy, macro-F1, balanced accuracy, metryki klasowe, AUC, kalibracja i obie jawnie opisane konwencje Brier. Seed 0: pełne bootstrapowe CI (2000 powtórzeń); pozostałe seedy: estymaty i dokładne CI proporcji. Trzy porównania YOLO11–pozostałe modele: sparowane różnice i dokładny McNemar z korektą Holma. Pięć seedów ocenia stabilność uczenia na tych samych obrazach, nie zwiększa liczby pacjentów.

CI na poziomie obrazów zakładają niezależnych pacjentów. Poprawność danych i ich przyporządkowania potwierdził autor; nie jest to niezależna weryfikacja powiązań dokumentacji pacjentów. Grupa morfologiczna syndesmofitów/parasyndesmofitów nie jest automatycznie rozpoznaniem axSpA.

Po ukończeniu ewaluacji ponowne uruchomienie tej komórki tylko sprawdza jej zgodność z zamrożonymi danymi, wagami i zapisanymi predykcjami. Zachowuje gotowe wyniki. Niepełny lub niespójny zapis ukończonej ewaluacji zatrzymuje dalszą pracę.


In [ ]:
# Preserve completed statistical results on reconnect or Run all.
if (ANALYSIS/'analysis_complete.json').exists():
    import base64, zlib
    guard_bytes = zlib.decompress(base64.b64decode('eNqdWt1u4zYWvvdTcKfASm4dJZl2il23LjCYpkAXnXbQGRSLBoZAS3SsRhK1opTEGeRu32gfYbHvtecckhIpyY47czGxJPLj+f8h+dlfzltVn2+y8lyUd6zaNztZfjl78eLFr4KnZ7LM9+xO1Nk2S3iTyZJtxFbWgqnbrKqy8obxkvG8hrF7lsiiykUjUqaaFp7FHc9bmhXNZr+2JWt2mWI7kVeiZhnMY6rdVLVMhFILdp81OxghmKyzm6zkOdvW8lGUAJsKlma1SBpZ72FOVeWZSGfiAf4mWZPvI/Zjw0oBdLJc8lQBcgGT8gWDlYDcQsFyW1GLMhELwGf3ddYIZcjkdZNtedKoaPaaFZlSyJbhBTkueH0LyEWrGuCe7XiZ5sDjZk/UlrIRGylvA9UTbhiXdYSCnAEfBYvjbdu0tYhjlhWVrBuQHMwl8ajZzL6rbypeK2GfE3Vnf+642uXZxj7qP/Aiapsst2//ULLU61W8weF2sXfwaAepfb9g2RYViECxsprNrv757urNh6vv4ze/fH8V//DjT1fv2Yp9nDH4FzQ1z8qY5Kqiah8sWKDAAkQMH0pFUjavRXYjyjjhhX3W4hDuZI1Z1QLYFXHKG24Gw6u7TNzHtBxoIs7Kqm0sNGitiR276qGs8OOCl9kWhkUgO5qCQKKOUVEqah4amPE0m81SsWVqx0MU1JLkM2dn34FN1EtCJHvEj5GsRBkG9SaYo6C0/vUY/FcLUGtp1RNtsxz4AQmoJtRDUVA7/vLV18E82okH821uSEDXiVFtQ0LSLGkmKQErlilIZhW0zfbsb9NUoYgEKA+RI3QKQ8ycRmRbtFuWgamrhoNXhDR+QYvOHd54pgT7Db9d1bWsw21w9VCBI4L9/+P9Lz8zufkDnpbsI5L3FGh0IxGC7Lj8VwseHCayTDPU3JKB04CDFuD7/EYsUe7E9s+yNGwYIvspA7IgojRZYQgzOOwLFgCEE3oYKkSxe/B+JiFCoOs3oowCqwCKbvu4C16OeYWfLyj6aK1ANKmlZlc/otXGtZT2RUffoX+dTTuTHFL/FBaIrwGZ8SoGH8oxOgu1hDDXgNJfXlxcnIDAm2QXq+xR2HlfvhxYHjLfsQ0D8DmqhZL5nQjn3Zf+Fc3q5LLwOR7xCohhR2Y3y13Am+9+GCD1n+aGcK3NGO0SlhkufM4CDsFirzLVaT5CXwm86TCz908P01q6NWsDcSOaMACXaloFfrlascB+CiDep8wfWFpbA6ur29JMeTnQXfAB0kxnnmeOaZvMlEowcOMq26wuICHngMIQMgqOUVrVGWDswYRMdtQh2pC+l7m8vMzPklwFHkVjTsZASohU41w8M1c8NKLGyE3BneK/bFW+j8UDLyB0IwzUDR9qCFBHBOP6fCeRLBUQJLY6VW+zBxhmSNUVApGDpB6X05SzaeamvgyofDNFYTdPK+2mrQ3l2RYkSGojFFMDuXZo3fEck6VsZCLzuKthYj1B2/L8Oug+BOvOo2PMVSCglUG/DuitSVJrTw4K2HfmEM8TZQLkuB80pUq2dSLOqGjD0AvSbVyuiCkowUpeUL7BZIjFoLNIBMVZoUI3DxlqiBRgHCfPI/BdXCGcay1CLnc/I6UafhwKtz61usRMIEPeiBSSGc5/ssQKk/BiraiuICLdThrGctoqYHQXcXFM99DTFwwdcel74cIdQR62ZCZaPHlqs4p14p4mP1ijXAY8DQzWyMaxVtBhA1FYGUUyqjBt5dt9Pe5DDikqqbOqsQZHmnLUfz1VMK5PciqRS6SykdABaFLB/T1GaOUhoWg6XqZB3+peVDm3/qQpdbnyRnUcDeOUGcRwkLUzBsUXmJ5TzBqqbAFrc1efGLFWLom4JBewnFfq+nOBEpiKjHlwPt/+6CFr/hodb6NoPkQxEegYxlQcx5EgFeQWQ4adc1RcBPMZ+wnqW0atKoZ5SBsJ1BpoTGfJTiS3vZFARwatosLSNEu1Reh2VFF8NXDaDM8L0XCkhpiEnAHV5WYPjvwNrrKHfhPKctNxQnmZ3ewazDm1bjUjwoJOK0pl2cTUcMY4nULNirKZHgKeCM9+OxfhWwzlBUW4OJe6+w4DE9+pcIiRZmQ70GUqGsaE63QlPy0FmRRLbOyC8Zl6A2zHlVN5H6ixgzfUsmqOpzp1tzSRaZuLLoZa8Y851QM1r0hQiP/NO9EY+iLxAALRY8MOTQ9DpwIBtDU6BwAJ8Sj6kg9UbrQt+p5SD/J93ne+aQByRTskdOpcz8kWRJLhgd9BmO2I6rP4REXqvNITTNTxY5UDeCwEkT97MrmeHrkeOfR4iRrMtk7VNKr9eApQ2VXbXsF7wswDkQTn25fXts9eH4q/7q6O9W4wSooSlcQmiDzcS3MUUpx2UtM0LA7AQ2/90mBS2MsTdeKUBNOML49y3elkeVhbzujnK/ClU377lUZv0aMM2sG6qb+BihTK7T6feiIcKO7KILhNj12GkcgHlWWSc6WotO29981Pr9+/v3qvv3ea7ovgj2F9Hehya71g8JsKq/V8ib+dCZP+4peVYEu0s3lQ6E89FcAxSBp7bXJ42qWwzRIC6f4EwNAwDIGA0EdoHISk9mPwyRtCimrLWLZN1TaTfbBe5hxKYpwdf8T/n/x2r6u+HShQb59EoRTNtQ6bHTTiO5lbBfc1+vR+xDZ4azZce7Qzi8Y6NKjJic6nheZYUxkNihESCJTbEI8hQdp9KpDOdYjEQpb8+6v5goW0iwhPl68u5gNhUT6BEjt7wGDtsrsNPhL2k6O1QVvc79URQoS/Y9Vu4XcYUI021zt4pbjPwa1WARBx0m6epw15j4aSZ5CBADP6HmjBvXpR2y2++Whe52OYnnC2kuiFYUUyq1BGfTjZ7GnYMUMf/IPaAjetwmqOJWqjri+oyyB5jYmx5pSDJJAZigK9wg6uinnhukZnysFO7oShsvc6BPPbG2L38E7YNnjX6RJyww6Ecg5+ilsqOqxM292CGVMYGqC2Hrnhmww+Z+TQZRVxxeua78Pr6y1UMw2Em21QxR+TJwgyRH1CMUOHrtOl7kUc4h366WZfiRWsSSt9/dWYvqamfWGXLAhuGALxS5zzDcbB+WFkGD2Fa7wChDQJ3n3+9BWs3XgSjiAqVwKVHvYmtGAT010zwkXUFtJIM4CbRzzPw2cn4yB/IvsOYvec/ZUNXn+7YpfzU/CSXKoRb20RXkLEulwwyGT56lKcvXoeC+Ueg6x4HnZSX/h2CYNuCv4A4PNj3vFjSVGZ9RHPhzknXX6Kl5Du5f2CEcKCVkBTeMwqigkLstNFb1XzA8HQ7oaw1VQEuwasUcBYH0Ay9QzOoG4vzlK9W9It4n446qaoCgKyNcMA5vlSwoBcoAGRjNi37Kv+FckL3jwLoOdi1a7DCzhXKh5CCtdTwrER/HI9P41DihoErrk061zTwidKqQ8OE0D4cX38NMOL43rLt9k7MRwXeeEx+mI9aZaUvcGH6MBsMouX1WMAHgkOK+/jKktuc7H6gedKUN4m31PLowamx0BlifzFuNUIDEfg3pjQ5w7nz8rOIvVG6cFgrnQM2Q+3J4NbUx1Bd6b9J4EHIcqus4/RkrACx7/zT4Xx4hOi+eH9OTt68/43Qv/53e9O1MOGUPEbKOY/IdRBo1lnifKa/wOVpRnqtv1TFmSHUV/sHNj0JxreiK4v649k8NfRPOIBEHVmIv4cjzCt3IlFnDfVdYOTrH+MMGrRNM6w1bs2d0KQ+fWxtPfWaKzbIvj0ajApdDn0KGqpwvArrEyer3TQttM0glIxKaBfGaZCKAgOG8fAL5xSzAgNT3vKbauwbys4vHogz4vRe2MnEsu8LaDDWcNySXFCOaULmG4R8wNycMAT6Id5ssffkJ8hJjUiWC9OLXQBvRC8DHXluvIEUWNZdNGXR5cv56do1rqzDvi02eS6+6co+lBL/8WKvZx5p17UkQYVB22ltB3G6wwcXpF/XmjnXzArwBhSDe2V6fssY+YCOyDG6w14d8befNlitqOuHgpJPPOkD+uJY7XxtsDUCdvBYXjgHeIfOs5i37GLqSO3t/7VKpCrPYJnOhYOT94+Y69ZvxnlzHvz+i1uWG9kC1Q1ErfolNn3HyvzG4OVSjoR5hVkhDvBFPBVNgALzauzu9Sdl+obI7hxUwBpAiupxm7rJ7yICaXkuvR0jmS7y0+B9+SMd+8ZwAcqyI5hWJLs6Wg/HUjvECLxAOnZOzWFb/4FBo+M3oY9VmCQd2zYf3TKVXuGPnmxwj2wxGeywIt+q4TU1m14ffR3/Ea3MpydU/f+xWCjcLR13k87faN9gGk3Q60hxeA9PawVwfl44ISnjsGgkDwNDCvOKTAbHw6CeNXEAAELkLGERzu4B1SJP12EafFN73+fcs7aIz2NYhVeKzxyy2Y2lZ+6OeaY7pk6YXDkfrGeBMU9AMepEf9W7HWVZK4V6j0OeLug63sY+B3jt9ccJjJWcIXO7AdLE/QwhrmBz+41j8/gIv863kcrsSUL9I1e0CXmPwi+sY1zNBFzx+hq0nJ8L6kcZbvYAqO7HsiIp5mJ5fv5A5Lnwoa+F+Je3bK3PwsADgeXDun2LR5T2pu40ev6pi1AwO/oS5gKfYcA+FnFcSqTOJ5PpHc8j9XXWSmq408sJc/Q6rtbqcBV96LXm37lpGlNE5aEMTfEhMHZWcC+MLdoqJzUtwmNm6SrD10XdWB6d0flzLm6YrDAGaBKFVve5s0KbxMeR8KbLGd0rWVq/pcvrR0qeATRHr1x+fkdLBKalegProVuYmiocUeRrrWmbVGpUMMuGGbBEqj1xgW/81v5v38nj7LkTNzjMskfHC/HVfWj2GTi5r//uYeupub3KY6J2O882cl7Xkp2rx73qrnNBHvkFRSMJZSL+zK7zb5hlSyyPyC7VrKU93iYvgEBgvdlXN8rhcQcU08TxxSe4hhtLY6DpTlkRsOb/R88TPZA'))
    assert hashlib.sha256(guard_bytes).hexdigest() == '0c89872a7bc9b9014321432c0f9a471c3c15e6c497c32e8b2bff59df99a1d3de', 'Uszkodzony kod weryfikacji ewaluacji'
    guard_dir = PROJECT/'technical_repairs'/'evaluation_resume_guard_v1'
    guard_dir.mkdir(parents=True, exist_ok=True)
    guard_script = guard_dir/'skip_completed_evaluation.py'
    if guard_script.exists() and sha(guard_script) != '0c89872a7bc9b9014321432c0f9a471c3c15e6c497c32e8b2bff59df99a1d3de':
        raise RuntimeError('Zmieniony kod weryfikacji zakończonej ewaluacji; nie nadpisano.')
    if not guard_script.exists():
        guard_script.write_bytes(guard_bytes)
    run_logged([sys.executable,'-u',str(guard_script), '--code',str(CODE),
        '--project',str(PROJECT),'--data-root',str(DATA/'dataset'),
        '--training-root',str(RUNS),'--evaluation-root',str(ANALYSIS),
        '--batch-size',str(EVALUATION_CONFIG['batch_size']),
        '--bootstrap-replicates',str(EVALUATION_CONFIG['bootstrap_replicates'])],
        'verify_completed_evaluation')
else:
    run_logged([sys.executable,'-u',str(CODE/'evaluate_models.py'),
        '--data-root',str(DATA/'dataset'),'--training-root',str(RUNS),
        '--output-dir',str(ANALYSIS),'--device','cuda:0',
        '--batch-size',str(EVALUATION_CONFIG['batch_size']),
        '--bootstrap-replicates',str(EVALUATION_CONFIG['bootstrap_replicates'])], 'model_evaluation')


## 10. Nowa, odtwarzalna Figure 5 — Eigen-CAM

Kod używa głównego YOLO11l-cls seed 0. Przechwytuje rzeczywisty tensor wejściowy i aktywacje w tym samym wywołaniu co predykcja. Mapa jest nakładana na rzeczywisty tensor po proporcjonalnym skalowaniu całego PNG i dodaniu czarnych marginesów. Zapisywane są parametry tego odwzorowania; nie rozciągamy mapy na inną geometrię. Dobór przykładów jest deterministyczny: do czterech poprawnych (po jednym z każdej klasy) i do czterech błędnych; przy braku błędów liczba paneli jest mniejsza.

Eigen-CAM jest wizualizacją aktywacji, niezależną od wybranej klasy. Nie dowodzi przyczynowej interpretacji modelu ani klinicznej trafności jego rozumowania. Powstają ryciny 600 dpi/PDF, surowe dane map, hashe źródeł, warstwa i reguły wyboru przykładów.

**Poprawka odtwarzania partii:** moduł Eigen-CAM odtwarza te same partie po 32 obrazy (ostatnia 22), ich skład i kolejność co zakończona ewaluacja. Przed tworzeniem paneli sprawdza prawdopodobieństwa i decyzje dla wszystkich 150 obrazów. Tolerancja porównania pozostaje taka sama. Pierwotny zamrożony kod i wyniki są zachowane; poprawka i jej audyt trafiają do osobnego katalogu. Wcześniejsza przerwana próba ryciny jest zachowywana.


In [ ]:
# axspa_fix_eigen_cam.py — run in the existing axSpA Colab notebook session.
# The original nine frozen assets and all study/evaluation records remain immutable.
from pathlib import Path
import base64, datetime, hashlib, json, os, shutil, subprocess, sys, time, zlib

REPLAY_SHA256 = '7a0b9deb225e36c8b426f6fc2dbf6cd993187733783db3625c8ec44a37a1020c'
REPLAY_PAYLOAD = 'eNq9fe2S28ax6P99CgSpWyZlEPsh2SdFha4rr2VHdRVJJclO5dAsFEiCuzgCAQQAd5fau8+RB8qL3f6aLwAEV45zVYmXAGZ6emZ6unt6unv++IfTXV2dLtP8NMlvvHLfXBf50xPf91/s1mkTL7Mk8IqySYs8zryX6VWSTy5f/NXbFJXXXCdentxme6+p4jRP1l5Zpdu42nt/f/v67fm5ty3WSRaenHy8TmsP/hd7b17+zfvHLs4AcpPeJF4MUPd1Wofeq8ZbF0nt5UXjVcmqyOum2q0aD6o2RZWuoPUf06tdlXjfhAAw8ep4m3jFZpOuUviGLYZllaxTqAPI3cYVoBPXtVfvyjJLAXJZFct4mULb8BQQ9sldvGpOoBp8WyV1DV1I83LXeE2S10UVAH5rry4BV2hik8QNtA+4vr1JqizeQ4cAnSq5AhQTaNprCoAaNycMI97AW2pmUxWfk9y7vS6yZPLuzU8Aty6TVTOBluukuknzKwBTp58TbhBGCACX8XoNX2D83hQ8lDzO8C6Q5zrJAAzMDXQnqZtJnQDqOy4BE7SNcxhrry521SqZwNRcJVZnsVWYlDKpYLy2gP4SJhJnCmDvsiT0aNoQCZoqqBenFf6hrkO/TqRfyU2c7WKYpa9qb1VsyyxpYGaqNY3JMm5W10n93EudCQYgK5zDGygi9aEbIRIegt16UbTZ4XBHkZduy6KC8cyhKhWrT07Uu+qqjKs6Uc+r+kb9vI7r6yxdqkf+Ay/CbdLE67iJ1Zf/qYtc/Qa0GhwN9VzU6le9rxkxqJo0KdCefFDPMAXw389FnnA5oBpsXxV7B4/8odmXNPT8/kW+173Jd9tyD7Th5eXJyeXrFx8+RG9e/PXlB2/mzf0c0IozP/D8Asit2KTNHh+g+3G9z9dJvdXvnOfFyQ+vPrx7/eLvFqw3GtZbhFVe74F88PGdBc28/eC+WZycnKyTjVdfxxfffBtt0iwZYW+n1MmxN/nOg8U7PfHg3xo4Rt1AozIdIVcajenrbdpc00CFRZnkI79a+mPs/zUsgyxhCPgPuc3qepd/guUJhJRUoyzeLtfxVEqGVRKvR+dnF8+8Jx7+GQfe0vfHBoLBJdyVOGUjgsdoVAlQWq6+Xyd3/AuQ5I7GVRXvI3lJD1OYojBf02+3w0DAf4Gu4uCUQBNrmO2E+cjlhNYELDMYRFiUwmRoZBLmJbscugVcrtjVIa4EBEhtwABCg3ENfLHh7/SakRk/YqTdviPJh2ugtnp07xOm/tTLUtW7kF7BGPqEPnzj1/QUQlcfAmdgO/+gW030KdnXs4/VLhmHSb4CfjUa9+HCoJuChmVEQzTzL/3x46bmtgJyiLA/FgkGHnKUZIqLi2bnDaxKnh4iNq7UJHeNPRJUJ/BgUpDtwFin6ezHOKvhXQr0nzeziyPdhsnKsuI2yuOca469rz3/19xHqDACsOxn/q7ZTP7kK/wTlKjRKt5GyBsiEEfpDfO4kfW7Q27NDrjs3LwMPPs3CsGFJsfLDMTgJL7KgW+kK2+TVkAmIKnzVVrG2QQ5NgxPju+K/2FxAnLVu/zXP//yr3/+zYi9EwJ4CSWBiJN4da0+wdIE3gziqAC5qMVlWdQpIR96H375wavTqxzFzSa9I1Ejs/uPXVohN0QxmQE3RykWL+si24EMyYp4TawyZzFKmFfp1TWwY3i/gwreDWBcVAQNxO8ykXZBYI24vELuNs1hkeUs84FXA1l6b5Ir1kNM12khErhVlpYlS/XPSVXQGiaVpwB4nXrYtxq0FK4wPwu88wWLUAJG3YfVewPDJ6XXICarLchzmhaEruXjrkbQpMwQSsAlsniZZDAJCOyHBIgmqWD1nJKiFMPkqWny9mmSrbGjQIsTQnwbl0CAd6AFrdIGtDVUr2CZrUNFIcxnFI8RzmKoT7jYDL5uYEqab5/x2kyBMwLRpVvvDzPvKSkcaT6KhXt4f/bO8Z08z88XwJvVw8UCPl8Y9gx6DXT5F1yBL6uqqEYb/+UdqkiJ0b8UQTqYXcFo3QvUB1/jhYOIncmyEfxJ6w0MM/Ka8XigTRCL+YRL2o08B2ge6ry3IJnUwK+lrVXgAbu5hbETJOjtNm6q9I5ewpTg6xEWhAG4HYcfw1VR7oUvS8nJTH6BdhLDGN6l9QxI6FOSlDC+wkbbncv3I65kdYomnCYSf9WjEWI3bs/g04uxGQbmsFg8oP8KeoF373A7f62pDgQCIgTyAeQuMFB49mGo1VKr1ZQBrSn69F3W6WcJLe1IreOIuG8NkOYLgEtLNypX8LpK43yVAHuMaTFAibPwrAUNV1cExa6LKhIkoBxyfcQxvo3MOo2ARnthtIvFd+1iD/QrCjTzATlzzYOdpbCRuQrrm7VMSeBtdlkW0QMo2yIOmGQ0w51B/fnZwqYH7gR8SfMGSRdYImBCv5b1SFcdjzU16Hdzrosr68wQhGntycybnLMQNFxr5gnC3v82Rceaaol6uI5iq9RfwCnd7rYjAynAsVI0TR+hJBHbSFXFWkoHAIkH3zXQU10JeqR+fuedeUBNiabmKEs/JRqcKBNJXsCU4vZDNwjla0BOy4gnesakddwCQGlD4C5x6w7MXAIYoFr1Zj79ZgHKDOlSY6vmIEEz1hrEGXDKJ94FDIrVuTGOjN1ZGhkXvf5lwC+sYr0LQqZKvw6Rl4/HA9VogXSr0Rx3et7cFu1Ri/IkrrJ9BKJ4DZCWRZGNnCWZwZZATxxQwzkJSeLnq6yoE3vEzKIEQRN4FUzB7DyZ/Beo3vLz3OJ5gt+DrWICRYLwQzY5sthkYC2W3s9ISqLLwTADVhEoGLgHrkdVcVuzWj0njSxAwooSlDXwPiUG8IzUOVPGaG1FBXK6mYrWU5MphtRX3LED0hvYXgMdgZoC6l3ovRSw9Ih6Sg41s0y0tjcFcuJNusYqk2WMlg5AF1jTc9EsvCIHdg3dSHOAyY1n+1PQEtVvUYoAfLpJUS43sFzi0HvReFtQLQGArujJGBCqDoLcHeGDN0qRjG/iNKMOtvoVmJ6geUJGoUoyUb9Q2Tc7SZDT6x2qObCMPTJ3QLtoPEnZ2qFmhvZam3hZUcmWHiQC9sz788yaL3x6NqQ7WEW3OxgN0ESXSXObAPZnRLfPRGNQhpEZ7ZKSNZEJivr9jDe1XjX1qrmvOhlhJ/2F5ve4Ku6732mPXKGqLA08jFEtw9LyYlD1+UGPmzO4NQI09hkPcZV+sO0pWdsUjsaFxYnasRNhIYAqzq+SEaJiGTVsVWwFA5TibrBGCFWnL9hv6HIDSkdEUKG/wJy5ARxc+Ij6cufjQjeBklK34toFVE/CGPT9fD26f/LEFCXe4mtDW3RVFbsSFR4hdf+Bh0Pm/tHow9y0kGZcYVXWsCM04H7X8RSwA8N5YMRolLFFBz9U8i2e5oyqU1ANrTOuPHC7mhbDIEk7sB70gAA1mk5NbcR7MP1udhDVJaixn+zqALiLDXIFaA3xfURPAcTYKYX1wnhNXw6sbk2IyV0jhFj1E59my/7DgVFyzCcKsDLcxZskIv5IzY/QGlonTVQVRaMsKA6CU2T2JKnw41RAZzBt+DxyympGBW9BUkdqNz8a427QD0OfSBg+lnHV1IN7wJ9zRNWwJMLZY4zunVbV3g8U1yK7IYpyugXKFJQnxRa/j5zNlKqE6GqoTeEAMDXHgyi/IhSTegXqc+0JBI9Hth/lHiTIpNpp50d4+6Zofix2+ZqbU3WcyVYvZbLZbpx+JgYerUFdSbN6BLI7r9HSXRsr2bBRibg8kMHC6ChvUWVIcxDjaJ2HX3SqwWdGrnUDFNbV9U1ak+kj1yL+Ok5zbegk43hdgv4RGeyUmfwDvn9PZyTv4nXgfXz/4s2HH9++/2v07v3bj28v3742MJzGuLoFEDbxH0+E1ULzQChXMCZNU1ljAovOPPgB7STNhouqpbVlVeyXqZd0YqGGRRl08OBMw36OCs+OlBMYwF1uHSYVfMgkRJLHW2bmpIHCDi2K8FUU8UkcL3/Ai9l3IraTCEtTtc74hR+Lj3LI9TF8IzSSLFQnVTtt8Ci3XOiPM+JYx2AdVDQiGo02jYidhwbBWjaEIxXAXTQCHOO88KA3Hh0aWB8R9y7dzH38Bvy9r/i3z86GZtich7IlM6ntjkLlSZneoe1RnfzJMd8ExOCkjNeGEny1MY1z1w7HKF0syDB0yI5TN+sDteDLoUq40waYbLbCzo6eBsSjoU73pRicsCVQhc8GlUlr1qskmSgLrMOJ5OxHGQDQSLTGM9t1cpOy3U2GRFObW71/KvdFVrgFfU3SI2U3o9OcBI+hRzyo1sD1tzb3saC/6BlKV8jjQHVboUl4RCNQrr+NwdF+XcRrGGs8AW8N8TrdwDKqhSsawtRUh5vbplgVme9IEB4VxhqXnEgSPq8CwRhVV8sRHXJHq+vbqSMmsK77BuB0zi/M47TP+KxhD9Buy/SsLMtnxKKePo4rvf/pe+/yL39Tp+vChvpsytBjVClitCU16/k0EOsi/nfhfU3dbr0dhzTOJVoqzgPvIvDOjtmnoZUxrTb4wQYYULAn58nkmX6J5hW0hYRnZ2fnQ2TxCuVynUwUSWjfBuCQxa6p03WCxxTnC0sGNQXZcK+UpoXHjVVcNy55AMZ4NoLYQp8C71wdZmmuHSX5TVoV+TbJmxGu6FVCKiQMob/arePpmU9kgPLQaBNL9IVI8BQ8YYcP0JwaQKsmXgE4M3J8XASjyluH5d52INC7eJH7qAooxXCXNa7Rj71dQJ9Wx/4hv4lw5AAaGsF99c0pJj8cEx+1FWH3oCg9hAInxJcBdT3XwHWhZbz6BNp+HdLn0LRtWyctZSqKs6uighHZ1hpGXCXRoUIwGaiSrQcg3sZVHqEBSAMEJfQgPF26FzJ3cglkcL2Nq0+HuqkLdKo67R6q7hRyQcRodd/usogPY5vN04teKHHI5UJTroPLERCASG9l4VUKEdD8Vqkz56ByRocKtUZzmcUw5EX1qS5jWFlkxrsCQEUdyipDaKBt/vz96xcfor+9ff9/Prx7cfkyunz75sdXP/nurONS9Gktyrq0PwNXwo/+ezkg9NY7OZwFnsGOP89thyzLMAS7eu26VZG6V9sreZPFV2yLo1/CZcTlKPFYg0L/otC3TbOofEKvPyEvAhV05MMSruJsD3NOjik0mPoHq/z4SI405M+S4vTgLxjlMivQAch2C2GuMJc20H7V9RPSK1JKGfWTppIoCfe5yoYJ/Jm8uMwQo0TyV+XONw2jvsb1ucyI+aIPosSuCK1Yj9DMOr1KG2iBrP/yutUb5VhxVe5mFopIclyBdg6j9fiQNwNUBCHfxFlEPhkHgIDqUMLGL01qABVy+W2yLar9EFzYFrMP3n5GZyT9sE0pgD1u726hj8riHpfkI2bOyiPyNxuhHkgb20C832jHrazxaLQAPatBf4OGiw07dsiaj0hPELjSZvQJ2OEVQMaHI2AUuiloLqtEYQNblYXZf5sDAC4sVh2vzrAKq3JoacfjRWsBTkhbWTuuj2L7f6f3nfBluyvZPo0aR33tLROoAY0AG9Mr/TrJYGZD7yUQ/l7GPVmrk0PxodyjHrG6TmBg1l58BdsNAJo2tSgZONzAK4riE+zHEjT0V7uc3S9gBfNeh5QuhTAuGTwH5XMIZbCyzuFPGbCMA2gql+9+blnu2zJ/VezokIUOkiwykBXMs2AM8aBujFpTZB+zIizl0wA7oRTPWfEhRdMif97QI1oHpPoRBQ30MvGQPJWGVU3lXsBv1xF1PjDPyt0j4PGOaPrhSd7LM6o7DwH9H4/V54sTOTPdSDWcoBH7ewpZMzETKbZcp8jFhl3WZlQMdG3baErb7jql835gaEqb5hXOW3zSbfmD1t6fWe9sFZ5G1DW08hh+hA1Bd6OZGNfT71Glt2QTDbAXN1SKqY56749t9AUF2m9g+4y3COkjeLwuaBlYbZLDn7ELeALHbr6227cmUVmQmT9aA2PtNduE1jJIO1QzT1GuMSD4japTDCxyzH2DvyCZ4L8kM/GJnRE0oSiKGiSVALcU0Fafy529/bbogyv00Ad/MPTR4ammTIdelBOS8/1iusB91MX4sdTkUlLb/xs9dVbXymmNuRKRmDWf7jJ0ZtRG7ctnVCHBk8qwvnBSheXPXKkWKtNjJBwZhSpPumEV4qNCju0zkZ/dilTJJhuu1lR70ym3bVw4IzMWZBqSfYjeUm7Jg7Q1haQMIJtDga/8/4XXz9BaPSrZhFni0NoyANjhkyeuDGcENujKk9mo0nhBN7eFPjjQw+C+lpMnezWT8kdCA7+4dOF+k86Me5gfk+p7eJduO9RKDo3Zvk8rENliKwW2xVn5GhOnAtF+A/s5GGqhYydewj2JLOWIiLbUMLCf09IRsOqbLQBhbOSwiJRUOioyhyrYafLN1W/6Fqs7AmTxEiSw7in7V2/Tmlao2r3I5sLY0vX5FCMC3ay75vxDjf6VGLjZ7rDviHhFrHjUnZGzGmVfVRhKu+mQ9hjD69dGm4FYBtpnwSPGSlPLBqbJYy+ibMLOIofQdd4rHsbNu9oJqhnp1Lv3WaxOuwIIXfqEc9mfLW720OGADy2bkzIOOmiNjU7kqkJ0bu8uN9RXeN8A+mi62fNeAdBg6rA3Db0nom13ngFlv885wt1w9O0pBgA6uwzh+w52ZvlyIe//8qlVd2PxnnqL7sFiD1ZxCRypw5o4KvZUhN1olNAAVgxFOSoJwVGIkHZbxiMr8q23WZDlxxx6b9FSUMdXVZKgbRCDgqhCLVbgdar91GFSQC0XB6C6KUpjBlwnQLjabZyYlxgJlOR2NwbUs4jOdNixkkdz7rPYNr41VkHQ9AePONrqJVVSnj/qWKPlKsPM8Zizgyr9YIsUF4R27VGvx+rkwS0XpnW9W+K+ZrBFpOyHwb3KB7Uf4x6ofu7y9B876HayXeI5Q7FhP37WhlfFNUyg8noyFBtuP61TEMwxEFPDrs3ong7rJCo+WZ7OTACuxRZ0x2ZXswcEWiCAkGqyBK2qBF25ol2DJkMVkxXmxe1IhWWF8A1NKAVabmPXPbNeVWlJrOPim2/RPmaFNpHUiuh3FI3RJGyoO7Kk5tRagzgd91aUjG9ZKJRV2+eFKsRIcI3V3J8esKajTsM12OwnJr3AGNYd70trwx6hJyTAxeOEwP1SOV/Y5zjCSY5Ikip/b8ssSXNLKwUjh1B7ARrCPpglhD6d+sGqS9wGp/DHVvgg85qITYg1C9VTtFVBA3fpCiY7Lq+BMzAAC6I1H3HVpBvQh+pIYgzRtVSieHzhcMrJnFkbP1lGR5EnvHnuaq4bOiGNq8Z4Yp0F9hCYTrfkMvq7oPCHYnOCMGU4X1tVFq0azC/mQx47QZ+3oFnc2KoL1Rg+5ilL3UC7UpHJFq2HWG180BfKYlOiTrt4u6G2vRK6JZ0Bm4PWvI5cREkdeKJqslRtCdRAJGagTTGuxxcNeLoW4xCzmrkmkIVbWvE3GLD71lD0uWoq0gL4/F0/DegMvuY9UEF+Lx6GprGDtPZflEWIUWt3EXrOR+Txqxdkuu7ioZc0kmRfVfqA3GHV7NA0bK9zHEMimB64NB2RNfHRcnV9S/zNogXfIQZVxHnZB5snBl32+ReyQaVcghiKM2XIo/ZYrXTnVugfF28guvXYXQm4uSmp767m2Vrdtq+Ce5pNK8hVsheHYqoc1DgUacau8fY5seBJ58fjbkUYcIpv0NEQVgiJbB9aruzfoh+6Rh9ECK58aZ+OGlCX7LQjC5X6i2qVgs0xK48GYuQQ79ukwyoSi33+oavs9M+NBBpZ4+H/zPj9P+sZFSXWpA0LecZepsk45HZBsFoMcqHLBXq4ZIsPYIljnMDiAlhc8YFAL89DC/Pr9ADA46wAdSn8itp1KizXLXoItqsHWeQ91bPTEwvjRl3RZqRTfc6ke0Mk5FD+2JASrd0bXKxME4fGVMJ5IpvS2DUmYb2N1oujEgEWWvXpEOhjRkNoS0hpeoDAjg1KG4p+Y2lpWqexXxwAjAYgE2zTXXm41mwwhxDUsn+DkUbx9sAC6KoJDx14SoqJMqaFGC217goEciAbgeLy05OBg7Z63ocVGk7JxZ8hsHWiTYDCyiQU8rDU/g8uL4I/LG1V+105eBjif1Aef+GKezhxiUAdXPMmb6aolZy9QNBhuAS/sbeuLfIR9uBvYtiurf1DOCo7AW0YZ1+6XXT/kUcPhUSxhl7PbJVSI3a4OuOqKmPkIpkWH9Xd8Ul7IR0Ze6R+lM4ONji2R+sdROKkvUIfA038dMlGhd7im3iXNTPjaC2grDNZtnq4/OS44fMNNIJ2LH2ijOYiMWrdJnT0LUZMMRezoHUtxsndKikb7yX9oeO9Gt9NjxAwRWLA1vz3pDaCSZ7XM9LgAA3jDB7w5xkaCfBD8FjitM5eTEINy2bjnWqpxPDY0h1iOT+Q7o9dA9LJ7wKODPWMrRh/6LRPi7KhM5KNT6mqdlk2YSMdt6lNkGxRVPYqy2aJpH4vzX7VWqBfwa6Mfz33/CF/jw2IycS7d7v91aFuf/VA7lEoq0DWIyHisXiRwR4E7Un4/k4MqiphU+i6ZaplosZPbN1sso02lDxrpEIgXcN0jz15XaaktZj4VIz9sIzICBWAx2IiPt3GpUfe7rU6asez76TGNUeHOjvEnpIpbOJPiffDu1dWb9uOm8ZdS6VJkMdwB3sA/8XVlQoDaJcPyz3+wiVaoqMQeR5h3BTt3+esNZI7mQ4HBRpL5p0wK4qJ+5TsW/yN6sNrdkhTYViBHZM1XrRapV9UkX9BVfkITdMvCf6DUdrmtbBoXKn0cdxXVZ1WUk60GfY1lDmGP6gpzEYK3BPvWSg2KqkLry7C/zrzvvbOwj99A3wCaBKIYOZz4jXK5mYst3Tay8Axgg3GJ11jTMvIghgo5LkWsLuGTSaUrsmPlysgxit1Ri7Re6w4BaZrZtctYFs5oKgJVUsFGXfrtbbmaZ5TF6grc6tdG94irHdL3TN22L4ln8vZWXj2dNwKTd807phAZSS7EbWF6V/OWvtHzlszXOd80W0mTLf1NUgK6e2cD9nQBR/3hSlm4ymLjFjDzMeQ+qS2PVt00x04KsDosVBwAjAxCRE+YhYw4B47CBYL0T6JP6Jisxm1+9UwUeWYg0loxS2i0SzjnANUpZYbuksiAd3KJV6etEVdt3dJ6zUrquIrvW67Y499aNImg0Xlj+4ZgYexd8/tPkxNvPqv+b2TZ22ukPjKhNl+tVg8DIoM/9f8BSn8jptSAEOfN7Sk/xSgtJr5iFzvJNsI6+gka+N4GE2zMT2KJoi2X/MSt1FWbWv/9NXiANhp+HTz4PTH5mKwAEvGHY+rs6LCQ4G9STr5a/4mue3NMemzh54wT/SpwKnt37v2gp5iIst+0Ba2508V9yqYfEEQ8XS1A7zQjZeyOE4kiyNPZfhr3h7Ydnqud5cvMGETJkzMsh3lWMD4KCWrgl8ppnAV72p0wIa+xDmHS4VtyI8YEv/7AvQf0YaRO5Dcpsyaj0lW+R/rpHewj216uSPSGvGMdAmrxEyY5XqjfaWUHrbxrcxrBC26B5XnAf4fQiU/aJXvFA8Bqu8gE98k8HNETQKYGfwfEAL5AbMAOrl/ew2isNUBVWe9OV4HJTybQLm2o/zNodkQNwHUXfqlnR3QyxREY8R6Lm/9696g79sEmUitHnWcGr+X41GKHhj0dO4mPjH2C6NUGn8b0P1+ESTtM1S0RnEcEqdMVMf/Ki1cLFySFOUdu+Rd/vzDi7CVykN69SWB1VJFb0Hs82D1zQmFdUfoMSfpAsZbF4QkG+BYbZboC5Xt9dTamWByx1bcOA72QIMfbddYTCxQ5W7ohmx4QbAn27LZuxqacg+wt7rk0bruMYPbBu9A27IpmFpnmgg82wTasbw9uB5a0tQEcQgxoWS/X5gT3fdOHx9SB7J49akGiS3JVvpAjh9anmFoNXSyY0juBz7ufcYhem1rbrvQEUS1/zd6ZIknljUrDPSAM9a/d74EPXRduxxfMjuGtRWeOHj+dbiDjjOQiR9gE6s78hJl7Hb2z96ZHYLsfvzOO7dRVqmhWh3cYaied26dFR3F/mUf1tiicn6x80BSwM8OUw+gG6bbJzeTW2sMKQ9Mm5KOofaB0iabAywmHtLMcf+G3rh5w7YNjMg45N/HHpsY/zDsXXAwX0kPc2SYplvqBOvoqv3AyDC/Ry6n/TgxbwaA+spB4auFWrMc3Iyskk/+dcoIx5uEzXhql0gHlvbndq4QvzH7H83j2U0O8yfj11OM1RryqjKtmawKx7yw3PwK1OBw6HfbcoUcj/F0gr83w6427FHjMH5QgDged60cfDlPlemV69vrzDb2QlW33iqu2U6EZIraX46tAtd9CCOEapuGVPietvb5/0k3P1a1VA7BSGuLoyf9zqS/g6bV1bJ6zHjp9qr+TDrXQTit+Owz3xj/4Pnbs7ODNXv94Bz3UwBAEfGHICgVJ6p2OaPcqtjWHOH904tu7Pgluf+BRgg7/s0Ew9b5QgLnrgG80ICXQ5xhwnAdOI4sFGZIBcy9hzXEke+Yf89Zr4GnFJxA2F7gGZrloED53eNDP4pZ2IMKSN6vOHe4uuxU77QUx5g5GLS1vEklGXFXX9NESIZwzCeurR2Ytwvl4TKxs/4Bf6bMMq5vLI3Hu1evFdek/EW/yRJLkMxnVZ92MrjZ64/Ro1pWWK8qgXvwE7VggR6VdjV6CgSJVDnor/pznVAKZN6dw+riIx1lhUZPVYCDDAkgIXjDHGxPVfJKrSWgF3UP+yMaNtGRddy7n3hJVZV0e0v1gGYrUnr26viLNO7nlO85JhOELjF1Tg6UsHvEXk7zlkNchT3ZHT9Fx0sZPfF6Mlk6mw7t3z6YTdlKuKiCo/2xnlR77dvHgZbJ95Boxx0g+R5oH9nA+/n1x/cvXv/946vLD9EvL99/ePX2jaOFHY7rdqLKSXXpgXX0pHFDRGf5P/9sEfV9D8QHz/b3NRoVOzdG2qERLZ+d/o7s0QuEh8/aAeEk6xgInpnZ/Tyq/urURhjnS0kH6JSUwnPV7S8tgxn5bJRFqruDJqchEnmhFBN9QqqpwhOqUBu255IiKOMwp1plvNE54knPvU6XqUnOrZJmUXAYZ7Ih6zT7VOiUWvQDA+p0BIpJsEjfxuMFkZAJXqTXnGhfPA7Iy4lL6ww1djsw/habH0xPc6mHUTR7N64JOIMNWbGGP2KgN2dQsVL99N7NIh6jaPdfZTsevoYMgFXD8W9chCUFO9LCQJG3OikUM9zT0K9xi/YC0N2zjbo3Id5dIXHL40lfYD00Q9AsboQpg0A+bxUUtJKp37Bml8Ca7Hza2HG8j0WbLnc504cifQ4nhK8/vnr/4aN3+fav716//PjSkal4GUiowOF5jLt+MXK+9nYlkL/KhhFnduynCpDBcVQ3LCgzhIL7XgIlJSye4gcxLo8vA7mOG15ciKkILjmvLmV7p1wVQtEtoNe062IX8aYafbGb+Njdfsynlic671r6AiutljGK0o6edJywrdx6NpiiCsVH21cmdifBngvjQKI918ujlWqPA3NTDNulxEfv9h9R4bDmRDUii2eJPLu28G05kx9AlGsJhiqOk99SxDGZJtFLg9/NJ+cLK3MfsuRLjhzc+48M9mxzXDfwkEcM8JZbMzR0icAlT3n0rxV0LhbWe5chEmGwQZmDvhGeG7WLX+n3DnUEctVURWuBubBjmOxmOPT1yKQWu4zDLDnOCJNN4j0s0CGaYemOis+mTjk9paZ0oDI3rJIWyDuK9p95G/+eEx7SW5wj7kkUPYTtL7in4BmUUzIrXZrOrKYHsz/9Zms5mISQKvQWWaJ1gqf6CtwgqUAAWD168Eb3Vmcexr4Nw5e7yuiasgL2WPFjD6wk1UONGawlTQ8vgt8c0EWOyqRfPs4FyonGCYQZRjosrD+Is6VGdUxZAEgpr8dCRzQoq69B/6ZXSaI3RXMt2gj73UB3kabR4I7JVNg9UwXKaeXnWi4sk82c8Hc8+K4jGNrO6RUfitd+q5xMhDoY4vYDT1l5ZioJiDLzkF7RE/tjBoimbHZu7XB+F6OhFTg87Kp7ciCi6C6wb2gxYNY9hnDziY/TnVc6Ktk09fsEcaAPtXF4dDA85htpmCIlHXTTLt4Fht/YA4ocB0cOht65pyHwylUcyYUXj7ttykkoRfLTGQmyEIQIQHKN6rYpnh8qtJ1Lr5b6DqDAM09nC1D1GNr7pMadIjqCff/q9as3L1+8H7y05o/efydVYW4/AhaJK26LmXl3OZkccMuQFx4FBPIdkaaLdFnTdXErqhn+uwZCiHjAtcVi7sOMLQt/MZKBGM/DMAy86dNFz10Qhl0Iy51RX9mFnA5dZCLm9m0jC94+jM7Cb8+8JzTl6JT1DB8UTpbJvknoXhVmANE9W4jPLtYPERvJlXnqq8V8en5h+3G0pw0wJwmBLT7xLr75Zmz1aQdf/jQe0wH1yGJEeHB+j0g8sOmBzsrHR9tQI/Jb2pG6rZbwiheo8znCTU1FV3ceqE+Y1GFefvaH03NZjvEzWGWiVtiLZGZfPTUIi9bgylqGs9aqPBnOE6ZW1Mxe2PhbCHEmf4fhOOxq5p48uclWeOLoEkZ1ghPjvjO9wt1g67oETvQh4fSc00YKhsaaxOyDkgHO2jdJGbcsFQsskqgn1T2fK7T9uNp3N/2+0Vi+6uKhYCxrCKLbdN1cR9dkYcNoLPOpBdQ625h2z0HUKTlp9FDAMhbMO4Xb+FqHhxxq1ErdoQLy1OF7XyMD1brNucEybjUd8fXvBIgdEZIsvwx4vrTJKtAhEFQOoedLCeq6/5TsTSKSObr5Oo69J4+N9Dka39MbMhscgq8ieYKDgTr9sTnj1pWcvmPytca7ZQtukSjvI2iXgblKzS4jcD9Gci+otfFoz18Hew4hosm+MzeMtsoxss6Fq3dt0jBcuAvZ4tCmDbdCTxO28tOm9RWm81Wiu/WR0UbJNFUyuT7tyEe8speF2IGitojruZWOJViEEqxbt1e8PbRYsLmVhB5tDs2bBOvqGzr7tZyMpx7ledbuwlOl3wRHGMGDcnVjA5Ubh2B2J87mal2m1ik0Iev4edM5NHcp0Mg7+Vzxm+003PYjFkcAbXin0OXDaasfk4pE2VVpqcuF5vgevVWTPm9Tfe05bgSHblC3k1y4GU5kLx2Y1v+/5D45afEX9qhDBgEVlZ+d8AtV4tEsyCexxJYTVzh1cqNYT4fSG5NruWuwwruyt5g4Gn2+QpX/2NxyrPI2+0cTt7QtBD7d9igx2hI6NiX/OCQcBH043KkjqSgNQ99sPDaCqs3DOkFguKhta8q8p8jiiwRyF+KxXW4LvBPh2jkQMKut1YqKCbNz7TiCqgT1YrWndDN0sNEU5SRLbpLMtRt66XYLEw1rBxaj2PIpuZRYUHEvBhtKsdhIgZu03pmLJg5g8NvlKNIejN4qodvrkJ76tFTs2tuNbHEz6y6IS8c2rLMAB3xIYucERF2/Jp8ySQRoMfWOEqsslTgFFU9INWDPtK/S0Uu79aZNro7JlG7emPJVJZZq6RbCmzOmdHHJAXVSCeBVAfIBb9AkPHw72mLCxzli9+Qkl8ZMetAJnlz/yHTqien0Od+JhAnQnes49NHpOq2JQ7T1GqiNBTGRdpyjxiibiVVVlCWe9RYViKhrNDgi5wMVI8LDKLFV60RHLah8BifpkRpzhNwtby8iYyoCjnldrLvkx9xa+fSbO4GvqnidoiU42tVW+qU+tcoXsxndEA6yUJbkRN0d7v3rnyrmXN8io+7XXdFF5CgyYZ5aN5H73Q2LbLyx0UtV0zYhShE64W6KY/eN+30XEJvrvbGV13IwTfvrKxLeAE3fbQ5MNk+b3ZoOS7awarU32XPPvbicbqerzb3l/tByof5laen1XVEuN5njufUNXv+BN2oKExRu/Vxf2EzxGcqkhrW6gkqZ7LDN74FfYNiYG0qGLTLeztpag1aS13THT3sctwVb0El/Kryf4h3wMDR32mQc4LkaEehEV3jOy2up8JA15llo9rMEbInMfGz3w37jKS9djRJ4z87+F5nh8O3X3rfwlHbvVXGCtVr3XKfbtJH7jKChV3a4iyJamAL0yqeLxuXIf4LcBnl3XxQMxYAZtiI+QWpcYL5hJamM7HSmv8KgCRiU3B0Fe7lrC0vPOj+Y603i5iLKVov8d7cd9dwr2b6c0zlyHne2VI+B+ocvhFoxf2zfquvmlXMMRsO37MoRPxQ5dnUtryBkBG5jkrxMXTWNfoGBe4EtpcPtvblXT1bYWT8qO4IaPI72pw0T7YSXNQoOvrpX1hi7y5rrcEHoAd6xvstX+aBwbgG+9hcv6q49hnaAog4lSex9L/kS9cHW1NO7PF/Cs2gfiRtgcjuw8vUdTjFgZJhpqS/HQLuetIjWIRzk5q7xxyE30+Cu4mQ4mNDVC63rt4a2epiMt7YSCvjGmcobodbBDIGC6RWd1Qk8no1D71Ld0Wzfz+zcdozN+61LuXVw1YF1wLdNs13ryD3NoQ38lbkzuhcfkasJ6VLQvWNXQFugW0uK8yPuSpQzlNhYtUFHP4fXjXcLGi9dyGoDR5dbvIOG0pwiGWohYrnk8JjxaTqwphJWK3qtFdblAqw6oieQBTzeNQWoLjC8j4msNEqlOY9XOrMzIComLrb9nJTKRKJMbEt2pR+4W0SDtn8yAlFCydaNKE9KsRSf6jQ3rlZqF2EBL2nO5D4xVU67nukbkZn0VMrh2nactK7zsgG/wNTmEr82kTPzbjyRyuoiN5N08ha3J5zUHLVClOcWCKG0pFa2ZZEjjRotyqS5jevG2k5acFs3BsgO09pVttx2cG9JHt97wmTlBstagG13dhM5SyrBgYjZQLsM4/XNdCWqrTy4qwt5/A2vGExvI66BHRUi9FSwDnvjpJR4EWN2Eu1dQdqIBZp2u0FrQ0RUXmyaW+y1uMbQCOgQSKG1eIUzEed7CvtEP34MMjbCBjAuENeZv2s2kz+py1N0Ng1l/es5YQq8I0GMzlFQx25vH9q4pyuB92S+8cuIbi198LU/Vcvhv5u0rmW6b1kM/H6bea+VuwvaGKxbRumgY2pmSynxlAG5SnwyXNU3ICHp+NDHEwoUdMArZr4fdOcGVztsbdaZ5dNHwhX9XABS+EO6av5GL0ZcLoCFmWRrshXMnDQjpi7LZ1xJSTVyr9WQY0jKXEVKhXueuYHZR9MunQiJAbl9HqRIKd2oN1zwoQNJ5YC6t+Z+auzSrgOJdjoONGkYLxz7bvdW0lK7y6DujrBh2y2LchfT7Rd8pYK50tqayfHYtS6ay7tbST9IDaDrMyiE1vb8qq6yYjnyn/ic7JOa02HWx9Qzlas5YtahNDPpgusJxyGP97Y6+yD6lHKlWFsamNY1QDD3xVHoy8SsREnkZC9xFTqNdG+YvPG9Ny8HDuItjddUsG7cloRLlNnfDueiPzrE6cWKcpDRLte4y1s+zMqpLL2zlEvRSJDRkrKoY+MVCN7szNpI0vzIKUWks+rTDDnVoSa+DHHw6pEDNMQYK1aYOzxAu9DqKpzNXMy5cmOeOldpX7Hr1pGuRlpvjMRPkYCgZVK06y+Gg0PGYM6Gq+aRjiTDeAzpwMVwjNLlgUns2xfw5NnxEIqQd3gVskOQOHV2r/EZ60dnyl2V41/UzCMIPE3Y5X0TrcMonYl2YDxuonXYzcBE0/VNTrnHTmULupm47kf7lGncissYBqt9xjvRNIMT/c5ZizrAxQSotjNAHNjuwQzpULsk+Ty0dK1XXNaaUX7hzqcF8Phs6nM8JHWUNCPYk9QcSrDltGj2VAU8cxff6kfoXp4032CkqZ/Q0QXokfAmWp75435GSln3TWjONyJc+DxGITKnPJZEMQsKdyA6sJPhy2DQbIoc8dElduw4qltQZSHTtehQqf3FHQooZDdg1pPDFAYDomCPA605BGBHRJnsA2rzLBpyr0RgZExoo42cZhm4V1AKo9w9ZbO2/nKPjJnmBk+VIJkIaMqpAOBkzyCh6n5vtpUOa3MQkvNOG+15P8qLwRwpaogJK3U5LWxhV/ZVNibOFc8AgdyqNmHNNVEtVB5UpkbKO+YwZRTJmkLx85m5qN7QoLkhjJ/nZ4u5ZRmK7IB7xU96C7QZmwXOZvwOQHsaHIY/fhS7swxYRmBSlIpifEcG2uqGNknN+BKxvr5a9/j01KQreuXwZUQZQ/ww9Em37ymM6ZUHs+r8nKP/u1V34ljNDC2j+mqwNSe6oPz7FEGH/AkG+9tnZ48Z1LrZrZUkIVawc4NOAcyEsnB67uGx0qa1M8Nh2XFAd1BZMSL7Qifc+Ck1oQJBqMC2Ggo5NkSDkNKumoFvOlqGBvooJUPBZTnfWQXjo8tkkK/15E5SaIsIXyZZgWYvtgbqgOpOXCztqttjRNvnL945464dTzyUK67aQL/nnTAXV1coczkrAgRqs1BlILYjVabWGnwczv/Cm7KMFw2N8VNK9aFeyQWdnFPF/XKusyb1Bscemgh16yumYfkz6XPfnf6ZrDbfoROdHedHJmJLq+jP1iAXWFBX7YOmcWDet9Pxu4NgarVUypDcQ0d2n4/mHnrvmsDdXC7Uz8mmyDAomG1NDiYoVDHOnWmaGo5hDRV1ejcah0RUyUgvnfA6uRMvSM7aEzuO0UdzPJEd1xgCzWVwrM5SPCwLzD4GqajSXF3TceV20G85ccePuFLDvlMjdt24HY9smzBch+p2Uo+eexX6r6/A9iwzkCiifUZAse/o+56O5gk6aUUq3ZeOmce2Woyt8Tvp06t7EDJ3pvZnRoIfWGsciv2HozOcG+vMRXXn35yp2FjUofmDvsJOyomCbWqpzj02t13QiRLlQydh2nJgAPtrREdSXnb303n5+ZAMQ6Yd1TvYr9yNfHK4tZh5XpLMGqn6ASadL26jMl19ylS4OvJvtrC6NyPzu7mzM11oT6ahLapocwqAWQtO7blzB1UfJR2Ap9ZKB5pWEx8Pax/hCuuB5RzsH4enctOR/3aCJyJ6/DrheVbU2rzqfG81Nh5/SU64yw+/kA7/5t1/tzNMWcfamCNR2ZV4KzD12pvjlr1p6vUmXOpa4/p9RB0TXO8tAl3jQD8kyybQD6etDUage/SDai2lR4LDtdQLTi2yAThKdeytb+uV/TD69kTTgQ1RPyJ9+9Dp0Y1q/5wJH4vUKN2kxa7O9pSjh7K1T1sb9+M1+rGG9UWuMh/Q5IgUTsdOdCzHpmVYL/H6BsmbTtmXAH6iz2XJUGmciZw7fClTjRo3FYxtFouY4fFsH1b01c3UXDvgJgVrZY8FraqmsyOoRL/DF9XVDp3F3tGX0TphF3cMw4uidbGKorFVk1Kxx1Jl5E8mIvEmKPFwgfIb9QjQmplvS0V4SYGqck2wCJSZe2Vm+991kpUzzFWKWyNM4ZjEuYpW9ySBmRz7p/kpHeOe0nRw7sHBDii6mhzBbhCI4RL/Fhg+fLHGUl6s00oPpnVA81ub0degqptk9HWog7XKVDWIGfIw7xVeETibqyRjCwMQU44NwSJfhglHd1kgVfWnF8do4SU5SlppBuVAx5j2Nu2Mh7AAMa+A1+zyhPPS4KdkuNPKIWCCR0k+B9Jj1v+6KaqEhfQxVH+h1AsKHwn0cNIv1qe2cxc0RLzip3c/n7LtQjmOYBYCPu8bRlpuqsa9y+TfXHNvJVjVpF9YJybvGgKL1xNGWdZhbdIonbL3Fv0VF3dBXJIkCf70B3tQEy+zLYcRNsjqNPGsOmy/DylT3U0yMrFQSkcXg6zkQwvLPfkolIBSZDCS1+pMyi2sD+/x2c3hiarVqIOkUvV70mcPpdA+BIdTgewBHToszmG0GrxBF4+r23XG+pBrSHCoBGmds9sT6+rHOnQTVNArx7gs71oWsbZKN/dtJLUebIeGYTfbqlT/WLhTe9LVHn63eX7odEMCadzoL+zIQPyXAtIUqyJT+6TuSFrmPl3YXMzMg2HZ/jgftYHZR2mqgGsgdKsdtRBaa3CpbwFlCKwzaSRhAO8fxqJIGYLgPHj2RytsyTHCOM1YCaLIm8H6RrsgGkEr0elR65C5V4ncEy2ZMZD113Qg7MXaJo6D03aIUJyZ6IdpGpW4FgJyhEI8/TYc7EZPusFHtI3uu+kqbWB//epUJPVz7whGu9xks+ysKgtJC7uF15nkTk3xmSX3rVVRCobozCJR6sCeb9Hcr0JBknZuYvJW5MCZ5+bUEHVGqyi714mE0+uPkNPJRfGTGccDOUd7eCp7EvSfzQxnqrAcDXrt8orptwbRJjRcWcQa1rttWY/urQBdB3Ed6IhMMnciGgZQ7MTTD6M7BMnxhjcPDwE5WEV4JYHk0GITEnwV+5Edt407Kr5lrJtz2p6VWc88icI2+03z1R/FO3vE9M1oDg/BtTYBjLMTGM5pJ48cpB1JcS1DIXkq8boVflGmX5bhemZ+Bk4WWxzQg6AM1c46VHxyAmtQ5wXEs+Iown1wFEleQNoUj0/+H6buD8U='
EXPECTED_FROZEN_CODE = {'eigen_cam.py': 'e159f0bcfede0da4ce3348ca54f444de934b2b015cb42a5f577bc1f76dda78cb', 'evaluate_models.py': '50f37e547b741f97295df291c64225a9b1d0acf96e186c6be3f7736be993e2b4', 'original_manifest.csv': 'b2b4d559fcee93f4f93f8b613d204fe78d6d7f506685910917616ab3f8e83746', 'prepare_data.py': '4ed7d4cc66148fc1c917c41797433d7b8cc8886d585b2a148071b9cf7c6e024e', 'preview_training_inputs.py': '1a5c92fd646779f586515678cbaccfa1582d92db426b740ab4dedd07c6feb718', 'spine_transforms.py': '0db3795dcb4818126e2e80af91ba83e696d2a5ae84f50e22f40c0dcc6f4a3b30', 'test_evaluation.py': 'f6def3b89a9e6275d15cc56497a89a89791ba30e9286c904fbaeb2286827595d', 'train_models.py': 'fbc3425fd99b52cf5e201900f23b6d797015c71499b88ce7d0cac34cab12db1e', 'trainer_notes.txt': '2005c62a9f92fd4f974f19e815c5ef59f5a7a9833237fa4f7bf6bb537d814ed0'}

def _cam_sha(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def _cam_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def _cam_atomic_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('w', encoding='utf-8') as stream:
        json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)
        stream.write('\n')
        stream.flush()
        os.fsync(stream.fileno())
    temporary.replace(path)

def _repair_eigen_cam():
    needed = ['PROJECT', 'DATA', 'RUNS', 'ANALYSIS', 'CAM', 'CODE', 'run_logged']
    missing = [key for key in needed if key not in globals()]
    if missing:
        raise RuntimeError('Brak zmiennych bieżącej sesji: ' + ', '.join(missing) +
                           '. Użyj aktualnego zeszytu i odtwórz konfigurację sesji.')
    project, data, runs, analysis, cam, code = map(lambda value: Path(value).resolve(),
        [PROJECT, DATA, RUNS, ANALYSIS, CAM, CODE])
    for value, suffix in [(runs,'runs'), (analysis,'analysis'), (cam,'eigen_cam'),
                          (code,'frozen_code_ampfix1')]:
        if value != project/suffix:
            raise RuntimeError('Nieoczekiwana ścieżka projektu: ' + str(value))
    protocol_path = project/'protocol_artifacts_frozen.json'
    protocol = _cam_json(protocol_path)
    artifacts = protocol['artifacts']
    if artifacts['code_sha256'] != EXPECTED_FROZEN_CODE:
        raise RuntimeError('Zapisany protokół nie odpowiada sprawdzonej wersji kodu; nie zmieniono wyników.')
    for name, digest in EXPECTED_FROZEN_CODE.items():
        if not (code/name).is_file() or _cam_sha(code/name) != digest:
            raise RuntimeError('Zmieniony plik zamrożonego kodu: ' + name)
    if _cam_sha(data/'clean_manifest.csv') != artifacts['clean_manifest_sha256']:
        raise RuntimeError('Manifest danych różni się od zamrożonego protokołu.')
    completed = _cam_json(analysis/'analysis_complete.json')
    if (completed.get('status') != 'complete' or completed.get('n_evaluated_runs') != 20
            or completed.get('primary_reference_model') != 'yolo11l-cls'
            or completed.get('primary_reference_seed') != 0):
        raise RuntimeError('Najpierw wymagane jest ukończenie i ocena wszystkich 20 przebiegów.')
    if completed.get('evaluation_script_sha256') != EXPECTED_FROZEN_CODE['evaluate_models.py']:
        raise RuntimeError('Ewaluację wykonano innym skryptem niż zweryfikowany.')
    if not (runs/'external_evaluation_started.json').is_file():
        raise RuntimeError('Brak blokady zakończonego treningu.')
    batch_size = artifacts['evaluation_config']['batch_size']
    if type(batch_size) is not int or batch_size != 32:
        raise RuntimeError('Ta poprawka odtwarza zamrożoną ewaluację z batch_size=32.')
    # Query a separate interpreter, exactly as during protocol freezing.
    frozen_runtime = artifacts['runtime']
    packages = list(frozen_runtime['packages'])
    env_code = "import json,platform,importlib.metadata as m,torch; print(json.dumps({'python':platform.python_version(),'packages':{p:m.version(p) for p in " + repr(packages) + "},'torch_cuda':torch.version.cuda,'cudnn':torch.backends.cudnn.version(),'gpu':torch.cuda.get_device_name(0),'gpu_total_bytes':torch.cuda.get_device_properties(0).total_memory,'gpu_capability':torch.cuda.get_device_capability(0)}))"
    runtime = json.loads(subprocess.check_output([sys.executable, '-c', env_code], text=True))
    if runtime != frozen_runtime:
        raise RuntimeError('Środowisko różni się od zamrożonego eksperymentu; zachowaj wyniki do sprawdzenia.')
    repair_dir = project/'technical_repairs'/'eigen_cam_batch_v1'
    repair_dir.mkdir(parents=True, exist_ok=True)
    script = repair_dir/'eigen_cam_replay.py'
    source = zlib.decompress(base64.b64decode(REPLAY_PAYLOAD))
    if hashlib.sha256(source).hexdigest() != REPLAY_SHA256:
        raise RuntimeError('Uszkodzony pakiet poprawki Eigen-CAM.')
    if script.exists() and _cam_sha(script) != REPLAY_SHA256:
        raise RuntimeError('W katalogu poprawki znajduje się inny kod; nie nadpisano go.')
    if not script.exists():
        temporary = script.with_suffix('.tmp')
        temporary.write_bytes(source)
        temporary.replace(script)
    command = [sys.executable, '-u', str(script), '--frozen-code-root', str(code),
        '--dataset-root', str(data/'dataset'), '--training-root', str(runs),
        '--evaluation-root', str(analysis), '--output-root', str(cam),
        '--batch-size', str(batch_size), '--device', 'cuda:0', '--dpi', '600']
    run_logged([*command, '--validate-only'], 'cam_replay_verify_locked_inputs')

    primary = runs/'yolo11l-cls'/'seed_0'
    predictions = analysis/'yolo11l-cls'/'seed_0'
    binding = {
        'analysis_complete_sha256': _cam_sha(analysis/'analysis_complete.json'),
        'evaluation_freeze_sha256': _cam_sha(analysis/'evaluation_freeze.json'),
        'test_predictions_csv_sha256': _cam_sha(predictions/'test_predictions.csv'),
        'test_predictions_npz_sha256': _cam_sha(predictions/'test_predictions.npz'),
        'test_metrics_sha256': _cam_sha(predictions/'test_metrics.json'),
        'run_complete_sha256': _cam_sha(primary/'run_complete.json'),
        'training_plan_sha256': _cam_sha(runs/'training_plan.json'),
    }
    # Snapshot all saved analysis files and small frozen records. No model fitting
    # or statistical evaluation is invoked by the new visualization command.
    immutable_paths = [protocol_path, data/'clean_manifest.csv', runs/'training_plan.json',
                       runs/'train_models_frozen.py', runs/'external_evaluation_started.json']
    immutable_paths += [p for p in analysis.rglob('*') if p.is_file()]
    immutable_paths += list(runs.glob('*/seed_*/run_complete.json'))
    immutable_paths += [code/name for name in EXPECTED_FROZEN_CODE]
    def snapshot():
        return {str(p): _cam_sha(p) for p in sorted(set(immutable_paths))}
    before = snapshot()
    index = cam/'artifact_hashes.json'
    if index.exists():
        hashes = _cam_json(index)
        if not hashes:
            raise RuntimeError('Pusty indeks istniejących map Eigen-CAM.')
        for name, digest in hashes.items():
            relative = Path(name)
            if relative.is_absolute() or '..' in relative.parts:
                raise RuntimeError('Nieprawidłowa ścieżka w indeksie ryciny.')
            if not (cam/relative).is_file() or _cam_sha(cam/relative) != digest:
                raise RuntimeError('Zmienione pliki ukończonej ryciny Eigen-CAM; zachowano je.')
        report = _cam_json(cam/'eigen_cam_provenance.json')
        if report.get('script_sha256') != REPLAY_SHA256:
            raise RuntimeError('Istniejąca kompletna rycina pochodzi z innego skryptu; zachowano ją.')
        if any(report.get('evaluation_provenance', {}).get(k) != v for k,v in binding.items()):
            raise RuntimeError('Istniejąca rycina nie odpowiada aktualnym zamrożonym wynikom.')
        replay = _cam_json(cam/'replay_verification.json')
        if (replay.get('status') != 'passed' or replay.get('n_verified_images') != 150
                or replay.get('n_failed_images') != 0):
            raise RuntimeError('Brak zaliczonej kontroli odtworzenia partii ewaluacyjnych.')
        print('Ukończone mapy Eigen-CAM i ich zgodność z wynikami sprawdzone. Przejdź do sekcji 11.')
        return
    archived = None
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    if cam.exists() and any(cam.iterdir()):
        archived = project/'interrupted_cam'/('before_batch_replay_' + stamp)
        archived.parent.mkdir(exist_ok=True)
        shutil.move(str(cam), str(archived))
        print('Zachowano wcześniejszą przerwaną próbę Eigen-CAM:', archived)
    audit_path = repair_dir/('attempt_' + stamp + '.json')
    audit = {'repair': 'eigen_cam_batch_v1', 'status': 'started',
             'started_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
             'repair_script_sha256': REPLAY_SHA256,
             'original_cam_script_sha256': EXPECTED_FROZEN_CODE['eigen_cam.py'],
             'fixed_primary_model': 'yolo11l-cls', 'fixed_seed': 0,
             'batch_size': batch_size, 'environment': runtime,
             'archived_previous_cam': str(archived) if archived else None,
             'locked_input_binding': binding, 'immutable_files_before': before,
             'scope': 'Replay frozen test batches for visualization provenance; no training, model selection or metric recalculation.'}
    _cam_atomic_json(audit_path, audit)
    error = None
    try:
        run_logged(command, 'eigen_cam_batch_replay')
        report = _cam_json(cam/'eigen_cam_provenance.json')
        if report.get('script_sha256') != REPLAY_SHA256:
            raise RuntimeError('Raport wygenerowano inną wersją poprawki.')
        if not (cam/'artifact_hashes.json').is_file():
            raise RuntimeError('Brak końcowego indeksu artefaktów Eigen-CAM.')
        replay = _cam_json(cam/'replay_verification.json')
        if (replay.get('status') != 'passed' or replay.get('n_verified_images') != 150
                or replay.get('n_failed_images') != 0):
            raise RuntimeError('Nie potwierdzono zgodności całej kohorty testowej.')
        audit['status'] = 'complete'
    except BaseException as exc:
        error = exc
        audit['status'] = 'failed'
        audit['error_type'] = type(exc).__name__
        audit['error'] = str(exc)
    finally:
        after = snapshot()
        audit['immutable_files_unchanged'] = before == after
        audit['finished_utc'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
        if before != after:
            audit['status'] = 'failed_integrity_check'
            audit['changed_immutable_files'] = [p for p in before if before[p] != after.get(p)]
        _cam_atomic_json(audit_path, audit)
    if before != after:
        raise RuntimeError('Zmienione chronione pliki eksperymentu — zatrzymano pracę; sprawdź audyt poprawki.')
    if error is not None:
        raise error
    print('EIGEN-CAM ZAKOŃCZONE. Zgodność z zapisaną ewaluacją sprawdzona dla całego testu.')
    print('Teraz uruchom sekcję 11, aby pobrać ZIP wyników do przeglądu.')

_repair_eigen_cam()


## 11. Paczka wyników do dalszej korekty artykułu

Lekka paczka zawiera kod, manifesty, konfiguracje, techniczne kontrole i logi, wszystkie końcowe predykcje, metryki, ryciny i dokumentację treningu. Pełne wagi oraz źródłowy ZIP pozostają w katalogu projektu na Drive; nie kasuj go. Paczka nie zawiera źródłowej bazy, ale ryciny Eigen-CAM zawierają wybrane obrazy medyczne — traktuj ją jak pozostałe niepubliczne materiały badania.

Po ukończeniu prześlij powstały `axspa_clean_results_for_review.zip`. Wtedy aktualizujemy artykuł, tabele, suplementy, flowchart, Figure 5, checklisty i publiczny kod na podstawie rzeczywistych wyników, również jeżeli są niższe. Dopiero potem tworzymy nowe wydanie i DOI. Stare wyniki nie stają się dowodem nowego treningu.


In [ ]:
complete = ANALYSIS/'analysis_complete.json'
assert complete.exists() and json.loads(complete.read_text())['n_evaluated_runs'] == 20, 'Najpierw dokończ ewaluację'
assert (CAM/'artifact_hashes.json').exists(), 'Najpierw dokończ generowanie Eigen-CAM'
REPORT_ZIP = PROJECT/'axspa_clean_results_for_review.zip'
items = [PROJECT/'protocol_artifacts_frozen.json']
for folder in [CODE, PROJECT/'data_audit', PREFLIGHT, PROJECT/'execution_logs', PROJECT/'technical_repairs', ANALYSIS, CAM]:
    if folder.exists():
        items += [p for p in folder.rglob('*') if p.is_file() and '__pycache__' not in p.parts]
for p in RUNS.rglob('*'):
    if p.is_file() and p.suffix.lower() in {'.json','.csv','.txt','.yaml','.yml','.py'} and '__pycache__' not in p.parts:
        items.append(p)
with zipfile.ZipFile(REPORT_ZIP.with_suffix('.tmp'), 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(set(items)):
        z.write(p, p.relative_to(PROJECT))
    index = [{'path':str(p.relative_to(PROJECT)), 'sha256':sha(p), 'bytes':p.stat().st_size} for p in sorted(set(items))]
    z.writestr('review_package_index.json', json.dumps(index, indent=2))
REPORT_ZIP.with_suffix('.tmp').replace(REPORT_ZIP)
print('Gotowy plik:', REPORT_ZIP, 'Rozmiar MB:', round(REPORT_ZIP.stat().st_size/1e6, 2))
from google.colab import files
files.download(str(REPORT_ZIP))


## Completed verification and limits

- All 20 model/seed runs completed, and their selected checkpoint hashes were frozen before the revision evaluation. All 20 received checkpoint files subsequently matched that independent freeze byte-for-byte.
- The accepted 824-image dataset and 579/95/150 split were preserved. All validation and external-test prediction files, reference labels and four class probabilities are exported.
- All 150 primary external probability vectors were reproduced exactly during the Eigen-CAM replay. Eight panels were selected by a fixed rule; input geometry, activations, maps and overlay provenance were independently checked.
- Statistical results were independently recomputed from saved predictions. The classes are nominal: unweighted kappa is used; ordinal distance metrics are not used.
- Patient-level bootstrap uncertainty and seed variability answer different questions. Bootstrap intervals are supplied for reference seed 0 for each configuration; other seeds have point estimates and exact proportion intervals. Five seeds do not increase the number of independent test patients.
- The primary YOLO11l-cls seed-0 result is 92.0% external accuracy. The five-seed mean is 88.4%, sample SD 3.52 percentage points. No test-based selection of the best seed replaces the primary result.
- The same external cohort had been examined in earlier experiments, so the revised benchmark is exploratory. Distinct optimization, normalization, EMA and epoch-budget settings mean that comparisons concern complete training pipelines. Bitwise reproducibility on different hardware or software is not guaranteed.
- Eigen-CAM is class-agnostic qualitative visualization, not validated lesion localization. Binary morphology grouping is not a clinical diagnosis of axSpA.

The released notebook changes explanatory Markdown only; every executable cell remains byte-identical to the completed notebook. The repository contains no source PNGs, checkpoint binaries or CAM image pixels.
